# Empirinio tyrimo programinis kodas

Preliminari magistro darbo programinio kodo versija. Išsaugoti tyrimo ir tarpiniai eksperimentai originalia seka; išvestys pašalintos. Tai nėra patikrintas nuosekliai paleidžiamas notebook. Reikalingi originalūs duomenys, tarpiniai failai ir vietoje įvedamas OpenAI API raktas. API užklausos gali kainuoti. Nepaleiskite visų langelių automatiškai.

In [ ]:
# 1. Importuojame pandas
import pandas as pd

# 2. Nurodome failą
file = "HI-Small_Trans-selected-columns.csv"

# 3. Užkrauname pirmas 10 000 operacijų
df_sample = pd.read_csv(file, nrows=10000)

# 4. Parodome pirmas 5
df_sample.head()

In [ ]:
import os

print("Failas:", file)
print("Failo dydis:", os.path.getsize(file), "bytes")

with open(file, "r", encoding="utf-8") as f:
    for i in range(10):
        print(repr(f.readline()))

In [ ]:
%pip install kagglehub

In [ ]:
import kagglehub
import os

file = kagglehub.dataset_download(
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml",
    path="HI-Small_Trans.csv"
)

print("Failas:", file)

In [ ]:
import pandas as pd

df_sample = pd.read_csv(file, nrows=10000)

df_sample.head()

In [ ]:
print("Operacijų skaičius:", len(df_sample))
print("Stulpelių skaičius:", len(df_sample.columns))

In [ ]:
total_rows = 0
licit = 0
illicit = 0

for chunk in pd.read_csv(file, chunksize=100000):
    total_rows += len(chunk)
    licit += (chunk["Is Laundering"] == 0).sum()
    illicit += (chunk["Is Laundering"] == 1).sum()

print("Visos operacijos:", total_rows)
print("Teisėtos operacijos:", licit)
print("Pinigų plovimo operacijos:", illicit)
print(f"Pinigų plovimo dalis: {illicit / total_rows * 100:.4f}%")

In [ ]:
print(df_sample.info())

print("\nUnikalios reikšmės kiekviename stulpelyje:")
print(df_sample.nunique())

print("\nTrūkstamos reikšmės:")
print(df_sample.isnull().sum())

In [ ]:
print("Gavimo valiutos:")
print(df_sample["Receiving Currency"].value_counts())

print("\nMokėjimo valiutos:")
print(df_sample["Payment Currency"].value_counts())

print("\nMokėjimo būdai:")
print(df_sample["Payment Format"].value_counts())

In [ ]:
print("Operacijų su skirtingomis sumomis skaičius:")
print(
    (df_sample["Amount Received"] != df_sample["Amount Paid"]).sum()
)

print("\nOperacijų su skirtingomis valiutomis skaičius:")
print(
    (df_sample["Receiving Currency"] != df_sample["Payment Currency"]).sum()
)

In [ ]:
print("Visiškai vienodų eilučių skaičius:")
print(df_sample.duplicated().sum())

print("\nVienodų operacijų be target skaičius:")
print(
    df_sample.drop(columns=["Is Laundering"])
    .duplicated()
    .sum()
)

In [ ]:
min_date = None
max_date = None

for chunk in pd.read_csv(
    file,
    usecols=["Timestamp"],
    chunksize=100000
):
    dates = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    )

    chunk_min = dates.min()
    chunk_max = dates.max()

    if min_date is None or chunk_min < min_date:
        min_date = chunk_min

    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

print("Pirmoji operacija:", min_date)
print("Paskutinė operacija:", max_date)
print("Laikotarpis dienomis:", (max_date - min_date).days)

In [ ]:
daily_counts = {}

for chunk in pd.read_csv(
    file,
    usecols=["Timestamp", "Is Laundering"],
    chunksize=100000
):
    chunk["Date"] = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    ).dt.date

    daily = chunk.groupby("Date")["Is Laundering"].agg(
        Total="count",
        Illicit="sum"
    )

    for date, row in daily.iterrows():
        if date not in daily_counts:
            daily_counts[date] = {"Total": 0, "Illicit": 0}

        daily_counts[date]["Total"] += int(row["Total"])
        daily_counts[date]["Illicit"] += int(row["Illicit"])

daily_df = pd.DataFrame.from_dict(
    daily_counts, orient="index"
).sort_index()

daily_df["Licit"] = daily_df["Total"] - daily_df["Illicit"]
daily_df["Illicit_%"] = (
    daily_df["Illicit"] / daily_df["Total"] * 100
)

display(daily_df)

In [ ]:
import pandas as pd

# Apskaičiuojame ribas
n = 5078345

train_end = int(n * 0.60)
val_end = int(n * 0.80)

# Nuskaitome tik laiką ir target
dates = pd.read_csv(
    file,
    usecols=["Timestamp", "Is Laundering"]
)

dates["Timestamp"] = pd.to_datetime(
    dates["Timestamp"],
    format="%Y/%m/%d %H:%M"
)

# Surikiuojame chronologiškai
dates = dates.sort_values("Timestamp").reset_index(drop=True)

# Suskirstome
train = dates.iloc[:train_end]
val = dates.iloc[train_end:val_end]
test = dates.iloc[val_end:]

for name, part in [
    ("TRAIN", train),
    ("VALIDATION", val),
    ("TEST", test)
]:
    print(f"\n{name}")
    print("Operacijos:", len(part))
    print("Nuo:", part["Timestamp"].min())
    print("Iki:", part["Timestamp"].max())
    print("Illicit:", part["Is Laundering"].sum())
    print(
        "Illicit %:",
        round(part["Is Laundering"].mean() * 100, 4)
    )

In [ ]:
# Randame preliminarias ribas
train_cutoff = dates.iloc[train_end]["Timestamp"]
val_cutoff = dates.iloc[val_end]["Timestamp"]

# Perskirstome pagal laiką
train = dates[dates["Timestamp"] < train_cutoff]
val = dates[
    (dates["Timestamp"] >= train_cutoff) &
    (dates["Timestamp"] < val_cutoff)
]
test = dates[dates["Timestamp"] >= val_cutoff]

for name, part in [
    ("TRAIN", train),
    ("VALIDATION", val),
    ("TEST", test)
]:
    print(f"\n{name}")
    print("Operacijos:", len(part))
    print("Nuo:", part["Timestamp"].min())
    print("Iki:", part["Timestamp"].max())
    print("Illicit:", part["Is Laundering"].sum())
    print("Illicit %:", round(part["Is Laundering"].mean() * 100, 4))

In [ ]:
TRAIN_CUTOFF = pd.Timestamp("2022-09-06 13:36:00")
VAL_CUTOFF = pd.Timestamp("2022-09-08 16:12:00")

print("Train pabaigos riba:", TRAIN_CUTOFF)
print("Test pradžios riba:", VAL_CUTOFF)

In [ ]:
import numpy as np

# Operacijų skaičius pagal dienas
day_counts = (
    dates.groupby(dates["Timestamp"].dt.floor("D"))
    .size()
    .to_numpy()
)

# Ieškome ribų, artimiausių 60/20/20
best_score = float("inf")
best_split = None

for i in range(1, len(day_counts) - 1):
    for j in range(i + 1, len(day_counts)):
        counts = [
            day_counts[:i].sum(),
            day_counts[i:j].sum(),
            day_counts[j:].sum()
        ]

        proportions = np.array(counts) / sum(counts)

        score = max(
            abs(proportions - np.array([0.6, 0.2, 0.2]))
            / np.array([0.6, 0.2, 0.2])
        )

        if score < best_score:
            best_score = score
            best_split = (i, j)

days = sorted(dates["Timestamp"].dt.floor("D").unique())

cutoff1 = days[best_split[0]]
cutoff2 = days[best_split[1]]

print("Train / Validation riba:", cutoff1)
print("Validation / Test riba:", cutoff2)

for name, part in [
    ("TRAIN", dates[dates["Timestamp"] < cutoff1]),
    ("VALIDATION", dates[
        (dates["Timestamp"] >= cutoff1) &
        (dates["Timestamp"] < cutoff2)
    ]),
    ("TEST", dates[dates["Timestamp"] >= cutoff2])
]:
    print(f"\n{name}")
    print("Operacijos:", len(part))
    print("Illicit:", part["Is Laundering"].sum())
    print("Illicit %:", round(part["Is Laundering"].mean() * 100, 4))

In [ ]:
import sys
import sklearn
import xgboost
import pandas
import numpy

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

In [ ]:
%pip install xgboost

In [ ]:
import xgboost

print("XGBoost versija:", xgboost.__version__)

In [ ]:
import sys
import sklearn
import xgboost
import pandas
import numpy

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    tree_method="hist",
    random_state=42,
    callbacks=[]
)

print("XGBoost modelis sukurtas sėkmingai!")
print("Callbacks:", model.get_params()["callbacks"])

In [ ]:
import pandas as pd

unique_from = set()
unique_to = set()

for chunk in pd.read_csv(
    file,
    usecols=["Account", "Account.1"],
    chunksize=100000
):
    unique_from.update(chunk["Account"].dropna().unique())
    unique_to.update(chunk["Account.1"].dropna().unique())

print("Unikalių siuntėjų sąskaitų:", len(unique_from))
print("Unikalių gavėjų sąskaitų:", len(unique_to))
print("Bendras skirtingų sąskaitų skaičius:",
      len(unique_from | unique_to))

In [ ]:
import psutil

memory = psutil.virtual_memory()

print("Bendra RAM:", round(memory.total / 1024**3, 2), "GB")
print("Laisva RAM:", round(memory.available / 1024**3, 2), "GB")

In [ ]:
import os
import psutil

process = psutil.Process(os.getpid())

print(
    "Jupyter proceso RAM:",
    round(process.memory_info().rss / 1024**3, 2),
    "GB"
)

print(
    "Sistemos laisva RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

In [ ]:
import pandas as pd

feature_cols = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

# Nedidelė bandomoji imtis iš mokymo laikotarpio
sample_parts = []

for chunk in pd.read_csv(
    file,
    usecols=feature_cols,
    chunksize=100000
):
    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    )

    train_chunk = chunk[
        chunk["Timestamp"] < "2022-09-07"
    ]

    sample_parts.append(train_chunk.head(5000))

    if sum(len(x) for x in sample_parts) >= 50000:
        break

train_sample = pd.concat(
    sample_parts,
    ignore_index=True
).head(50000)

print("Operacijų:", len(train_sample))
print("Pinigų plovimo atvejų:", train_sample["Is Laundering"].sum())
print("Stulpeliai:", train_sample.columns.tolist())
print("RAM:", round(train_sample.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Laikas sekundėmis nuo pirmosios dienos vidurnakčio
train_sample["Timestamp_sec"] = (
    train_sample["Timestamp"]
    - pd.Timestamp("2022-09-01")
).dt.total_seconds().astype(int)

numeric_cols = [
    "Timestamp_sec",
    "Amount Received",
    "Amount Paid"
]

categorical_cols = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_cols)
    ]
)

X_sample = train_sample[
    numeric_cols + categorical_cols
]

X_encoded = preprocessor.fit_transform(X_sample)

print("Pradinių požymių:", X_sample.shape[1])
print("Po kodavimo požymių:", X_encoded.shape[1])
print("Matricos tipas:", type(X_encoded).__name__)
print("Matricos forma:", X_encoded.shape)

In [ ]:
from xgboost import XGBClassifier
import time

y_sample = train_sample["Is Laundering"]

pos_weight = (y_sample == 0).sum() / (y_sample == 1).sum()

xgb_test = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    scale_pos_weight=pos_weight,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

start = time.time()

xgb_test.fit(X_encoded, y_sample)

print("Modelis sėkmingai apmokytas!")
print("Mokymo trukmė:", round(time.time() - start, 2), "sek.")
print("Klasių svoris:", round(pos_weight, 2))

In [ ]:
import psutil
import os

process = psutil.Process(os.getpid())

matrix_mb = (
    X_encoded.data.nbytes
    + X_encoded.indices.nbytes
    + X_encoded.indptr.nbytes
) / 1024**2

print("Užkoduotos matricos dydis:", round(matrix_mb, 2), "MB")
print(
    "Jupyter proceso RAM:",
    round(process.memory_info().rss / 1024**3, 2),
    "GB"
)
print(
    "Prieinama sistemos RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

In [ ]:
# 200 000 operacijų iš mokymo laikotarpio
train_200k = pd.read_csv(
    file,
    nrows=200000,
    usecols=feature_cols
)

train_200k["Timestamp"] = pd.to_datetime(
    train_200k["Timestamp"],
    format="%Y/%m/%d %H:%M"
)

train_200k = train_200k[
    train_200k["Timestamp"] < "2022-09-07"
].copy()

print("Operacijų:", len(train_200k))
print("Pinigų plovimo atvejų:",
      train_200k["Is Laundering"].sum())
print("Duomenų RAM:",
      round(train_200k.memory_usage(deep=True).sum() / 1024**2, 2),
      "MB")

In [ ]:
train_200k["Timestamp_sec"] = (
    train_200k["Timestamp"]
    - pd.Timestamp("2022-09-01")
).dt.total_seconds().astype(int)

X_200k = train_200k[numeric_cols + categorical_cols]

# Naujas kodavimo objektas, kad nepakeistume ankstesnio
from sklearn.base import clone

preprocessor_200k = clone(preprocessor)

X_200k_encoded = preprocessor_200k.fit_transform(X_200k)

matrix_mb = (
    X_200k_encoded.data.nbytes
    + X_200k_encoded.indices.nbytes
    + X_200k_encoded.indptr.nbytes
) / 1024**2

print("Operacijų:", X_200k_encoded.shape[0])
print("Požymių po kodavimo:", X_200k_encoded.shape[1])
print("Matricos dydis:", round(matrix_mb, 2), "MB")
print("Matricos tipas:", type(X_200k_encoded).__name__)

In [ ]:
from sklearn.base import clone
import time
import psutil
import os

y_200k = train_200k["Is Laundering"]

pos_weight_200k = (
    (y_200k == 0).sum() / (y_200k == 1).sum()
)

xgb_200k = clone(xgb_test)
xgb_200k.set_params(scale_pos_weight=pos_weight_200k)

start = time.time()

xgb_200k.fit(X_200k_encoded, y_200k)

training_time = time.time() - start

print("Modelis apmokytas!")
print("Mokymo trukmė:", round(training_time, 2), "sek.")
print("Klasių svoris:", round(pos_weight_200k, 2))
print(
    "Jupyter proceso RAM:",
    round(psutil.Process(os.getpid()).memory_info().rss / 1024**3, 2),
    "GB"
)

In [ ]:
import pandas as pd
import psutil
import os

train_parts = []

for chunk in pd.read_csv(file, chunksize=100000):
    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    )

    chunk = chunk[
        chunk["Timestamp"] < "2022-09-07"
    ]

    if not chunk.empty:
        train_parts.append(chunk)

train_full = pd.concat(train_parts, ignore_index=True)

print("Mokymo operacijų:", len(train_full))
print("Pinigų plovimo atvejų:", train_full["Is Laundering"].sum())
print(
    "Jupyter proceso RAM:",
    round(psutil.Process(os.getpid()).memory_info().rss / 1024**3, 2),
    "GB"
)

In [ ]:
# Laikas sekundėmis nuo pirmos dienos vidurnakčio
train_full["Timestamp_sec"] = (
    train_full["Timestamp"]
    - train_full["Timestamp"].min().normalize()
).dt.total_seconds().astype(int)

# Originalūs straipsnio požymiai
X_train = train_full[numeric_cols + categorical_cols]
y_train = train_full["Is Laundering"]

# Klasių disbalanso svoris
pos_weight_full = (y_train == 0).sum() / (y_train == 1).sum()

print("X_train forma:", X_train.shape)
print("y_train forma:", y_train.shape)
print("scale_pos_weight:", round(pos_weight_full, 2))
print("Trūkstamų reikšmių:", X_train.isna().sum().sum())

In [ ]:
from sklearn.base import clone
import time
import psutil
import os

preprocessor_full = clone(preprocessor)

start = time.time()

X_train_encoded = preprocessor_full.fit_transform(X_train)

matrix_mb = (
    X_train_encoded.data.nbytes
    + X_train_encoded.indices.nbytes
    + X_train_encoded.indptr.nbytes
) / 1024**2

print("Kodavimas baigtas!")
print("Matricos forma:", X_train_encoded.shape)
print("Matricos tipas:", type(X_train_encoded).__name__)
print("Matricos dydis:", round(matrix_mb, 2), "MB")
print("Kodavimo trukmė:", round(time.time() - start, 2), "sek.")
print(
    "Jupyter proceso RAM:",
    round(psutil.Process(os.getpid()).memory_info().rss / 1024**3, 2),
    "GB"
)

In [ ]:
from xgboost import XGBClassifier
import time

xgb_final = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    scale_pos_weight=pos_weight_full,
    n_jobs=2,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

start = time.time()

xgb_final.fit(X_train_encoded, y_train)

print("Galutinis XGBoost modelis apmokytas!")
print("Mokymo trukmė:", round(time.time() - start, 2), "sek.")
print(
    "Jupyter proceso RAM:",
    round(psutil.Process(os.getpid()).memory_info().rss / 1024**3, 2),
    "GB"
)

In [ ]:
import joblib
import os

os.makedirs("aml_models", exist_ok=True)

joblib.dump(xgb_final, "aml_models/xgb_model.joblib")
joblib.dump(preprocessor_full, "aml_models/preprocessor.joblib")

print("Modelis išsaugotas!")
print("XGBoost:", os.path.getsize("aml_models/xgb_model.joblib") // 1024, "KB")
print("Preprocessor:", os.path.getsize("aml_models/preprocessor.joblib") // 1024, "KB")

In [ ]:
validation_parts = []

for chunk in pd.read_csv(file, chunksize=100000):
    timestamps = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    )

    mask = (
        (timestamps >= "2022-09-07") &
        (timestamps < "2022-09-09")
    )

    if mask.any():
        part = chunk.loc[mask].copy()
        part["Timestamp"] = timestamps.loc[mask]
        validation_parts.append(part)

validation_full = pd.concat(
    validation_parts,
    ignore_index=True
)

print("Validacijos operacijų:", len(validation_full))
print(
    "Pinigų plovimo atvejų:",
    validation_full["Is Laundering"].sum()
)
print(
    "Validacijos duomenų RAM:",
    round(
        validation_full.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

In [ ]:
# Laikas sekundėmis nuo mokymo laikotarpio pradžios
validation_full["Timestamp_sec"] = (
    validation_full["Timestamp"]
    - pd.Timestamp("2022-09-01")
).dt.total_seconds().astype(int)

# Atskiriame požymius ir tikrąsias klases
X_val = validation_full[numeric_cols + categorical_cols]
y_val = validation_full["Is Laundering"]

# Naudojame mokymo metu išmoktas kodavimo taisykles
X_val_encoded = preprocessor_full.transform(X_val)

# Apskaičiuojame pinigų plovimo tikimybes
val_prob = xgb_final.predict_proba(X_val_encoded)[:, 1]

print("Prognozių skaičius:", len(val_prob))
print("Mažiausia tikimybė:", round(val_prob.min(), 6))
print("Didžiausia tikimybė:", round(val_prob.max(), 6))
print("Vidutinė tikimybė:", round(val_prob.mean(), 6))

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val, val_prob
)

# F1 kiekvienam galimam slenksčiui
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Geriausias slenkstis:", round(best_threshold, 6))
print("Precision:", round(precision[best_idx], 4))
print("Recall:", round(recall[best_idx], 4))
print("F1:", round(f1_scores[best_idx], 4))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

val_pred_05 = (val_prob >= 0.5).astype(int)

print("Slenkstis: 0.5")
print("Precision:", round(precision_score(y_val, val_pred_05), 4))
print("Recall:", round(recall_score(y_val, val_pred_05), 4))
print("F1:", round(f1_score(y_val, val_pred_05), 4))
print("Įtartinomis pažymėtos operacijos:", val_pred_05.sum())

In [ ]:
from sklearn.metrics import confusion_matrix

# Slenkstį užfiksuojame prieš testavimą
final_threshold = float(best_threshold)

val_pred_best = (val_prob >= final_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val, val_pred_best,
    labels=[0, 1]
).ravel()

print("Pasirinktas slenkstis:", round(final_threshold, 6))
print("TP – teisingai aptiktas pinigų plovimas:", tp)
print("FP – klaidingi perspėjimai:", fp)
print("FN – neaptiktas pinigų plovimas:", fn)
print("TN – teisingai atpažintos teisėtos operacijos:", tn)
print("Iš viso įtartinomis pažymėta:", tp + fp)

In [ ]:
test_parts = []

for chunk in pd.read_csv(file, chunksize=100000):
    timestamps = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M"
    )

    mask = timestamps >= "2022-09-09"

    if mask.any():
        part = chunk.loc[mask].copy()
        part["Timestamp"] = timestamps.loc[mask]
        test_parts.append(part)

test_full = pd.concat(test_parts, ignore_index=True)

print("Testavimo operacijų:", len(test_full))
print("Pinigų plovimo atvejų:", test_full["Is Laundering"].sum())
print(
    "Testavimo duomenų RAM:",
    round(test_full.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

In [ ]:
import time

# Laiko požymis pagal originalią metodiką
test_full["Timestamp_sec"] = (
    test_full["Timestamp"]
    - pd.Timestamp("2022-09-01")
).dt.total_seconds().astype(int)

X_test = test_full[numeric_cols + categorical_cols]
y_test = test_full["Is Laundering"]

# Taikome tik mokymo metu išmoktas kodavimo taisykles
X_test_encoded = preprocessor_full.transform(X_test)

# Apskaičiuojame modelio prognozes
start = time.time()

test_prob = xgb_final.predict_proba(X_test_encoded)[:, 1]

prediction_time = time.time() - start

# Naudojame validacijoje užfiksuotą slenkstį
test_pred_best = (test_prob >= final_threshold).astype(int)

print("Prognozių skaičius:", len(test_prob))
print("Naudotas slenkstis:", round(final_threshold, 6))
print("Prognozavimo trukmė:", round(prediction_time, 2), "sek.")
print("Įtartinomis pažymėta:", test_pred_best.sum())

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred_best,
    labels=[0, 1]
).ravel()

print("=== XGBoost TEST rezultatai ===")
print("Precision:", round(precision_score(y_test, test_pred_best), 4))
print("Recall:", round(recall_score(y_test, test_pred_best), 4))
print("F1-score:", round(f1_score(y_test, test_pred_best), 4))

print("\n=== Klaidų matrica ===")
print("TP – aptiktas pinigų plovimas:", tp)
print("FP – klaidingi perspėjimai:", fp)
print("FN – neaptiktas pinigų plovimas:", fn)
print("TN – teisingai atpažintos teisėtos operacijos:", tn)

In [ ]:
import numpy as np
import pandas as pd

xgb_results = pd.DataFrame({
    "test_row_id": np.arange(len(y_test)),
    "y_true": y_test.to_numpy(),
    "xgb_probability": test_prob,
    "xgb_prediction": test_pred_best
})

xgb_results.to_csv(
    "aml_models/xgb_test_predictions.csv",
    index=False
)

print("Išsaugota operacijų:", len(xgb_results))
print("Teigiamos klasės atvejų:", xgb_results["y_true"].sum())
print("XGBoost aptikta:", xgb_results["xgb_prediction"].sum())
print(xgb_results.head())

In [ ]:
import pandas as pd

# Jei tęsiame naujoje Jupyter sesijoje
xgb_results = pd.read_csv(
    "aml_models/xgb_test_predictions.csv"
)

llm_sample = pd.concat([
    xgb_results[xgb_results["y_true"] == 1].sample(
        n=500, random_state=42
    ),
    xgb_results[xgb_results["y_true"] == 0].sample(
        n=500, random_state=42
    )
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Iš viso operacijų:", len(llm_sample))
print("Pinigų plovimo atvejų:", llm_sample["y_true"].sum())
print("Teisėtų operacijų:", (llm_sample["y_true"] == 0).sum())
print("Unikalių testinių eilučių:",
      llm_sample["test_row_id"].nunique())

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true_1000 = llm_sample["y_true"]
y_pred_1000 = llm_sample["xgb_prediction"]

tn, fp, fn, tp = confusion_matrix(
    y_true_1000,
    y_pred_1000,
    labels=[0, 1]
).ravel()

print("=== XGBoost: 1000 operacijų ===")
print("Precision:", round(precision_score(y_true_1000, y_pred_1000), 4))
print("Recall:", round(recall_score(y_true_1000, y_pred_1000), 4))
print("F1-score:", round(f1_score(y_true_1000, y_pred_1000), 4))

print("\n=== Klaidų matrica ===")
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

In [ ]:
# Paimame tas pačias 1000 testinių operacijų
llm_features = test_full.iloc[
    llm_sample["test_row_id"].to_numpy()
].copy().reset_index(drop=True)

# Pridedame identifikatorių rezultatams susieti
llm_features.insert(
    0,
    "test_row_id",
    llm_sample["test_row_id"].to_numpy()
)

# Paliekame tik pradinius operacijų požymius
llm_features = llm_features[
    ["test_row_id"] + [
        "Timestamp",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "Amount Received",
        "Receiving Currency",
        "Amount Paid",
        "Payment Currency",
        "Payment Format"
    ]
]

print("Operacijų:", len(llm_features))
print("Stulpelių:", len(llm_features.columns))
print("Ar yra Is Laundering?", "Is Laundering" in llm_features.columns)
print("Ar yra XGBoost prognozė?", "xgb_prediction" in llm_features.columns)
print(llm_features.head(3).to_string(index=False))

In [ ]:
%pip install -U openai

In [ ]:
import openai

print("OpenAI biblioteka įdiegta!")
print("Versija:", openai.__version__)

In [ ]:
import os
import getpass
from openai import OpenAI

api_key = getpass.getpass("Įklijuok OpenAI API raktą: ")

client = OpenAI(api_key=api_key)

del api_key

print("OpenAI klientas sukurtas!")

In [ ]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="Atsakyk tik vienu žodžiu: VEIKIA",
    max_output_tokens=30
)

print("Modelio atsakymas:", response.output_text)
print("Įvesties žetonai:", response.usage.input_tokens)
print("Išvesties žetonai:", response.usage.output_tokens)

In [ ]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="Atsakyk tik vienu žodžiu: VEIKIA",
    max_output_tokens=30
)

print("Modelio atsakymas:", response.output_text)
print("Įvesties žetonai:", response.usage.input_tokens)
print("Išvesties žetonai:", response.usage.output_tokens)

In [ ]:
import getpass
from openai import OpenAI

client = OpenAI(
    api_key=getpass.getpass("OpenAI API raktas: ")
)

print("Naujas API klientas sukurtas!")

In [ ]:
from openai import AuthenticationError, APIStatusError

try:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="Atsakyk tik vienu žodžiu: VEIKIA",
        max_output_tokens=30
    )

    print("Modelio atsakymas:", response.output_text)
    print("Įvesties žetonai:", response.usage.input_tokens)
    print("Išvesties žetonai:", response.usage.output_tokens)

except AuthenticationError as e:
    print("Autentifikavimo klaida:", e)

except APIStatusError as e:
    print("API klaidos kodas:", e.status_code)
    print("Klaidos pranešimas:", e.message)

In [ ]:
# Pasirenkame vieną validacijos operaciją
validation_transaction = validation_full.iloc[0]

# Paliekame tik operacijos požymius
feature_columns = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

transaction_data = validation_transaction[feature_columns].to_dict()

print("Pirmoji validacijos operacija:")
for key, value in transaction_data.items():
    print(f"{key}: {value}")

In [ ]:
import json

aml_prompt = """
You are an AI system evaluating financial transactions
for potential money laundering.

Classify the transaction using only the provided information.

Rules:
- Return 1 if the transaction appears suspicious
  for potential money laundering.
- Return 0 if the transaction does not appear suspicious.
- Do not assume that a large amount, particular currency,
  or payment method alone indicates money laundering.
- Do not invent transaction history or account behavior.
- If evidence is insufficient, acknowledge uncertainty.
- Return only a valid JSON object with:
  "prediction": 0 or 1,
  "reason": a brief explanation.
"""

response = client.responses.create(
    model="gpt-5.4-mini",
    instructions=aml_prompt,
    input=json.dumps(transaction_data, default=str),
    max_output_tokens=200
)

print("GPT atsakymas:")
print(response.output_text)

print("\nŽetonų naudojimas:")
print("Įvestis:", response.usage.input_tokens)
print("Išvestis:", response.usage.output_tokens)

In [ ]:
import json

gpt_result = json.loads(response.output_text)

true_label = int(validation_full.iloc[0]["Is Laundering"])
gpt_prediction = int(gpt_result["prediction"])

print("Tikroji klasė:", true_label)
print("GPT prognozė:", gpt_prediction)
print("Ar prognozė teisinga?", true_label == gpt_prediction)

In [ ]:
aml_prompt_v2 = """
You are an AI system classifying financial transactions
for potential money laundering.

Use only the information provided in the transaction.

Classification:
0 = No sufficient evidence of suspicious activity.
1 = Evidence of potentially suspicious activity.

Important rules:
- A large amount alone is not evidence of money laundering.
- Different currencies or amounts may reflect legitimate
  currency conversion.
- A transfer involving the same bank or account is not
  automatically suspicious.
- Do not invent transaction history, customer profiles,
  or behavioral patterns.
- Do not infer criminal activity from a currency,
  country, or payment method alone.
- If there is insufficient evidence, classify as 0.
- Apply the same criteria consistently to all transactions.

Return only valid JSON:
{"prediction": 0 or 1, "reason": "brief explanation"}
"""

print("Antroji AML instrukcijos versija paruošta.")

In [ ]:
# Atsitiktinė, subalansuota validacijos imtis
validation_sample = pd.concat([
    validation_full[
        validation_full["Is Laundering"] == 1
    ].sample(n=10, random_state=42),

    validation_full[
        validation_full["Is Laundering"] == 0
    ].sample(n=10, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Validacijos operacijų:", len(validation_sample))

print("\nTikrosios klasės:")
print(validation_sample["Is Laundering"].value_counts())

print("\nPirmosios 3 operacijos:")
print(validation_sample[feature_columns].head(3).to_string(index=False))

In [ ]:
import json
import time

gpt_validation_results = []

for i, (_, row) in enumerate(validation_sample.iterrows(), start=1):
    transaction = row[feature_columns].to_dict()

    try:
        start = time.perf_counter()

        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=aml_prompt_v2,
            input=json.dumps(transaction, default=str),
            max_output_tokens=200
        )

        elapsed = time.perf_counter() - start
        result = json.loads(response.output_text)

        prediction = int(result["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Prognozė turi būti 0 arba 1")

        gpt_validation_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": prediction,
            "reason": result.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "seconds": elapsed,
            "error": None
        })

        print(f"{i}/20 | GPT prognozė: {prediction}")

    except Exception as e:
        gpt_validation_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": None,
            "reason": "",
            "input_tokens": None,
            "output_tokens": None,
            "seconds": None,
            "error": str(e)
        })

        print(f"{i}/20 | Klaida: {type(e).__name__}")

gpt_validation_df = pd.DataFrame(gpt_validation_results)

print("\nSėkmingai klasifikuota:",
      gpt_validation_df["gpt_prediction"].notna().sum())

print("Klaidų:",
      gpt_validation_df["error"].notna().sum())

print("\nPrognozių pasiskirstymas:")
print(gpt_validation_df["gpt_prediction"].value_counts(dropna=False))

In [ ]:
pd.set_option("display.max_colwidth", 300)

print(
    gpt_validation_df[
        ["true_label", "gpt_prediction", "reason"]
    ].to_string(index=False)
)

In [ ]:
from pathlib import Path

Path("aml_models").mkdir(exist_ok=True)

gpt_validation_df.to_csv(
    "aml_models/gpt_validation_v2_20.csv",
    index=False
)

Path("aml_models/gpt_prompt_v2.txt").write_text(
    aml_prompt_v2,
    encoding="utf-8"
)

print("Validacijos rezultatai ir instrukcija išsaugoti!")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Tie patys 20 validacijos įrašų
X_val_sample = validation_sample[feature_columns].copy()

# Tokia pati požymių transformacija kaip mokymo metu
X_val_sample_encoded = preprocessor_full.transform(X_val_sample)

# XGBoost prognozės
xgb_sample_prob = xgb_final.predict_proba(
    X_val_sample_encoded
)[:, 1]

xgb_sample_pred = (
    xgb_sample_prob >= final_threshold
).astype(int)

y_sample = validation_sample["Is Laundering"].astype(int)

print("Modelis       Precision   Recall   F1")
print("-" * 43)

for name, pred in [
    ("XGBoost", xgb_sample_pred),
    ("GPT", gpt_validation_df["gpt_prediction"].astype(int))
]:
    print(
        f"{name:<13}"
        f"{precision_score(y_sample, pred, zero_division=0):<12.4f}"
        f"{recall_score(y_sample, pred):<9.4f}"
        f"{f1_score(y_sample, pred, zero_division=0):.4f}"
    )

In [ ]:
# Paruošiame tuos pačius 20 įrašų XGBoost modeliui
X_val_sample = validation_sample[feature_columns].copy()

# Konvertuojame datą į sekundes nuo 2022-09-01 vidurnakčio
X_val_sample["Timestamp_sec"] = (
    pd.to_datetime(X_val_sample["Timestamp"])
    - pd.Timestamp("2022-09-01")
).dt.total_seconds()

# Pašaliname pradinį Timestamp stulpelį
X_val_sample = X_val_sample.drop(columns=["Timestamp"])

# Atkuriame mokymo metu naudotą stulpelių tvarką
X_val_sample = X_val_sample[
    list(preprocessor_full.feature_names_in_)
]

# Transformuojame požymius
X_val_sample_encoded = preprocessor_full.transform(X_val_sample)

print("Transformacija sėkminga!")
print("Duomenų forma:", X_val_sample_encoded.shape)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# XGBoost prognozės
xgb_sample_prob = xgb_final.predict_proba(
    X_val_sample_encoded
)[:, 1]

xgb_sample_pred = (
    xgb_sample_prob >= final_threshold
).astype(int)

# Tikrosios klasės ir GPT prognozės
y_sample = validation_sample["Is Laundering"].astype(int)
gpt_sample_pred = gpt_validation_df["gpt_prediction"].astype(int)

# Rezultatų palyginimas
for name, predictions in [
    ("XGBoost", xgb_sample_pred),
    ("GPT", gpt_sample_pred)
]:
    print(f"\n{name}")
    print("Precision:", round(precision_score(
        y_sample, predictions, zero_division=0), 4))
    print("Recall:", round(recall_score(
        y_sample, predictions, zero_division=0), 4))
    print("F1:", round(f1_score(
        y_sample, predictions, zero_division=0), 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_sample, predictions, labels=[0, 1]))

In [ ]:
aml_prompt_v3 = """
You are a financial transaction risk classification system.

Your task is to identify transactions potentially associated
with money laundering using the provided transaction features.

Classify each transaction as:
0 = Lower risk of potential money laundering
1 = Higher risk of potential money laundering

Instructions:
- Assess potential risk, not proof of criminal activity.
- Consider the combination of transaction attributes.
- Do not classify a transaction as suspicious solely
  because of its amount, currency, or payment method.
- Do not assume that different currencies or amounts
  necessarily indicate suspicious activity.
- Do not invent transaction histories or customer profiles.
- Recognize that suspicious transactions may appear ordinary.
- Make a binary risk classification using available evidence.
- Do not automatically assign 0 because information is limited.

Return only valid JSON:
{"prediction": 0 or 1, "reason": "brief explanation"}
"""

print("Trečioji AML instrukcijos versija paruošta.")

In [ ]:
import json
import time

gpt_v3_results = []

for i, (_, row) in enumerate(validation_sample.iterrows(), start=1):
    transaction = row[feature_columns].to_dict()

    try:
        start = time.perf_counter()

        response_v3 = client.responses.create(
            model="gpt-5.4-mini",
            instructions=aml_prompt_v3,
            input=json.dumps(transaction, default=str),
            max_output_tokens=200
        )

        elapsed = time.perf_counter() - start
        result = json.loads(response_v3.output_text)
        prediction = int(result["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Prognozė turi būti 0 arba 1")

        gpt_v3_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": prediction,
            "reason": result.get("reason", ""),
            "input_tokens": response_v3.usage.input_tokens,
            "output_tokens": response_v3.usage.output_tokens,
            "seconds": elapsed,
            "error": None
        })

        print(f"{i}/20 | GPT prognozė: {prediction}")

    except Exception as e:
        gpt_v3_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": None,
            "reason": "",
            "input_tokens": None,
            "output_tokens": None,
            "seconds": None,
            "error": str(e)
        })

        print(f"{i}/20 | Klaida: {type(e).__name__}")

gpt_v3_df = pd.DataFrame(gpt_v3_results)

print("\nSėkmingai klasifikuota:",
      gpt_v3_df["gpt_prediction"].notna().sum())

print("Klaidų:", gpt_v3_df["error"].notna().sum())

print("\nPrognozių pasiskirstymas:")
print(gpt_v3_df["gpt_prediction"].value_counts(dropna=False))

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = gpt_v3_df["true_label"].astype(int)
y_pred = gpt_v3_df["gpt_prediction"].astype(int)

print("GPT v3 rezultatai")
print("-------------------------")
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))

print("\nKlaidų matrica:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nOperacija, kurią GPT pažymėjo kaip įtartiną:")
print(
    gpt_v3_df.loc[
        gpt_v3_df["gpt_prediction"] == 1,
        ["true_label", "gpt_prediction", "reason"]
    ].to_string(index=False)
)

In [ ]:
# Paimame pirmosios validacijos operacijos siuntėjo sąskaitą
sample_row = validation_sample.iloc[0]

bank = sample_row["From Bank"]
account = sample_row["Account"]
timestamp = pd.to_datetime(sample_row["Timestamp"])

# Ieškome ankstesnių operacijų validacijos rinkinyje
history = validation_full[
    (
        (
            (validation_full["From Bank"] == bank) &
            (validation_full["Account"] == account)
        ) |
        (
            (validation_full["To Bank"] == bank) &
            (validation_full["Account.1"] == account)
        )
    ) &
    (pd.to_datetime(validation_full["Timestamp"]) < timestamp)
].sort_values("Timestamp")

print("Ankstesnių operacijų skaičius:", len(history))

print(
    history[
        ["Timestamp", "From Bank", "Account",
         "To Bank", "Account.1", "Amount Paid",
         "Payment Currency", "Payment Format"]
    ].tail(5).to_string(index=False)
)

In [ ]:
# Pasirinktos operacijos laikas
timestamp = pd.to_datetime(sample_row["Timestamp"])

# Tik ankstesnės 24 valandos
history_24h = history[
    pd.to_datetime(history["Timestamp"]) >=
    timestamp - pd.Timedelta(hours=24)
]

print("Operacijos laikas:", timestamp)
print("Ankstesnių operacijų per 24 val.:", len(history_24h))

print("Bendra Amount Paid suma:",
      round(history_24h["Amount Paid"].sum(), 2))

print("Vidutinė Amount Paid suma:",
      round(history_24h["Amount Paid"].mean(), 2))

print("Skirtingų mokėjimo formatų skaičius:",
      history_24h["Payment Format"].nunique())

print("\nMokėjimo formatų pasiskirstymas:")
print(history_24h["Payment Format"].value_counts())

In [ ]:
print("Analizuojama operacija:")
print(sample_row[
    ["Timestamp", "From Bank", "Account",
     "To Bank", "Account.1",
     "Amount Paid", "Payment Currency"]
].to_string())

print("\nIstorijos laikotarpis:")
print("Pirma operacija:", history_24h["Timestamp"].min())
print("Paskutinė operacija:", history_24h["Timestamp"].max())

print("\nValiutų pasiskirstymas:")
print(history_24h["Payment Currency"].value_counts())

print("\nSiunčiamos ir gaunamos operacijos:")
sent = (
    (history_24h["From Bank"] == bank) &
    (history_24h["Account"] == account)
).sum()

received = (
    (history_24h["To Bank"] == bank) &
    (history_24h["Account.1"] == account)
).sum()

print("Siųstos:", sent)
print("Gautos:", received)

In [ ]:
# Suskaičiuojame kiekvienos siuntėjo sąskaitos operacijas
account_activity = (
    validation_full
    .groupby(["From Bank", "Account"])
    .size()
    .rename("transaction_count")
)

# Pasirinktos sąskaitos aktyvumas
selected_count = account_activity.loc[(bank, account)]

print("Analizuojamos sąskaitos operacijos:", selected_count)

print("\nVisų siuntėjų sąskaitų aktyvumo statistika:")
print(
    account_activity.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).round(2)
)

print("\nSąskaitų, kurių aktyvumas didesnis, procentas:")
print(
    round(
        (account_activity > selected_count).mean() * 100,
        2
    ),
    "%"
)

In [ ]:
print("Analizuojamos operacijos informacija:")
print("------------------------------------")
print("Data:", sample_row["Timestamp"])
print("Siuntėjo bankas:", sample_row["From Bank"])
print("Siuntėjo sąskaita:", sample_row["Account"])
print("Suma:", sample_row["Amount Paid"])
print("Valiuta:", sample_row["Payment Currency"])

print("\nTikroji klasė:")
print("Is Laundering =", sample_row["Is Laundering"])

print("\nGPT v3 prognozė:")
print(gpt_v3_df.iloc[0]["gpt_prediction"])

print("\nXGBoost prognozė:")
print(xgb_sample_pred[0])

In [ ]:
# Sujungiame mokymo ir validacijos duomenis
historical_data = pd.concat(
    [train_full, validation_full],
    ignore_index=True
)

# Sutvarkome laiko formatą
historical_data["Timestamp"] = pd.to_datetime(
    historical_data["Timestamp"]
)

# Surikiuojame chronologiškai
historical_data = historical_data.sort_values(
    "Timestamp"
).reset_index(drop=True)

print("Bendras operacijų skaičius:", len(historical_data))

print("Laikotarpio pradžia:",
      historical_data["Timestamp"].min())

print("Laikotarpio pabaiga:",
      historical_data["Timestamp"].max())

print("Ar operacijos surikiuotos chronologiškai?",
      historical_data["Timestamp"].is_monotonic_increasing)

In [ ]:
# Skaičiuojame siuntėjo operacijas per ankstesnes 24 val.
# Operacijos su vienoda laiko žyma viena kitos neįtraukia.

sender_counts = (
    historical_data
    .groupby(["From Bank", "Account", "Timestamp"])
    .size()
    .rename("same_time_count")
    .reset_index()
)

sender_counts["previous_24h_count"] = (
    sender_counts
    .groupby(["From Bank", "Account"], sort=False)
    .rolling("24h", on="Timestamp", closed="left")["same_time_count"]
    .sum()
    .reset_index(level=[0, 1], drop=True)
    .fillna(0)
    .astype(int)
)

print(sender_counts[
    ["From Bank", "Account", "Timestamp",
     "previous_24h_count"]
].head(10).to_string(index=False))

In [ ]:
import numpy as np

# Užtikriname teisingą chronologinę tvarką
sender_counts = sender_counts.sort_values(
    ["From Bank", "Account", "Timestamp"]
).reset_index(drop=True)

# Paruošiame rezultatų masyvą
previous_counts = np.zeros(len(sender_counts), dtype=np.int64)

# Skaičiuojame kiekvienai sąskaitai atskirai
for _, group in sender_counts.groupby(
    ["From Bank", "Account"], sort=False
):
    positions = group.index.to_numpy()

    times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    counts = group["same_time_count"].to_numpy()

    # Kiekvieno momento 24 val. lango pradžia
    window_starts = np.searchsorted(
        times,
        times - np.timedelta64(24, "h"),
        side="left"
    )

    # Kaupiamieji operacijų skaičiai
    cumulative = np.concatenate(([0], np.cumsum(counts)))

    # Įtraukiame tik ankstesnius laiko momentus
    previous_counts[positions] = (
        cumulative[np.arange(len(group))]
        - cumulative[window_starts]
    )

sender_counts["previous_24h_count"] = previous_counts

print("Skaičiavimas sėkmingas!")
print("Eilučių skaičius:", len(sender_counts))

print("\nPirmosios 10 eilučių:")
print(
    sender_counts[
        ["From Bank", "Account", "Timestamp",
         "previous_24h_count"]
    ].head(10).to_string(index=False)
)

In [ ]:
# Analizuojamos operacijos duomenys
bank = sample_row["From Bank"]
account = sample_row["Account"]
timestamp = pd.to_datetime(sample_row["Timestamp"])

# Surandame apskaičiuotą istorinį požymį
selected_history = sender_counts.loc[
    (sender_counts["From Bank"] == bank) &
    (sender_counts["Account"] == account) &
    (sender_counts["Timestamp"] == timestamp),
    "previous_24h_count"
]

# Nepriklausomai patikriname pagal pradines operacijas
manual_count = historical_data.loc[
    (historical_data["From Bank"] == bank) &
    (historical_data["Account"] == account) &
    (historical_data["Timestamp"] < timestamp) &
    (historical_data["Timestamp"] >= timestamp - pd.Timedelta(hours=24))
].shape[0]

print("Operacijos laikas:", timestamp)
print("Apskaičiuotas ankstesnių 24 val. skaičius:",
      selected_history.iloc[0] if len(selected_history) else "Nerasta")
print("Nepriklausomo patikrinimo rezultatas:", manual_count)

if len(selected_history):
    print("Ar rezultatai sutampa?",
          selected_history.iloc[0] == manual_count)

In [ ]:
# Sukuriame naują lentelę, nekeisdami pradinės imties
validation_context = validation_sample.copy()

validation_context["Timestamp"] = pd.to_datetime(
    validation_context["Timestamp"]
)

# Pridedame istorinių operacijų skaičių
validation_context = validation_context.merge(
    sender_counts[
        ["From Bank", "Account", "Timestamp",
         "previous_24h_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

# Patikriname sujungimą
print("Operacijų skaičius:", len(validation_context))

print("Trūkstamos istorinės reikšmės:",
      validation_context["previous_24h_count"].isna().sum())

print("\nIstorinio požymio pasiskirstymas:")
print(
    validation_context["previous_24h_count"]
    .describe()
    .round(2)
)

print("\nPirmosios 5 operacijos:")
print(
    validation_context[
        ["Timestamp", "Is Laundering",
         "previous_24h_count"]
    ].head().to_string(index=False)
)

In [ ]:
aml_prompt_context = """
You are an anti-money laundering (AML) transaction risk classifier.

Evaluate the potential money laundering risk of a transaction
using its attributes and the provided historical account activity.

The field "previous_24h_count" represents the number of outgoing
transactions made by the sender's bank-account pair during the
24 hours strictly before the current transaction.

Classification:
0 = Lower potential money laundering risk
1 = Higher potential money laundering risk

Instructions:
- Consider transaction attributes together with historical activity.
- Unusually frequent outgoing transactions may indicate elevated risk,
  but high activity alone does not prove money laundering.
- Do not assume that large amounts, cryptocurrencies, currency
  differences, or payment methods automatically indicate laundering.
- Do not invent customer profiles or additional transaction history.
- Evaluate potential risk rather than requiring proof of a crime.
- Make a binary classification using the available information.
- Do not automatically assign 0 because information is limited.

Return only valid JSON:
{"prediction": 0 or 1, "reason": "brief explanation"}
"""

print("Kontekstinė GPT instrukcija paruošta.")

In [ ]:
import json
import time

gpt_context_results = []

for i, (_, row) in enumerate(validation_context.iterrows(), start=1):

    # Tik modelio požymiai, be tikrosios klasės
    transaction = row[feature_columns].to_dict()
    transaction["previous_24h_count"] = int(row["previous_24h_count"])

    try:
        start = time.perf_counter()

        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=aml_prompt_context,
            input=json.dumps(transaction, default=str),
            max_output_tokens=200
        )

        elapsed = time.perf_counter() - start
        result = json.loads(response.output_text)
        prediction = int(result["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Prognozė turi būti 0 arba 1")

        gpt_context_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": prediction,
            "reason": result.get("reason", ""),
            "previous_24h_count": int(row["previous_24h_count"]),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "seconds": elapsed,
            "error": None
        })

        print(f"{i}/20 | GPT: {prediction}")

    except Exception as e:
        gpt_context_results.append({
            "true_label": int(row["Is Laundering"]),
            "gpt_prediction": None,
            "reason": "",
            "previous_24h_count": int(row["previous_24h_count"]),
            "input_tokens": None,
            "output_tokens": None,
            "seconds": None,
            "error": str(e)
        })

        print(f"{i}/20 | Klaida: {type(e).__name__}")

gpt_context_df = pd.DataFrame(gpt_context_results)

print("\nSėkmingai klasifikuota:",
      gpt_context_df["gpt_prediction"].notna().sum())

print("Klaidų:", gpt_context_df["error"].notna().sum())

print("\nPrognozių pasiskirstymas:")
print(gpt_context_df["gpt_prediction"].value_counts(dropna=False))

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = gpt_context_df["true_label"].astype(int)
y_pred = gpt_context_df["gpt_prediction"].astype(int)

print("GPT su istoriniu kontekstu")
print("-" * 35)

print("Precision:", round(
    precision_score(y_true, y_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_true, y_pred, zero_division=0), 4
))
print("F1:", round(
    f1_score(y_true, y_pred, zero_division=0), 4
))

print("\nKlaidų matrica:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nPrognozės pagal tikrąją klasę:")
print(pd.crosstab(
    gpt_context_df["true_label"],
    gpt_context_df["gpt_prediction"],
    rownames=["Tikroji klasė"],
    colnames=["GPT prognozė"]
))

In [ ]:
# GPT prognozių analizė pagal sąskaitos aktyvumą
analysis_df = gpt_context_df[
    ["true_label", "gpt_prediction",
     "previous_24h_count", "reason"]
].copy()

analysis_df["result"] = analysis_df.apply(
    lambda row:
        "TP" if row["true_label"] == 1 and row["gpt_prediction"] == 1
        else "FP" if row["true_label"] == 0 and row["gpt_prediction"] == 1
        else "FN" if row["true_label"] == 1
        else "TN",
    axis=1
)

print(
    analysis_df[
        ["result", "previous_24h_count", "reason"]
    ]
    .sort_values("previous_24h_count", ascending=False)
    .to_string(index=False, max_colwidth=100)
)

In [ ]:
from sklearn.metrics import roc_auc_score

y = validation_context["Is Laundering"].astype(int)
activity = validation_context["previous_24h_count"]

print("Istorinio požymio ROC-AUC:",
      round(roc_auc_score(y, activity), 4))

print("\nAktyvumas pagal tikrąją klasę:")
print(
    validation_context.groupby("Is Laundering")[
        "previous_24h_count"
    ].describe().round(2)
)

In [ ]:
from sklearn.metrics import roc_auc_score

# Paruošiame visą validacijos imtį
validation_activity = validation_full[
    ["Timestamp", "From Bank", "Account", "Is Laundering"]
].copy()

validation_activity["Timestamp"] = pd.to_datetime(
    validation_activity["Timestamp"]
)

# Pridedame anksčiau apskaičiuotą istorinį požymį
validation_activity = validation_activity.merge(
    sender_counts[
        ["From Bank", "Account", "Timestamp", "previous_24h_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

assert len(validation_activity) == len(validation_full)
assert validation_activity["previous_24h_count"].notna().all()

# Apskaičiuojame ROC-AUC
auc = roc_auc_score(
    validation_activity["Is Laundering"],
    validation_activity["previous_24h_count"]
)

print("Validacijos operacijų:", len(validation_activity))
print("Pinigų plovimo operacijų:",
      validation_activity["Is Laundering"].sum())
print("Istorinio požymio ROC-AUC:", round(auc, 4))

print("\nAktyvumo mediana pagal klasę:")
print(
    validation_activity.groupby("Is Laundering")[
        "previous_24h_count"
    ].median()
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = validation_activity["Is Laundering"]
activity = validation_activity["previous_24h_count"]

auc_high = roc_auc_score(y_true, activity)
auc_low = roc_auc_score(y_true, -activity)

print("Didesnis aktyvumas = didesnė rizika")
print("ROC-AUC:", round(auc_high, 4))

print("\nMažesnis aktyvumas = didesnė rizika")
print("ROC-AUC:", round(auc_low, 4))

print("\nPinigų plovimo dalis pagal aktyvumo grupes:")

activity_groups = pd.cut(
    activity,
    bins=[-1, 0, 2, 5, 10, 50, float("inf")],
    labels=["0", "1–2", "3–5", "6–10", "11–50", ">50"]
)

summary = (
    validation_activity
    .groupby(activity_groups, observed=True)["Is Laundering"]
    .agg(["count", "sum", "mean"])
)

summary["laundering_rate_pct"] = summary["mean"] * 100

print(
    summary[["count", "sum", "laundering_rate_pct"]]
    .round(4)
)

In [ ]:
from pathlib import Path
import hashlib

output_dir = Path("aml_models")
output_dir.mkdir(exist_ok=True)

# Užfiksuojame GPT modelio konfigūraciją
final_gpt_model = "gpt-5.4-mini"
final_gpt_prompt = aml_prompt_v3

prompt_path = output_dir / "gpt_final_prompt.txt"
prompt_path.write_text(final_gpt_prompt, encoding="utf-8")

# Kontrolinis kodas, kad vėliau galėtume patikrinti,
# ar instrukcija nebuvo pakeista
prompt_hash = hashlib.sha256(
    final_gpt_prompt.encode("utf-8")
).hexdigest()

print("GPT modelis:", final_gpt_model)
print("Instrukcija išsaugota:", prompt_path)
print("SHA-256:", prompt_hash)
print("Testavimo imties dydis:", len(llm_sample))

In [ ]:
import json
import time
import pandas as pd
from pathlib import Path

results_path = Path("aml_models/gpt_test_predictions.csv")

# Jei rezultatai jau yra, jų neperrašome
if results_path.exists():
    results = pd.read_csv(results_path).to_dict("records")
else:
    results = []

completed_ids = {
    int(r["test_row_id"])
    for r in results
    if pd.notna(r.get("gpt_prediction"))
}

processed = 0

for _, row in llm_features.iterrows():
    row_id = int(row["test_row_id"])

    if row_id in completed_ids:
        continue

    transaction = row[feature_columns].to_dict()

    try:
        start = time.perf_counter()

        response = client.responses.create(
            model=final_gpt_model,
            instructions=final_gpt_prompt,
            input=json.dumps(transaction, default=str),
            max_output_tokens=200
        )

        result = json.loads(response.output_text)
        prediction = int(result["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Neteisinga prognozės reikšmė")

        results.append({
            "test_row_id": row_id,
            "gpt_prediction": prediction,
            "reason": result.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "seconds": round(time.perf_counter() - start, 2),
            "error": None
        })

        completed_ids.add(row_id)
        processed += 1

        # Išsaugome po kiekvienos operacijos
        pd.DataFrame(results).to_csv(results_path, index=False)

        print(f"{processed}/10 | ID: {row_id} | GPT: {prediction}")

    except Exception as e:
        print(f"ID: {row_id} | Klaida: {e}")
        break

    if processed >= 10:
        break

print("\nNaujai klasifikuota:", processed)
print("Iš viso išsaugota:", len(completed_ids))
print("Rezultatų failas:", results_path)

In [ ]:
import json
import time
import pandas as pd

# Įkeliame jau išsaugotus rezultatus
results = pd.read_csv(results_path).to_dict("records")

completed_ids = {
    int(r["test_row_id"])
    for r in results
    if pd.notna(r.get("gpt_prediction"))
}

processed = 0
batch_size = 100

for _, row in llm_features.iterrows():
    row_id = int(row["test_row_id"])

    if row_id in completed_ids:
        continue

    transaction = row[feature_columns].to_dict()

    try:
        start = time.perf_counter()

        response = client.responses.create(
            model=final_gpt_model,
            instructions=final_gpt_prompt,
            input=json.dumps(transaction, default=str),
            max_output_tokens=200
        )

        result = json.loads(response.output_text)
        prediction = int(result["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Prognozė turi būti 0 arba 1")

        results.append({
            "test_row_id": row_id,
            "gpt_prediction": prediction,
            "reason": result.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "seconds": round(time.perf_counter() - start, 2),
            "error": None
        })

        completed_ids.add(row_id)
        processed += 1

        # Išsaugome kiekvieną rezultatą
        pd.DataFrame(results).to_csv(results_path, index=False)

        print(f"{processed}/{batch_size} | ID: {row_id} | GPT: {prediction}")

    except Exception as e:
        print(f"Klaida ties ID {row_id}: {e}")
        break

    if processed >= batch_size:
        break

print("\nNaujai klasifikuota:", processed)
print("Iš viso išsaugota:", len(completed_ids))
print("Liko:", len(llm_features) - len(completed_ids))

print("\nPrognozių pasiskirstymas:")
print(pd.DataFrame(results)["gpt_prediction"].value_counts())

In [ ]:
def continue_gpt_testing(batch_size=100):
    import json
    import time
    import pandas as pd

    results = pd.read_csv(results_path).to_dict("records")

    completed_ids = set(
        pd.to_numeric(
            pd.DataFrame(results)["test_row_id"],
            errors="coerce"
        ).dropna().astype(int)
    )

    processed = 0

    for _, row in llm_features.iterrows():
        row_id = int(row["test_row_id"])

        if row_id in completed_ids:
            continue

        transaction = row[feature_columns].to_dict()

        try:
            start = time.perf_counter()

            response = client.responses.create(
                model=final_gpt_model,
                instructions=final_gpt_prompt,
                input=json.dumps(transaction, default=str),
                max_output_tokens=200
            )

            result = json.loads(response.output_text)
            prediction = int(result["prediction"])

            if prediction not in (0, 1):
                raise ValueError("Neteisinga prognozė")

            results.append({
                "test_row_id": row_id,
                "gpt_prediction": prediction,
                "reason": result.get("reason", ""),
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "seconds": round(time.perf_counter() - start, 2),
                "error": None
            })

            completed_ids.add(row_id)
            processed += 1

            pd.DataFrame(results).to_csv(results_path, index=False)

            if processed % 10 == 0:
                print(f"Atlikta: {processed}/{batch_size}")

        except Exception as e:
            print(f"Klaida ties ID {row_id}: {e}")
            break

        if processed >= batch_size:
            break

    df = pd.DataFrame(results)

    print("\nIš viso klasifikuota:", len(completed_ids))
    print("Liko:", len(llm_features) - len(completed_ids))
    print("\nPrognozių pasiskirstymas:")
    print(df["gpt_prediction"].value_counts(dropna=False))

# Paleidžiame kitą 100 operacijų partiją
continue_gpt_testing(batch_size=100)

In [ ]:
continue_gpt_testing(batch_size=200)

In [ ]:
continue_gpt_testing(batch_size=200)

In [ ]:
continue_gpt_testing(batch_size=200)

In [ ]:
continue_gpt_testing(batch_size=190)

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score
)
import pandas as pd

gpt_results = pd.read_csv("aml_models/gpt_test_predictions.csv")

# Sujungiame prognozes su tikrosiomis klasėmis
comparison = llm_sample[["test_row_id", "Is Laundering"]].merge(
    gpt_results[["test_row_id", "gpt_prediction"]],
    on="test_row_id",
    how="left",
    validate="one_to_one"
)

assert len(comparison) == 1000
assert comparison["gpt_prediction"].notna().all()
assert comparison["test_row_id"].is_unique

y_true = comparison["Is Laundering"].astype(int)
y_pred = comparison["gpt_prediction"].astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("GPT rezultatai:")
print("TP:", tp, "| FP:", fp, "| FN:", fn, "| TN:", tn)
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:   ", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score: ", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("Accuracy: ", round(accuracy_score(y_true, y_pred), 4))

In [ ]:
print("llm_sample stulpeliai:")
print(llm_sample.columns.tolist())

print("\nllm_features stulpeliai:")
print(llm_features.columns.tolist())

print("\ntest_full stulpeliai:")
print(test_full.columns.tolist())

print("\nPirmosios llm_sample eilutės:")
print(llm_sample.head(3).to_string())

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

gpt_results = pd.read_csv("aml_models/gpt_test_predictions.csv")

comparison = llm_sample.merge(
    gpt_results[["test_row_id", "gpt_prediction"]],
    on="test_row_id",
    how="left",
    validate="one_to_one"
)

assert len(comparison) == 1000
assert comparison["test_row_id"].is_unique
assert comparison[["y_true", "xgb_prediction", "gpt_prediction"]].notna().all()
assert comparison["y_true"].value_counts().to_dict() == {0: 500, 1: 500}

for model, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT-5.4-mini", "gpt_prediction")
]:
    y_true = comparison["y_true"].astype(int)
    y_pred = comparison[column].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    print(f"\n{model}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

comparison.to_csv("aml_models/final_model_comparison.csv", index=False)
print("\nPalyginimas išsaugotas: aml_models/final_model_comparison.csv")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

assert comparison[["y_true", "xgb_prediction", "gpt_prediction"]].notna().all().all()
assert comparison["y_true"].value_counts().to_dict() == {0: 500, 1: 500}

for model, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT-5.4-mini", "gpt_prediction")
]:
    y_true = comparison["y_true"].astype(int)
    y_pred = comparison[column].astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    print(f"\n{model}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

comparison.to_csv("aml_models/final_model_comparison.csv", index=False)
print("\nRezultatai išsaugoti.")

In [ ]:
gpt_analysis = comparison.merge(
    gpt_results[["test_row_id", "reason"]],
    on="test_row_id",
    how="left",
    validate="one_to_one"
)

gpt_positive = gpt_analysis[
    gpt_analysis["gpt_prediction"] == 1
].copy()

gpt_positive["result"] = gpt_positive["y_true"].map({
    0: "False Positive",
    1: "True Positive"
})

print(gpt_positive[
    ["test_row_id", "y_true", "xgb_prediction", "result", "reason"]
].to_string(index=False))

In [ ]:
import pandas as pd

positive = gpt_analysis[gpt_analysis["gpt_prediction"] == 1].copy()
reason = positive["reason"].fillna("").str.lower()

positive["cash"] = reason.str.contains(r"cash")
positive["crypto"] = reason.str.contains(r"bitcoin|crypto")
positive["self_transfer"] = reason.str.contains(
    r"self-transfer|same account|sender and receiver account are the same"
)

summary = pd.DataFrame({
    "Požymis": ["Grynieji pinigai", "Kriptovaliutos", "Pervedimai į tą pačią sąskaitą"],
    "Iš viso paminėta": [
        positive["cash"].sum(),
        positive["crypto"].sum(),
        positive["self_transfer"].sum()
    ],
    "Teisingos prognozės (TP)": [
        (positive["cash"] & (positive["y_true"] == 1)).sum(),
        (positive["crypto"] & (positive["y_true"] == 1)).sum(),
        (positive["self_transfer"] & (positive["y_true"] == 1)).sum()
    ],
    "Klaidingos prognozės (FP)": [
        (positive["cash"] & (positive["y_true"] == 0)).sum(),
        (positive["crypto"] & (positive["y_true"] == 0)).sum(),
        (positive["self_transfer"] & (positive["y_true"] == 0)).sum()
    ]
})

print(summary.to_string(index=False))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

comparison["combined_prediction"] = (
    (comparison["xgb_prediction"] == 1) |
    (comparison["gpt_prediction"] == 1)
).astype(int)

y_true = comparison["y_true"].astype(int)
y_combined = comparison["combined_prediction"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_combined, labels=[0, 1]
).ravel()

print("XGBoost + GPT (OR taisyklė)")
print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")
print(f"Precision: {precision_score(y_true, y_combined):.4f}")
print(f"Recall:    {recall_score(y_true, y_combined):.4f}")
print(f"F1-score:  {f1_score(y_true, y_combined):.4f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

rows = []

for name, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT-5.4-mini", "gpt_prediction"),
    ("XGBoost + GPT (OR)", "combined_prediction")
]:
    y = comparison["y_true"].astype(int)
    pred = comparison[column].astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    rows.append({
        "Model": name,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Precision": round(precision_score(y, pred), 4),
        "Recall": round(recall_score(y, pred), 4),
        "F1": round(f1_score(y, pred), 4)
    })

final_metrics = pd.DataFrame(rows)
final_metrics.to_csv("aml_models/final_metrics.csv", index=False)

print(final_metrics.to_string(index=False))
print("\nIšsaugota: aml_models/final_metrics.csv")

In [ ]:
n = len(test_full)
positives = int(test_full["Is Laundering"].sum())

print("Testavimo operacijų:", n)
print("Pinigų plovimo operacijų:", positives)
print("Teisėtų operacijų:", n - positives)
print(f"Pinigų plovimo dalis: {positives / n * 100:.4f}%")
print(f"Tikėtinas teigiamų atvejų skaičius 3000 imtyje: {3000 * positives / n:.2f}")

In [ ]:
from pathlib import Path

print("Dabartinis aplankas:", Path.cwd())

print("\nIšsaugoti eksperimento failai:")
for file in sorted(Path("aml_models").glob("*")):
    print(file.name)

print("\nDuomenų CSV failai:")
for file in Path(".").rglob("*.csv"):
    print(file)

In [ ]:
import pandas as pd

data_path = "HI-Small_Trans-selected-columns.csv"

df_preview = pd.read_csv(data_path, nrows=5)

print("Stulpeliai:")
print(df_preview.columns.tolist())

print("\nPirmosios 5 operacijos:")
print(df_preview.to_string(index=False))

print("\nFailo dydis (MB):")
from pathlib import Path
print(round(Path(data_path).stat().st_size / 1024**2, 2))

In [ ]:
from pathlib import Path

folders = [
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
    Path.cwd()
]

for folder in folders:
    print(f"\n{folder}")
    if folder.exists():
        for file in folder.rglob("*HI-Small*"):
            if file.is_file():
                print(f"  {file} ({file.stat().st_size / 1024**2:.1f} MB)")
            

In [ ]:
import pandas as pd

data_path = "HI-Small_Trans.csv"

df = pd.read_csv(data_path)

print("Duomenų dydis:", df.shape)
print("\nStulpeliai:")
print(df.columns.tolist())

print("\nPinigų plovimo operacijų pasiskirstymas:")
print(df["Is Laundering"].value_counts())

print("\nDatų intervalas:")
print(df["Timestamp"].min(), "–", df["Timestamp"].max())

In [ ]:
# Paverčiame datas į datetime formatą
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%Y/%m/%d %H:%M"
)

# Atkuriame ankstesnį chronologinį skaidymą
train_full = df[df["Timestamp"] < "2022-09-07"].copy()

validation_full = df[
    (df["Timestamp"] >= "2022-09-07") &
    (df["Timestamp"] < "2022-09-09")
].copy()

test_full = df[df["Timestamp"] >= "2022-09-09"].copy()

for name, data in [
    ("Mokymo", train_full),
    ("Validavimo", validation_full),
    ("Testavimo", test_full)
]:
    positives = int(data["Is Laundering"].sum())

    print(
        f"{name}: {len(data):,} operacijų | "
        f"Pinigų plovimo: {positives:,} | "
        f"Dalis: {positives / len(data) * 100:.4f}%"
    )

In [ ]:
# Atsitiktinė 3000 operacijų imtis iš testavimo duomenų
test_3000 = test_full.sample(
    n=3000,
    random_state=42
).copy()

# Išsaugome originalių testavimo eilučių identifikatorius
test_3000["test_row_id"] = test_3000.index

print("Imties dydis:", len(test_3000))
print("\nKlasių pasiskirstymas:")
print(test_3000["Is Laundering"].value_counts())

print("\nPinigų plovimo dalis:")
print(f"{test_3000['Is Laundering'].mean() * 100:.4f}%")

In [ ]:
from pathlib import Path

output_dir = Path("aml_models")
output_dir.mkdir(exist_ok=True)

sample_path = output_dir / "test_3000_natural.csv"

test_3000.to_csv(sample_path, index=False)

print("Imtis išsaugota:", sample_path)
print("Operacijų:", len(test_3000))
print("Unikalių ID:", test_3000["test_row_id"].nunique())
print("Pinigų plovimo atvejų:", test_3000["Is Laundering"].sum())

In [ ]:
import numpy as np

# Sąskaitą identifikuojame pagal banką ir sąskaitos numerį
history = df[["Timestamp", "From Bank", "Account"]].copy()
history["original_id"] = history.index

# Suskaičiuojame operacijas kiekvienu laiko momentu
counts = (
    history.groupby(
        ["From Bank", "Account", "Timestamp"],
        dropna=False,
        sort=False
    )
    .size()
    .rename("count_at_time")
    .reset_index()
)

# Kiekvienai sąskaitai skaičiuojame ankstesnes 24 val.
counts = counts.sort_values(
    ["From Bank", "Account", "Timestamp"]
)

counts["previous_24h_count"] = (
    counts.groupby(["From Bank", "Account"], dropna=False)
    .rolling("24h", on="Timestamp", closed="left")["count_at_time"]
    .sum()
    .reset_index(level=[0, 1], drop=True)
    .fillna(0)
    .to_numpy()
)

# Prijungiame kontekstą prie 3000 atrinktų operacijų
test_3000_context = test_3000.merge(
    counts[
        ["From Bank", "Account", "Timestamp", "previous_24h_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų:", len(test_3000_context))
print("Trūkstamų konteksto reikšmių:",
      test_3000_context["previous_24h_count"].isna().sum())

print("\nAnkstesnių 24 val. operacijų skaičius:")
print(test_3000_context["previous_24h_count"].describe())

In [ ]:
# Patikriname 5 atsitiktines operacijas
check_rows = test_3000_context.sample(n=5, random_state=42)

for _, row in check_rows.iterrows():
    actual_count = (
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] >= row["Timestamp"] - pd.Timedelta(hours=24)) &
        (df["Timestamp"] < row["Timestamp"])
    ).sum()

    calculated_count = int(row["previous_24h_count"])

    print(
        f"ID: {row['test_row_id']} | "
        f"Apskaičiuota: {calculated_count} | "
        f"Patikrinta: {actual_count} | "
        f"Sutampa: {calculated_count == actual_count}"
    )

In [ ]:
# Skaičiuojame ankstesnių 7 dienų operacijas
counts["previous_7d_count"] = (
    counts.groupby(["From Bank", "Account"], dropna=False)
    .rolling(
        "7D",
        on="Timestamp",
        closed="left"
    )["count_at_time"]
    .sum()
    .reset_index(level=[0, 1], drop=True)
    .fillna(0)
    .to_numpy()
)

# Pridedame naują požymį prie 3000 operacijų imties
test_3000_context = test_3000_context.merge(
    counts[
        ["From Bank", "Account", "Timestamp", "previous_7d_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų:", len(test_3000_context))
print("Trūkstamų reikšmių:",
      test_3000_context["previous_7d_count"].isna().sum())

print("\nAnkstesnių 7 dienų operacijų skaičius:")
print(test_3000_context["previous_7d_count"].describe())

print("\nPatikra: 7 dienų skaičius >= 24 val. skaičiaus")
print(
    (test_3000_context["previous_7d_count"] >=
     test_3000_context["previous_24h_count"]).all()
)

In [ ]:
# Patikriname 5 atsitiktines operacijas
check_rows = test_3000_context.sample(n=5, random_state=123)

for _, row in check_rows.iterrows():
    actual_count = (
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] >= row["Timestamp"] - pd.Timedelta(days=7)) &
        (df["Timestamp"] < row["Timestamp"])
    ).sum()

    calculated_count = int(row["previous_7d_count"])

    print(
        f"ID: {row['test_row_id']} | "
        f"Apskaičiuota: {calculated_count} | "
        f"Patikrinta: {actual_count} | "
        f"Sutampa: {calculated_count == actual_count}"
    )

In [ ]:
# Ankstesnių 24 val. išsiųsta suma ta pačia valiuta
amounts = (
    df.groupby(
        ["From Bank", "Account", "Payment Currency", "Timestamp"],
        dropna=False,
        sort=False
    )["Amount Paid"]
    .sum()
    .rename("amount_at_time")
    .reset_index()
)

amounts = amounts.sort_values(
    ["From Bank", "Account", "Payment Currency", "Timestamp"]
).reset_index(drop=True)

amounts["previous_24h_amount"] = (
    amounts.groupby(
        ["From Bank", "Account", "Payment Currency"],
        dropna=False
    )
    .rolling(
        "24h",
        on="Timestamp",
        closed="left"
    )["amount_at_time"]
    .sum()
    .reset_index(level=[0, 1, 2], drop=True)
    .fillna(0)
    .to_numpy()
)

test_3000_context = test_3000_context.merge(
    amounts[
        ["From Bank", "Account", "Payment Currency",
         "Timestamp", "previous_24h_amount"]
    ],
    on=["From Bank", "Account", "Payment Currency", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų:", len(test_3000_context))
print("Trūkstamų reikšmių:",
      test_3000_context["previous_24h_amount"].isna().sum())

print("\nAnkstesnių 24 val. sumų statistika:")
print(test_3000_context["previous_24h_amount"].describe())

In [ ]:
import numpy as np

# Patikriname 5 atsitiktines operacijas
check_rows = test_3000_context.sample(n=5, random_state=123)

for _, row in check_rows.iterrows():
    mask = (
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Payment Currency"] == row["Payment Currency"]) &
        (df["Timestamp"] >= row["Timestamp"] - pd.Timedelta(hours=24)) &
        (df["Timestamp"] < row["Timestamp"])
    )

    actual_amount = df.loc[mask, "Amount Paid"].sum()
    calculated_amount = row["previous_24h_amount"]

    print(
        f"ID: {row['test_row_id']} | "
        f"Valiuta: {row['Payment Currency']} | "
        f"Apskaičiuota: {calculated_amount:.2f} | "
        f"Patikrinta: {actual_amount:.2f} | "
        f"Sutampa: {np.isclose(calculated_amount, actual_amount)}"
    )

In [ ]:
# Unikalių gavėjų skaičius per ankstesnes 7 dienas
sender_keys = ["From Bank", "Account"]

selected_senders = test_3000_context[
    sender_keys
].drop_duplicates()

sender_history = df.merge(
    selected_senders,
    on=sender_keys,
    how="inner"
)

recipient_counts = []

for _, row in test_3000_context.iterrows():
    previous = sender_history[
        (sender_history["From Bank"] == row["From Bank"]) &
        (sender_history["Account"] == row["Account"]) &
        (sender_history["Timestamp"] >= row["Timestamp"] - pd.Timedelta(days=7)) &
        (sender_history["Timestamp"] < row["Timestamp"])
    ]

    unique_recipients = previous[
        ["To Bank", "Account.1"]
    ].drop_duplicates().shape[0]

    recipient_counts.append(unique_recipients)

test_3000_context["previous_7d_unique_recipients"] = recipient_counts

print("Operacijų:", len(test_3000_context))
print("\nUnikalių gavėjų skaičius per ankstesnes 7 dienas:")
print(test_3000_context["previous_7d_unique_recipients"].describe())

print("\nTrūkstamų reikšmių:",
      test_3000_context["previous_7d_unique_recipients"].isna().sum())

In [ ]:
from pathlib import Path

context_features = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients"
]

# Patikriname logines sąlygas
checks = {
    "3000 operacijų": len(test_3000_context) == 3000,
    "Unikalūs ID": test_3000_context["test_row_id"].is_unique,
    "Nėra trūkstamų reikšmių":
        test_3000_context[context_features].notna().all().all(),
    "7 dienų skaičius >= 24 val. skaičiaus":
        (test_3000_context["previous_7d_count"] >=
         test_3000_context["previous_24h_count"]).all(),
    "Gavėjų skaičius <= 7 dienų operacijų skaičiaus":
        (test_3000_context["previous_7d_unique_recipients"] <=
         test_3000_context["previous_7d_count"]).all()
}

for name, result in checks.items():
    print(f"{name}: {result}")

if all(checks.values()):
    path = Path("aml_models/test_3000_context.csv")
    test_3000_context.to_csv(path, index=False)
    print("\nIšsaugota:", path)
else:
    print("\nDuomenys neišsaugoti – reikia patikrinti klaidas.")

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

prompt_path = Path("aml_models/gpt_final_prompt.txt")
results_path = Path("aml_models/gpt_test_predictions.csv")

prompt = prompt_path.read_text(encoding="utf-8")

print("ANKSTESNIS GPT PROMPTAS:")
print(prompt)

print("\nPROMPTO SHA-256:")
print(hashlib.sha256(prompt.encode("utf-8")).hexdigest())

previous_results = pd.read_csv(results_path)

print("\nANKSTESNIŲ REZULTATŲ STULPELIAI:")
print(previous_results.columns.tolist())

print("\nANKSTESNIŲ REZULTATŲ SKAIČIUS:")
print(len(previous_results))

print("\nAR TURIME API KLIENTĄ?")
print("Taip" if "client" in globals() else "Ne")

In [ ]:
from openai import OpenAI
from getpass import getpass

api_key = getpass("Įvesk OpenAI API raktą: ")

client = OpenAI(api_key=api_key)

# Patikriname, ar klientas sukurtas
print("API klientas sukurtas:", isinstance(client, OpenAI))

# Pašaliname atskirą rakto kintamąjį
del api_key

In [ ]:
# Ankstesnių GPT prognozių identifikatoriai
previous_ids = set(previous_results["test_row_id"])

# Naujos 3000 operacijų imties identifikatoriai
new_ids = set(test_3000_context["test_row_id"])

overlap = previous_ids & new_ids

print("Ankstesnių GPT prognozių:", len(previous_ids))
print("Naujos imties operacijų:", len(new_ids))
print("Sutampančių operacijų:", len(overlap))
print("Naujų GPT užklausų be konteksto reikėtų:",
      len(new_ids - previous_ids))

print("\nSutampančių operacijų klasių pasiskirstymas:")
print(
    test_3000_context.loc[
        test_3000_context["test_row_id"].isin(overlap),
        "Is Laundering"
    ].value_counts()
)

In [ ]:
import pandas as pd

xgb_saved = pd.read_csv("aml_models/xgb_test_predictions.csv")

print("Išsaugotų XGBoost prognozių:", len(xgb_saved))

print("\nAnkstesnių XGBoost ID:")
print(xgb_saved["test_row_id"].head(10).tolist())
print("ID intervalas:",
      xgb_saved["test_row_id"].min(),
      "–",
      xgb_saved["test_row_id"].max())

print("\nNaujos imties ID:")
print(test_3000_context["test_row_id"].head(10).tolist())

print("\nTestavimo duomenų originalūs indeksai:")
print(test_full.index[:10].tolist())

print("\nAnkstesnių GPT ID:")
print(previous_results["test_row_id"].head(10).tolist())

In [ ]:
from openai import AuthenticationError, OpenAIError

try:
    models = client.models.list()
    print("✅ API raktas veikia! Prisijungimas sėkmingas.")
except AuthenticationError:
    print("❌ API raktas neteisingas arba nebegalioja.")
except OpenAIError as e:
    print("⚠️ Nepavyko patikrinti prisijungimo.")
    print("Klaidos tipas:", type(e).__name__)

In [ ]:
from openai import OpenAIError

try:
    model = client.models.retrieve("gpt-5.4-mini")
    print("✅ Prieiga prie modelio patvirtinta:", model.id)
except OpenAIError as e:
    print("Klaidos tipas:", type(e).__name__)
    print("Klaidos kodas:", getattr(e, "code", None))
    print("HTTP statusas:", getattr(e, "status_code", None))
    print("Klaidos pranešimas:", str(e)[:500])

In [ ]:
from openai import OpenAIError

try:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="Reply with exactly: OK",
        max_output_tokens=10
    )
    print("✅ Responses API veikia!")
    print("Atsakymas:", response.output_text)

except OpenAIError as e:
    print("❌ Užklausa nepavyko")
    print("Klaidos tipas:", type(e).__name__)
    print("HTTP statusas:", getattr(e, "status_code", None))
    print("Pranešimas:", str(e)[:600])

In [ ]:
from openai import OpenAIError

try:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="Reply with exactly: OK",
        max_output_tokens=32
    )

    print("✅ API raktas veikia!")
    print("Modelio atsakymas:", response.output_text)

except OpenAIError as e:
    print("❌ Užklausa nepavyko")
    print("Klaidos tipas:", type(e).__name__)
    print("HTTP statusas:", getattr(e, "status_code", None))
    print("Pranešimas:", str(e)[:600])

In [ ]:
# Ankstesnio eksperimento testavimo pozicijos
test_id_mapping = pd.DataFrame({
    "original_id": test_full.index,
    "old_test_row_id": range(len(test_full))
})

# Susiejame naujos imties ID su ankstesniais
test_3000_context = test_3000_context.merge(
    test_id_mapping,
    left_on="test_row_id",
    right_on="original_id",
    how="left",
    validate="one_to_one"
)

# Patikriname ankstesnių GPT prognozių sutapimus
overlap = set(test_3000_context["old_test_row_id"]) & set(
    previous_results["test_row_id"]
)

print("Sutampančių ankstesnių GPT prognozių:", len(overlap))
print("Naujų GPT užklausų be konteksto reikėtų:",
      3000 - len(overlap))

# Patikriname XGBoost tikrųjų klasių atitikimą
matched = test_3000_context.merge(
    xgb_saved[["test_row_id", "y_true"]],
    left_on="old_test_row_id",
    right_on="test_row_id",
    how="left"
)

print("XGBoost klasių sutapimas:",
      (matched["Is Laundering"] == matched["y_true"]).all())

print("Trūkstamų ankstesnių ID:",
      test_3000_context["old_test_row_id"].isna().sum())

In [ ]:
import joblib
import numpy as np

preprocessor = joblib.load("aml_models/preprocessor.joblib")
xgb_model = joblib.load("aml_models/xgb_model.joblib")

# Paimame 20 operacijų
check = matched.sample(n=20, random_state=42).copy()

# Naudojame tuos pačius požymius kaip ankstesniame eksperimente
check["Timestamp_sec"] = (
    check["Timestamp"] - pd.Timestamp("2022-09-01")
).dt.total_seconds()

features = list(preprocessor.feature_names_in_)

X_check = preprocessor.transform(check[features])
new_probabilities = xgb_model.predict_proba(X_check)[:, 1]

# Palyginame su išsaugotomis prognozėmis
saved_probabilities = check["xgb_probability"].to_numpy() if (
    "xgb_probability" in check.columns
) else check["old_test_row_id"].map(
    xgb_saved.set_index("test_row_id")["xgb_probability"]
).to_numpy()

print("Didžiausias prognozių skirtumas:",
      np.max(np.abs(new_probabilities - saved_probabilities)))

print("Visos prognozės sutampa:",
      np.allclose(new_probabilities, saved_probabilities, atol=1e-6))

In [ ]:
# Pridedame ankstesnes XGBoost prognozes
comparison_3000 = test_3000_context.merge(
    xgb_saved[
        ["test_row_id", "xgb_probability", "xgb_prediction"]
    ],
    left_on="old_test_row_id",
    right_on="test_row_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_xgb")
)

# Patikriname duomenų vientisumą
assert len(comparison_3000) == 3000
assert comparison_3000["old_test_row_id"].is_unique
assert comparison_3000["xgb_probability"].notna().all()
assert comparison_3000["xgb_prediction"].notna().all()

# Išsaugome imtį
path = "aml_models/comparison_3000_context.csv"
comparison_3000.to_csv(path, index=False)

print("Išsaugota:", path)
print("Operacijų:", len(comparison_3000))
print("Pinigų plovimo atvejų:",
      int(comparison_3000["Is Laundering"].sum()))

print("\nXGBoost prognozių pasiskirstymas:")
print(comparison_3000["xgb_prediction"].value_counts())

print("\nXGBoost aptikti pinigų plovimo atvejai:")
print(
    ((comparison_3000["Is Laundering"] == 1) &
     (comparison_3000["xgb_prediction"] == 1)).sum()
)

In [ ]:
import json

base_features = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1",
    "Amount Received", "Receiving Currency",
    "Amount Paid", "Payment Currency",
    "Payment Format"
]

context_features = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients"
]

# Viena operacija užklausų formato patikrai
example = comparison_3000.iloc[0]

def prepare_transaction(row, features):
    result = {}
    for feature in features:
        value = row[feature]
        if isinstance(value, pd.Timestamp):
            value = value.strftime("%Y-%m-%d %H:%M:%S")
        elif hasattr(value, "item"):
            value = value.item()
        result[feature] = value
    return result

without_context = prepare_transaction(example, base_features)
with_context = prepare_transaction(
    example, base_features + context_features
)

print("BE KONTEKSTO:")
print(json.dumps(without_context, indent=2, ensure_ascii=False))

print("\nSU KONTEKSTU:")
print(json.dumps(with_context, indent=2, ensure_ascii=False))

In [ ]:
import json
import time

context_description = """
Additional historical features:
- previous_24h_count: Number of earlier outgoing transactions
  from the sender account during the preceding 24 hours.
- previous_7d_count: Number of earlier outgoing transactions
  from the sender account during the preceding 7 days.
- previous_24h_amount: Total amount paid by the sender account
  during the preceding 24 hours, in the current payment currency.
- previous_7d_unique_recipients: Number of distinct recipient
  bank-account pairs during the preceding 7 days.

Historical features exclude the current transaction and
transactions at the same or later timestamp.
"""

pilot_rows = comparison_3000.head(5)
pilot_results = []

for _, row in pilot_rows.iterrows():
    for variant in ["without_context", "with_context"]:
        features = (
            base_features if variant == "without_context"
            else base_features + context_features
        )

        transaction = prepare_transaction(row, features)

        instructions = prompt
        if variant == "with_context":
            instructions += "\n" + context_description

        start = time.time()

        try:
            response = client.responses.create(
                model="gpt-5.4-mini",
                instructions=instructions,
                input=json.dumps(transaction, ensure_ascii=False),
                max_output_tokens=150
            )

            result = json.loads(response.output_text)

            pilot_results.append({
                "old_test_row_id": row["old_test_row_id"],
                "variant": variant,
                "prediction": result["prediction"],
                "reason": result["reason"],
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "seconds": round(time.time() - start, 2),
                "error": ""
            })

        except Exception as e:
            pilot_results.append({
                "old_test_row_id": row["old_test_row_id"],
                "variant": variant,
                "prediction": None,
                "reason": "",
                "input_tokens": None,
                "output_tokens": None,
                "seconds": round(time.time() - start, 2),
                "error": str(e)[:200]
            })

        # Išsaugome po kiekvienos užklausos
        pd.DataFrame(pilot_results).to_csv(
            "aml_models/gpt_context_pilot.csv",
            index=False
        )

pilot_df = pd.DataFrame(pilot_results)

print(pilot_df[
    ["old_test_row_id", "variant", "prediction",
     "input_tokens", "output_tokens", "seconds", "error"]
].to_string(index=False))

print("\nIš viso įvesties tokenų:",
      pilot_df["input_tokens"].sum())
print("Iš viso išvesties tokenų:",
      pilot_df["output_tokens"].sum())

In [ ]:
# Sujungiame abiejų GPT variantų prognozes
pilot_comparison = pilot_df.pivot(
    index="old_test_row_id",
    columns="variant",
    values="prediction"
).reset_index()

# Pridedame tikrąsias klases ir XGBoost prognozes
pilot_comparison = pilot_comparison.merge(
    comparison_3000[
        ["old_test_row_id", "Is Laundering", "xgb_prediction"]
    ],
    on="old_test_row_id",
    how="left",
    validate="one_to_one"
)

print(pilot_comparison.to_string(index=False))

print("\nTeisingų prognozių skaičius:")
for column in ["without_context", "with_context", "xgb_prediction"]:
    correct = (
        pilot_comparison[column] ==
        pilot_comparison["Is Laundering"]
    ).sum()

    print(f"{column}: {correct} iš {len(pilot_comparison)}")

In [ ]:
for transaction_id in pilot_comparison["old_test_row_id"]:
    print(f"\n{'=' * 65}")
    print(f"OPERACIJA: {transaction_id}")

    row = comparison_3000.loc[
        comparison_3000["old_test_row_id"] == transaction_id
    ].iloc[0]

    print("24 val. operacijų:", row["previous_24h_count"])
    print("7 dienų operacijų:", row["previous_7d_count"])
    print("24 val. suma:", row["previous_24h_amount"])
    print("7 dienų gavėjų:", row["previous_7d_unique_recipients"])

    for variant in ["without_context", "with_context"]:
        result = pilot_df.loc[
            (pilot_df["old_test_row_id"] == transaction_id) &
            (pilot_df["variant"] == variant)
        ].iloc[0]

        print(f"\n{variant}: {result['prediction']}")
        print("Paaiškinimas:", result["reason"])

In [ ]:
from pathlib import Path
import hashlib

folder = Path("aml_models")

# Išsaugome abiejų variantų instrukcijas
(folder / "gpt_3000_base_prompt.txt").write_text(
    prompt, encoding="utf-8"
)

(folder / "gpt_3000_context_prompt.txt").write_text(
    prompt + "\n" + context_description,
    encoding="utf-8"
)

# Išsaugome bandomojo eksperimento rezultatus
pilot_df.to_csv(
    folder / "gpt_3000_pilot_predictions.csv",
    index=False
)

for name in ["base", "context"]:
    path = folder / f"gpt_3000_{name}_prompt.txt"
    content = path.read_bytes()

    print(f"{name} prompt SHA-256:",
          hashlib.sha256(content).hexdigest())

print("\nIšsaugotų bandomųjų prognozių:", len(pilot_df))
print("Klaidų:", pilot_df["error"].fillna("").ne("").sum())

In [ ]:
from pathlib import Path
import pandas as pd

results_path = Path("aml_models/gpt_3000_results.csv")

# Jei pagrindinio eksperimento failas jau yra, jo neperrašome
if results_path.exists():
    results = pd.read_csv(results_path)
    print("Rastas ankstesnis rezultatų failas.")
else:
    results = pd.read_csv("aml_models/gpt_3000_pilot_predictions.csv")
    results.to_csv(results_path, index=False)
    print("Sukurtas rezultatų failas iš bandomojo eksperimento.")

# Sėkmingai atliktas užklausas identifikuojame pagal operaciją ir variantą
successful = results[
    results["error"].fillna("").eq("") &
    results["prediction"].isin([0, 1])
]

completed = set(
    zip(successful["old_test_row_id"].astype(int),
        successful["variant"])
)

all_tasks = [
    (int(row_id), variant)
    for row_id in comparison_3000["old_test_row_id"]
    for variant in ["without_context", "with_context"]
]

remaining = [task for task in all_tasks if task not in completed]

print("Visų suplanuotų užklausų:", len(all_tasks))
print("Sėkmingai atlikta:", len(completed))
print("Liko atlikti:", len(remaining))
print("Rezultatų failas:", results_path)

In [ ]:
import time
import json
import pandas as pd

results_path = "aml_models/gpt_3000_results.csv"
results = pd.read_csv(results_path)

# Tik sėkmingai atliktos užklausos
completed = set(
    zip(
        results.loc[
            results["error"].fillna("").eq("") &
            results["prediction"].isin([0, 1]),
            "old_test_row_id"
        ].astype(int),
        results.loc[
            results["error"].fillna("").eq("") &
            results["prediction"].isin([0, 1]),
            "variant"
        ]
    )
)

# Paruošiame pirmas 20 dar neatliktų užklausų
tasks = []

for _, row in comparison_3000.iterrows():
    for variant in ["without_context", "with_context"]:
        key = (int(row["old_test_row_id"]), variant)

        if key not in completed:
            tasks.append((row, variant))

        if len(tasks) == 20:
            break

    if len(tasks) == 20:
        break

for row, variant in tasks:
    features = (
        base_features if variant == "without_context"
        else base_features + context_features
    )

    instructions = (
        prompt if variant == "without_context"
        else prompt + "\n" + context_description
    )

    transaction = prepare_transaction(row, features)
    start = time.time()

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=instructions,
            input=json.dumps(transaction, ensure_ascii=False),
            max_output_tokens=150
        )

        answer = json.loads(response.output_text)
        prediction = int(answer["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Netinkama prognozės reikšmė")

        record = {
            "old_test_row_id": int(row["old_test_row_id"]),
            "variant": variant,
            "prediction": prediction,
            "reason": answer.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "seconds": round(time.time() - start, 2),
            "error": ""
        }

    except Exception as e:
        record = {
            "old_test_row_id": int(row["old_test_row_id"]),
            "variant": variant,
            "prediction": None,
            "reason": "",
            "input_tokens": None,
            "output_tokens": None,
            "seconds": round(time.time() - start, 2),
            "error": str(e)[:200]
        }

    # Išsaugome po kiekvienos užklausos
    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )
    results.to_csv(results_path, index=False)

print("Šioje partijoje atlikta užklausų:", len(tasks))
print("Iš viso išsaugota įrašų:", len(results))

print("\nPaskutinių 20 užklausų rezultatai:")
print(
    results.tail(20)[
        ["variant", "prediction", "input_tokens",
         "output_tokens", "seconds", "error"]
    ].to_string(index=False)
)

print("\nBendras įvesties tokenų skaičius:",
      results["input_tokens"].sum())
print("Bendras išvesties tokenų skaičius:",
      results["output_tokens"].sum())

In [ ]:
# Vidutinės tokenų sąnaudos pagal GPT variantą
summary = results.groupby("variant").agg(
    requests=("prediction", "count"),
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_seconds=("seconds", "mean"),
    positive_predictions=("prediction", "sum")
)

summary["estimated_total_input"] = (
    summary["avg_input_tokens"] * 3000
).round()

summary["estimated_total_output"] = (
    summary["avg_output_tokens"] * 3000
).round()

print(summary.to_string())

print("\nPROGNOZUOJAMOS VISO EKSPERIMENTO SĄNAUDOS:")
print("Įvesties tokenų:",
      int(summary["estimated_total_input"].sum()))
print("Išvesties tokenų:",
      int(summary["estimated_total_output"].sum()))

print("\nNumatoma vykdymo trukmė (val.):",
      round(summary["avg_seconds"].sum() * 3000 / 3600, 2))

In [ ]:
# Sujungiame pirmųjų 15 operacijų prognozes
early_comparison = results.pivot(
    index="old_test_row_id",
    columns="variant",
    values="prediction"
).reset_index()

early_comparison = early_comparison.merge(
    comparison_3000[
        ["old_test_row_id", "Is Laundering", "xgb_prediction"]
    ],
    on="old_test_row_id",
    how="left",
    validate="one_to_one"
)

print("Operacijų:", len(early_comparison))
print("Tikrų pinigų plovimo atvejų:",
      int(early_comparison["Is Laundering"].sum()))

for model in ["without_context", "with_context", "xgb_prediction"]:
    tp = (
        (early_comparison[model] == 1) &
        (early_comparison["Is Laundering"] == 1)
    ).sum()

    fp = (
        (early_comparison[model] == 1) &
        (early_comparison["Is Laundering"] == 0)
    ).sum()

    print(f"\n{model}")
    print("Teisingai aptikti atvejai (TP):", tp)
    print("Klaidingi įspėjimai (FP):", fp)

In [ ]:
# Sudarome atskirą diagnostinę validavimo imtį
validation_100 = pd.concat([
    validation_full[
        validation_full["Is Laundering"] == 0
    ].sample(n=50, random_state=123),

    validation_full[
        validation_full["Is Laundering"] == 1
    ].sample(n=50, random_state=123)
]).sample(frac=1, random_state=123).copy()

validation_100["original_id"] = validation_100.index

print("Validavimo imties dydis:", len(validation_100))
print("\nKlasių pasiskirstymas:")
print(validation_100["Is Laundering"].value_counts())

print("\nDatų intervalas:")
print(
    validation_100["Timestamp"].min(),
    "–",
    validation_100["Timestamp"].max()
)

In [ ]:
# Pridedame 24 val. ir 7 dienų operacijų skaičius
validation_100_context = validation_100.merge(
    counts[
        ["From Bank", "Account", "Timestamp",
         "previous_24h_count", "previous_7d_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

# Pridedame ankstesnių 24 val. sumą ta pačia valiuta
validation_100_context = validation_100_context.merge(
    amounts[
        ["From Bank", "Account", "Payment Currency",
         "Timestamp", "previous_24h_amount"]
    ],
    on=["From Bank", "Account", "Payment Currency", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų:", len(validation_100_context))

print("\nTrūkstamos reikšmės:")
print(validation_100_context[
    ["previous_24h_count", "previous_7d_count",
     "previous_24h_amount"]
].isna().sum())

print("\nKontekstinių požymių statistika:")
print(validation_100_context[
    ["previous_24h_count", "previous_7d_count",
     "previous_24h_amount"]
].describe())

In [ ]:
import numpy as np
import pandas as pd

# Atrenkame tik sąskaitas, esančias validavimo imtyje
sender_keys = pd.MultiIndex.from_frame(
    validation_100_context[["From Bank", "Account"]].drop_duplicates()
)

history_keys = pd.MultiIndex.from_frame(
    df[["From Bank", "Account"]]
)

history = df.loc[
    history_keys.isin(sender_keys),
    ["From Bank", "Account", "Timestamp", "To Bank", "Account.1"]
].copy()

history = history.sort_values("Timestamp")

# Apskaičiuojame unikalius gavėjus kiekvienai validavimo operacijai
unique_recipients = []

for _, row in validation_100_context.iterrows():
    sender_history = history[
        (history["From Bank"] == row["From Bank"]) &
        (history["Account"] == row["Account"]) &
        (history["Timestamp"] < row["Timestamp"]) &
        (
            history["Timestamp"] >=
            row["Timestamp"] - pd.Timedelta(days=7)
        )
    ]

    count = sender_history[
        ["To Bank", "Account.1"]
    ].drop_duplicates().shape[0]

    unique_recipients.append(count)

validation_100_context[
    "previous_7d_unique_recipients"
] = unique_recipients

# Išsaugome validavimo duomenis
validation_100_context.to_csv(
    "aml_models/validation_100_context.csv",
    index=False
)

print("Operacijų:", len(validation_100_context))

print("\nKetvirtojo požymio statistika:")
print(
    validation_100_context[
        "previous_7d_unique_recipients"
    ].describe()
)

print("\nTrūkstamos reikšmės visuose 4 požymiuose:")
print(
    validation_100_context[context_features].isna().sum()
)

print("\nFailas išsaugotas: aml_models/validation_100_context.csv")

In [ ]:
import numpy as np
import pandas as pd

check_rows = validation_100_context.sample(
    n=5, random_state=42
)

checks = []

for _, row in check_rows.iterrows():
    sender_history = df[
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] < row["Timestamp"])
    ]

    last_24h = sender_history[
        sender_history["Timestamp"] >=
        row["Timestamp"] - pd.Timedelta(hours=24)
    ]

    last_7d = sender_history[
        sender_history["Timestamp"] >=
        row["Timestamp"] - pd.Timedelta(days=7)
    ]

    same_currency = last_24h[
        last_24h["Payment Currency"] == row["Payment Currency"]
    ]

    expected = {
        "previous_24h_count": len(last_24h),
        "previous_7d_count": len(last_7d),
        "previous_24h_amount": same_currency["Amount Paid"].sum(),
        "previous_7d_unique_recipients": last_7d[
            ["To Bank", "Account.1"]
        ].drop_duplicates().shape[0]
    }

    for feature, calculated_value in expected.items():
        saved_value = row[feature]

        match = np.isclose(
            saved_value,
            calculated_value,
            rtol=1e-9,
            atol=1e-5
        )

        checks.append({
            "original_id": row["original_id"],
            "feature": feature,
            "saved": saved_value,
            "recalculated": calculated_value,
            "match": match
        })

check_df = pd.DataFrame(checks)

print(check_df.to_string(index=False))

print("\nPatikrintų reikšmių:", len(check_df))
print("Sutampa:", int(check_df["match"].sum()))
print("Nesutampa:", int((~check_df["match"]).sum()))

In [ ]:
import pandas as pd

sample_sizes = [3000, 5000, 10000, 20000, 50000]

rows = []

for n in sample_sizes:
    sample = test_full.sample(n=n, random_state=42)

    aml_count = int(sample["Is Laundering"].sum())

    rows.append({
        "Imties dydis": n,
        "Teisėtos operacijos": n - aml_count,
        "Pinigų plovimo atvejai": aml_count,
        "Pinigų plovimo dalis (%)": round(aml_count / n * 100, 4)
    })

sample_comparison = pd.DataFrame(rows)

print(sample_comparison.to_string(index=False))

In [ ]:
# Paimame jau gautas GPT prognozes su kontekstu
context_results = results[
    (results["variant"] == "with_context") &
    (results["error"].fillna("") == "")
][["old_test_row_id", "prediction"]].copy()

# Sujungiame su operacijų kontekstiniais požymiais
context_analysis = comparison_3000.merge(
    context_results,
    on="old_test_row_id",
    how="inner",
    validate="one_to_one"
)

print("Įvertintų operacijų:", len(context_analysis))

print("\nKontekstiniai požymiai pagal GPT prognozę:")
print(
    context_analysis.groupby("prediction")[context_features]
    .agg(["median", "min", "max"])
    .round(2)
    .to_string()
)

print("\nGPT prognozių pasiskirstymas:")
print(context_analysis["prediction"].value_counts())

In [ ]:
validation_ids = set(validation_100_context["original_id"].astype(int))
test_ids = set(comparison_3000["test_row_id"].astype(int))

overlap = validation_ids.intersection(test_ids)

print("Validavimo operacijų:", len(validation_ids))
print("Testavimo operacijų:", len(test_ids))
print("Sutampančių operacijų:", len(overlap))

print("\nValidavimo laikotarpis:")
print(validation_100_context["Timestamp"].min(),
      "–", validation_100_context["Timestamp"].max())

print("\nTestavimo laikotarpis:")
print(comparison_3000["Timestamp"].min(),
      "–", comparison_3000["Timestamp"].max())

In [ ]:
from pathlib import Path
import hashlib

folder = Path("aml_models")

base_prompt = (
    folder / "gpt_3000_base_prompt.txt"
).read_text(encoding="utf-8")

context_prompt_v2 = base_prompt + """

Additional historical features:
- previous_24h_count: Earlier outgoing transactions in the last 24 hours.
- previous_7d_count: Earlier outgoing transactions in the last 7 days.
- previous_24h_amount: Total earlier outgoing amount in the last 24 hours,
  measured in the current payment currency.
- previous_7d_unique_recipients: Distinct recipient bank-account pairs
  in the last 7 days.

Historical features exclude the current transaction and transactions
at the same or later timestamp.

Interpretation guidelines:
- High transaction frequency or high total amount alone is not
  sufficient evidence of money laundering risk.
- Some accounts legitimately process many transactions.
- Evaluate historical activity together with the current transaction.
- Consider whether the current transaction is unusual relative to
  the available account history, when such a comparison is possible.
- Do not assume that multiple recipients automatically indicate risk.
- Do not invent information about the customer's business,
  expected behavior, or identity.
- If the available evidence is inconclusive, make the best
  risk classification supported by the provided attributes.
"""

path_v2 = folder / "gpt_context_prompt_v2.txt"

# Neleidžiame netyčia perrašyti jau išsaugotos versijos
if path_v2.exists():
    assert path_v2.read_text(encoding="utf-8") == context_prompt_v2, (
        "Failas jau egzistuoja, bet jo turinys skiriasi."
    )
else:
    path_v2.write_text(context_prompt_v2, encoding="utf-8")

print("Naujas promptas išsaugotas:", path_v2)
print("SHA-256:", hashlib.sha256(path_v2.read_bytes()).hexdigest())
print("Simbolių skaičius:", len(context_prompt_v2))

print("\nPradinis kontekstinis promptas išsaugotas atskirai:",
      (folder / "gpt_3000_context_prompt.txt").exists())

In [ ]:
# Natūrali, atsitiktinė validavimo imtis
validation_natural_100 = validation_full.sample(
    n=100,
    random_state=2026
).copy()

validation_natural_100["original_id"] = validation_natural_100.index

print("Operacijų:", len(validation_natural_100))

print("\nKlasių pasiskirstymas:")
print(validation_natural_100["Is Laundering"].value_counts())

print("\nPinigų plovimo dalis (%):")
print(
    round(
        validation_natural_100["Is Laundering"].mean() * 100,
        4
    )
)

print("\nViso validavimo rinkinio pinigų plovimo dalis (%):")
print(
    round(validation_full["Is Laundering"].mean() * 100, 4)
)

In [ ]:
validation_counts = (
    validation_full["Is Laundering"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

print("Visas validavimo rinkinys:", len(validation_full))
print("Teisėtos operacijos:", validation_counts[0])
print("Pinigų plovimo operacijos:", validation_counts[1])

print("\nKlasių proporcijos (%):")
print((validation_counts / len(validation_full) * 100).round(4))

print("\nOperacijų skaičius vienam pinigų plovimo atvejui:")
print(round(len(validation_full) / validation_counts[1]))

In [ ]:
# Atskiros grupės FPR ir Recall vertinimui

validation_legitimate = validation_natural_100.copy()

validation_laundering = validation_full[
    validation_full["Is Laundering"] == 1
].sample(n=100, random_state=2026).copy()

validation_laundering["original_id"] = validation_laundering.index

print("FPR vertinimo grupė:")
print("Teisėtos operacijos:", len(validation_legitimate))
print("Pinigų plovimo operacijos:",
      int(validation_legitimate["Is Laundering"].sum()))

print("\nRecall vertinimo grupė:")
print("Pinigų plovimo operacijos:", len(validation_laundering))

print("\nSutampančių operacijų:",
      len(
          set(validation_legitimate["original_id"]) &
          set(validation_laundering["original_id"])
      ))

print("\nTikrasis AML paplitimas validavimo rinkinyje:",
      round(validation_full["Is Laundering"].mean() * 100, 4), "%")

In [ ]:
import pandas as pd
from pathlib import Path

validation_200 = pd.concat(
    [validation_legitimate, validation_laundering],
    ignore_index=True
).copy()

# Tikriname, ar operacijos nesikartoja
assert len(validation_200) == 200
assert validation_200["original_id"].is_unique

# Istorijos ieškosime tik reikalingoms siuntėjų sąskaitoms
sender_keys = pd.MultiIndex.from_frame(
    validation_200[["From Bank", "Account"]].drop_duplicates()
)

df_sender_keys = pd.MultiIndex.from_frame(
    df[["From Bank", "Account"]]
)

history = df.loc[
    df_sender_keys.isin(sender_keys),
    [
        "From Bank", "Account", "Timestamp",
        "To Bank", "Account.1",
        "Amount Paid", "Payment Currency"
    ]
].copy()

context_records = []

for _, row in validation_200.iterrows():
    sender_history = history[
        (history["From Bank"] == row["From Bank"]) &
        (history["Account"] == row["Account"]) &
        (history["Timestamp"] < row["Timestamp"])
    ]

    last_24h = sender_history[
        sender_history["Timestamp"] >=
        row["Timestamp"] - pd.Timedelta(hours=24)
    ]

    last_7d = sender_history[
        sender_history["Timestamp"] >=
        row["Timestamp"] - pd.Timedelta(days=7)
    ]

    same_currency = last_24h[
        last_24h["Payment Currency"] == row["Payment Currency"]
    ]

    context_records.append({
        "previous_24h_count": len(last_24h),
        "previous_7d_count": len(last_7d),
        "previous_24h_amount": same_currency["Amount Paid"].sum(),
        "previous_7d_unique_recipients": last_7d[
            ["To Bank", "Account.1"]
        ].drop_duplicates().shape[0]
    })

validation_200 = pd.concat(
    [
        validation_200.reset_index(drop=True),
        pd.DataFrame(context_records)
    ],
    axis=1
)

# Patikriname ir išsaugome
assert validation_200[context_features].notna().all().all()

output_path = Path("aml_models/validation_200_context.csv")
validation_200.to_csv(output_path, index=False)

print("Operacijų:", len(validation_200))
print("\nKlasių pasiskirstymas:")
print(validation_200["Is Laundering"].value_counts())

print("\nTrūkstamos kontekstinės reikšmės:")
print(validation_200[context_features].isna().sum())

print("\nFailas:", output_path)

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path

folder = Path("aml_models")
results_path = folder / "gpt_context_validation.csv"

prompts = {
    "V1": (folder / "gpt_3000_context_prompt.txt").read_text(encoding="utf-8"),
    "V2": (folder / "gpt_context_prompt_v2.txt").read_text(encoding="utf-8")
}

# Po 5 operacijas iš kiekvienos klasės
batch = pd.concat([
    validation_200[validation_200["Is Laundering"] == 0].head(5),
    validation_200[validation_200["Is Laundering"] == 1].head(5)
])

if results_path.exists():
    validation_results = pd.read_csv(results_path)
else:
    validation_results = pd.DataFrame()

completed = set()

if not validation_results.empty:
    successful = validation_results[
        validation_results["error"].fillna("").eq("") &
        validation_results["prediction"].isin([0, 1])
    ]
    completed = set(zip(
        successful["original_id"].astype(int),
        successful["variant"]
    ))

new_requests = 0

for _, row in batch.iterrows():
    for variant in ["V1", "V2"]:
        key = (int(row["original_id"]), variant)

        if key in completed:
            continue

        transaction = prepare_transaction(
            row, base_features + context_features
        )

        start = time.time()

        try:
            response = client.responses.create(
                model="gpt-5.4-mini",
                instructions=prompts[variant],
                input=json.dumps(transaction, ensure_ascii=False),
                max_output_tokens=150
            )

            answer = json.loads(response.output_text)
            prediction = int(answer["prediction"])

            if prediction not in (0, 1):
                raise ValueError("Prognozė turi būti 0 arba 1")

            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": prediction,
                "reason": answer.get("reason", ""),
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "seconds": round(time.time() - start, 2),
                "error": ""
            }

        except Exception as e:
            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": None,
                "reason": "",
                "input_tokens": None,
                "output_tokens": None,
                "seconds": round(time.time() - start, 2),
                "error": str(e)[:200]
            }

        validation_results = pd.concat(
            [validation_results, pd.DataFrame([record])],
            ignore_index=True
        )

        validation_results.to_csv(results_path, index=False)
        new_requests += 1

print("Naujų API užklausų:", new_requests)
print("Iš viso išsaugota rezultatų:", len(validation_results))

print("\nPrognozės pagal tikrąją klasę:")
print(
    validation_results[
        validation_results["error"].fillna("").eq("")
    ].groupby(["variant", "y_true"])["prediction"]
    .agg(["count", "sum"])
    .to_string()
)

print("\nKlaidų:",
      validation_results["error"].fillna("").ne("").sum())

In [ ]:
import pandas as pd

path = "aml_models/gpt_context_validation.csv"
validation_results = pd.read_csv(path)

analysis = validation_results[
    validation_results["error"].fillna("").eq("")
].copy()

analysis = analysis.merge(
    validation_200[
        ["original_id"] + context_features
    ],
    on="original_id",
    how="left",
    validate="many_to_one"
)

columns = [
    "original_id",
    "y_true",
    "variant",
    "prediction",
    "previous_24h_count",
    "previous_7d_count",
    "previous_7d_unique_recipients",
    "reason"
]

print(
    analysis[columns]
    .sort_values(["y_true", "original_id", "variant"])
    .to_string(index=False, max_colwidth=150)
)

In [ ]:
import numpy as np
import pandas as pd

# Ankstesnių 24 valandų vidutinė operacijos suma
# Pastaba: skaičius apima visas valiutas, todėl šis santykis
# yra tik diagnostinis ir nėra tinkamas galutiniam modeliui.
validation_200["historical_average_24h"] = np.where(
    validation_200["previous_24h_count"] > 0,
    validation_200["previous_24h_amount"] /
    validation_200["previous_24h_count"].replace(0, np.nan),
    np.nan
)

print("Operacijų:", len(validation_200))
print("\nDabartinės sumos statistika pagal tikrąją klasę:")
print(
    validation_200.groupby("Is Laundering")["Amount Paid"]
    .describe()[["count", "50%", "mean", "max"]]
    .round(2)
)

print("\nOperacijų be ankstesnės 24 val. istorijos:")
print(
    validation_200.groupby("Is Laundering")[
        "previous_24h_count"
    ].apply(lambda x: int((x == 0).sum()))
)

In [ ]:
# Sujungiame validavimo operacijas su istorinio aktyvumo duomenimis
validation_activity = validation_full.merge(
    counts[
        ["From Bank", "Account", "Timestamp",
         "previous_24h_count"]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

assert len(validation_activity) == len(validation_full)

print("Validavimo operacijų:", len(validation_activity))
print("Trūkstamų istorinio aktyvumo reikšmių:",
      validation_activity["previous_24h_count"].isna().sum())

print("\nOperacijos be ankstesnio 24 val. aktyvumo:")

summary = validation_activity.groupby("Is Laundering").agg(
    operaciju_skaicius=("previous_24h_count", "size"),
    be_ankstesniu_operaciju=(
        "previous_24h_count",
        lambda x: int((x == 0).sum())
    ),
    aktyvumo_mediana=("previous_24h_count", "median")
)

summary["be_ankstesniu_operaciju_pct"] = (
    summary["be_ankstesniu_operaciju"] /
    summary["operaciju_skaicius"] * 100
).round(2)

print(summary.to_string())

In [ ]:
import pandas as pd

validation_activity["no_previous_24h"] = (
    validation_activity["previous_24h_count"] == 0
)

risk_table = validation_activity.groupby(
    "no_previous_24h"
).agg(
    total_transactions=("Is Laundering", "size"),
    laundering_transactions=("Is Laundering", "sum")
)

risk_table["laundering_rate_pct"] = (
    risk_table["laundering_transactions"] /
    risk_table["total_transactions"] * 100
)

risk_table.index = risk_table.index.map({
    False: "Buvo ankstesnių operacijų",
    True: "Nebuvo ankstesnių operacijų"
})

print(risk_table.round(4).to_string())

rate_no_history = risk_table.loc[
    "Nebuvo ankstesnių operacijų", "laundering_rate_pct"
]

rate_with_history = risk_table.loc[
    "Buvo ankstesnių operacijų", "laundering_rate_pct"
]

print(
    "\nAML dažnio santykis:",
    round(rate_no_history / rate_with_history, 2)
)

In [ ]:
import pandas as pd

# Ankstesnių 24 valandų operacijų skaičiaus intervalai
bins = [-1, 0, 1, 2, 5, 10, 25, 100, float("inf")]
labels = ["0", "1", "2", "3–5", "6–10", "11–25", "26–100", ">100"]

validation_activity["activity_group"] = pd.cut(
    validation_activity["previous_24h_count"],
    bins=bins,
    labels=labels
)

activity_risk = (
    validation_activity
    .groupby("activity_group", observed=True)
    .agg(
        total=("Is Laundering", "size"),
        aml_cases=("Is Laundering", "sum")
    )
)

activity_risk["aml_rate_pct"] = (
    activity_risk["aml_cases"] /
    activity_risk["total"] * 100
)

activity_risk["share_of_all_aml_pct"] = (
    activity_risk["aml_cases"] /
    validation_activity["Is Laundering"].sum() * 100
)

print(activity_risk.round(4).to_string())

In [ ]:
import pandas as pd

validation_results = pd.read_csv(
    "aml_models/gpt_context_validation.csv"
)

analysis = validation_results.merge(
    validation_200[
        ["original_id", "previous_24h_count"]
    ],
    on="original_id",
    how="left",
    validate="many_to_one"
)

analysis = analysis[
    analysis["error"].fillna("").eq("")
].copy()

analysis["activity_group"] = pd.cut(
    analysis["previous_24h_count"],
    bins=[-1, 0, 2, 10, 100, float("inf")],
    labels=["0", "1–2", "3–10", "11–100", ">100"]
)

summary = (
    analysis.groupby(
        ["variant", "activity_group", "y_true"],
        observed=True
    )
    .agg(
        transactions=("prediction", "size"),
        predicted_aml=("prediction", "sum")
    )
)

print(summary.to_string())

In [ ]:
from pathlib import Path
import hashlib

folder = Path("aml_models")

base_prompt = (
    folder / "gpt_3000_base_prompt.txt"
).read_text(encoding="utf-8")

context_description_v3 = """
Additional historical features:
- previous_24h_count: Earlier outgoing transactions in the preceding 24 hours.
- previous_7d_count: Earlier outgoing transactions in the preceding 7 days.
- previous_24h_amount: Total earlier outgoing amount in the preceding
  24 hours, measured in the current payment currency.
- previous_7d_unique_recipients: Distinct recipient bank-account pairs
  in the preceding 7 days.

Historical features exclude the current transaction and transactions
at the same or later timestamp.

Interpretation guidelines:
- High transaction frequency alone does not establish money laundering risk.
- Absence of recent outgoing transactions does not establish money laundering risk.
- Evaluate the current transaction together with all available attributes.
- Do not invent customer profiles or expected transaction behavior.
- Consider both low and high historical activity without automatically
  treating either as suspicious.
- Do not assume that common payment formats or matching currencies
  guarantee that a transaction is legitimate.

Validation-data observations:
- In the validation period, transactions without earlier outgoing
  activity in the preceding 24 hours had a higher observed laundering
  rate than transactions with prior activity.
- Very high historical activity was also associated with elevated
  laundering rates compared with several intermediate activity groups.
- These are statistical associations in a synthetic dataset,
  not deterministic classification rules.
- Most transactions in every activity group were legitimate.
"""

prompt_v3 = base_prompt + "\n" + context_description_v3

path_v3 = folder / "gpt_context_prompt_v3.txt"

if path_v3.exists():
    assert path_v3.read_text(encoding="utf-8") == prompt_v3, (
        "V3 failas jau egzistuoja, tačiau jo turinys skiriasi."
    )
else:
    path_v3.write_text(prompt_v3, encoding="utf-8")

print("V3 išsaugotas:", path_v3)
print("SHA-256:", hashlib.sha256(path_v3.read_bytes()).hexdigest())
print("Simbolių skaičius:", len(prompt_v3))

print("\nAnkstesni promptai išliko:")
for name in [
    "gpt_3000_context_prompt.txt",
    "gpt_context_prompt_v2.txt"
]:
    print(name, (folder / name).exists())

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path

folder = Path("aml_models")
results_path = folder / "gpt_context_validation_v3.csv"

prompts = {
    "V1": (folder / "gpt_3000_context_prompt.txt").read_text(encoding="utf-8"),
    "V2": (folder / "gpt_context_prompt_v2.txt").read_text(encoding="utf-8"),
    "V3": (folder / "gpt_context_prompt_v3.txt").read_text(encoding="utf-8")
}

# Atmetame 10 anksčiau analizuotų operacijų
previous_results = pd.read_csv(folder / "gpt_context_validation.csv")
previous_ids = set(previous_results["original_id"].astype(int))

available = validation_200[
    ~validation_200["original_id"].isin(previous_ids)
].copy()

batch = pd.concat([
    available[available["Is Laundering"] == 0]
    .sample(n=5, random_state=2027),

    available[available["Is Laundering"] == 1]
    .sample(n=5, random_state=2027)
])

if results_path.exists():
    results_v3 = pd.read_csv(results_path)
else:
    results_v3 = pd.DataFrame()

completed = set()

if not results_v3.empty:
    successful = results_v3[
        results_v3["error"].fillna("").eq("") &
        results_v3["prediction"].isin([0, 1])
    ]
    completed = set(zip(
        successful["original_id"].astype(int),
        successful["variant"]
    ))

new_requests = 0

for _, row in batch.iterrows():
    for variant, instructions in prompts.items():

        key = (int(row["original_id"]), variant)

        if key in completed:
            continue

        transaction = prepare_transaction(
            row, base_features + context_features
        )

        start = time.time()

        try:
            response = client.responses.create(
                model="gpt-5.4-mini",
                instructions=instructions,
                input=json.dumps(transaction, ensure_ascii=False),
                max_output_tokens=150
            )

            answer = json.loads(response.output_text)
            prediction = int(answer["prediction"])

            if prediction not in (0, 1):
                raise ValueError("Prognozė turi būti 0 arba 1")

            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": prediction,
                "reason": answer.get("reason", ""),
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "seconds": round(time.time() - start, 2),
                "error": ""
            }

        except Exception as e:
            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": None,
                "reason": "",
                "input_tokens": None,
                "output_tokens": None,
                "seconds": round(time.time() - start, 2),
                "error": str(e)[:200]
            }

        results_v3 = pd.concat(
            [results_v3, pd.DataFrame([record])],
            ignore_index=True
        )
        results_v3.to_csv(results_path, index=False)
        new_requests += 1

print("Naujų API užklausų:", new_requests)
print("Išsaugotų rezultatų:", len(results_v3))

print("\nPrognozės pagal tikrąją klasę:")
print(
    results_v3[
        results_v3["error"].fillna("").eq("")
    ].groupby(["variant", "y_true"])["prediction"]
    .agg(["count", "sum"])
    .to_string()
)

print("\nKlaidų:", results_v3["error"].fillna("").ne("").sum())

In [ ]:
import pandas as pd

v3_results = pd.read_csv(
    "aml_models/gpt_context_validation_v3.csv"
)

v3_analysis = v3_results[
    (v3_results["variant"] == "V3") &
    (v3_results["error"].fillna("") == "")
].merge(
    validation_200[
        ["original_id", "Amount Paid", "Payment Currency",
         "Payment Format"] + context_features
    ],
    on="original_id",
    how="left",
    validate="many_to_one"
)

print(
    v3_analysis[
        ["original_id", "y_true", "prediction",
         "Amount Paid", "Payment Currency",
         "Payment Format", "previous_24h_count",
         "previous_7d_count",
         "previous_7d_unique_recipients", "reason"]
    ].sort_values(["y_true", "original_id"])
    .to_string(index=False, max_colwidth=180)
)

In [ ]:
import pandas as pd
import re

v3 = pd.read_csv("aml_models/gpt_context_validation_v3.csv")
v3 = v3[
    (v3["variant"] == "V3") &
    (v3["error"].fillna("") == "")
].copy()

patterns = {
    "new_recipient": r"\bnew (recipient|beneficiary|counterparty)\b",
    "usual_behavior": r"\b(usual|typical|normal|expected) (behavior|activity|pattern)\b",
    "customer_profile": r"\b(customer|business|account) (profile|type|purpose)\b",
    "layering_claim": r"\blayering\b",
    "relative_amount_claim": r"\b(unusually|abnormally) (large|small|high|low)\b"
}

for name, pattern in patterns.items():
    v3[name] = v3["reason"].fillna("").str.contains(
        pattern, flags=re.IGNORECASE, regex=True
    )

print("Galimai nepagrįstų teiginių patikra:")
print(v3[list(patterns)].sum().to_string())

print("\nPaaiškinimai, kuriuos verta peržiūrėti:")
flagged = v3[v3[list(patterns)].any(axis=1)]

for _, row in flagged.iterrows():
    print(f"\nID: {row['original_id']} | Tikra: {row['y_true']} | GPT: {row['prediction']}")
    print("Požymiai:", [
        name for name in patterns if row[name]
    ])
    print("Paaiškinimas:", row["reason"])

print("\nIš viso pažymėtų paaiškinimų:", len(flagged))

In [ ]:
import pandas as pd

v3_rows = v3_analysis.copy()

results = []

for _, row in v3_rows.iterrows():
    current_time = pd.Timestamp(
        validation_200.loc[
            validation_200["original_id"] == row["original_id"],
            "Timestamp"
        ].iloc[0]
    )

    current = validation_200.loc[
        validation_200["original_id"] == row["original_id"]
    ].iloc[0]

    history = df[
        (df["From Bank"] == current["From Bank"]) &
        (df["Account"] == current["Account"]) &
        (df["Timestamp"] < current_time) &
        (df["Timestamp"] >= current_time - pd.Timedelta(days=7))
    ]

    same_recipient = history[
        (history["To Bank"] == current["To Bank"]) &
        (history["Account.1"] == current["Account.1"])
    ]

    results.append({
        "original_id": int(row["original_id"]),
        "y_true": int(row["y_true"]),
        "gpt_prediction": int(row["prediction"]),
        "previous_7d_count": len(history),
        "same_recipient_7d_count": len(same_recipient),
        "recipient_seen_before_7d": int(len(same_recipient) > 0)
    })

recipient_analysis = pd.DataFrame(results)

print(
    recipient_analysis.sort_values(
        ["y_true", "original_id"]
    ).to_string(index=False)
)

print("\nNaujų gavėjų per 7 dienas skaičius pagal klasę:")
print(
    recipient_analysis.groupby("y_true")[
        "recipient_seen_before_7d"
    ].agg(["count", "sum"])
)

In [ ]:
import pandas as pd

recipient_results = []

# Naudojame jau atrinktą validavimo imtį
for _, row in validation_200.iterrows():

    current_time = pd.Timestamp(row["Timestamp"])

    history = df.loc[
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] < current_time) &
        (df["Timestamp"] >= current_time - pd.Timedelta(days=7)),
        ["To Bank", "Account.1"]
    ]

    same_recipient_count = (
        (history["To Bank"] == row["To Bank"]) &
        (history["Account.1"] == row["Account.1"])
    ).sum()

    recipient_results.append({
        "original_id": int(row["original_id"]),
        "y_true": int(row["Is Laundering"]),
        "same_recipient_7d_count": int(same_recipient_count),
        "new_recipient_7d": int(same_recipient_count == 0)
    })

recipient_validation_200 = pd.DataFrame(recipient_results)

summary = recipient_validation_200.groupby("y_true").agg(
    total=("new_recipient_7d", "size"),
    new_recipients=("new_recipient_7d", "sum"),
    known_recipients=("new_recipient_7d", lambda x: (x == 0).sum())
)

summary["new_recipient_pct"] = (
    summary["new_recipients"] / summary["total"] * 100
)

print(summary.round(2).to_string())

recipient_validation_200.to_csv(
    "aml_models/validation_200_recipient_features.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd

recipient_check = validation_200[
    ["original_id", "Is Laundering", "previous_7d_count"]
].merge(
    recipient_validation_200[
        ["original_id", "new_recipient_7d"]
    ],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

recipient_check["activity_group"] = pd.cut(
    recipient_check["previous_7d_count"],
    bins=[-1, 0, 5, 25, 100, float("inf")],
    labels=["0", "1–5", "6–25", "26–100", ">100"]
)

summary = (
    recipient_check
    .groupby(
        ["activity_group", "Is Laundering"],
        observed=True
    )
    .agg(
        total=("new_recipient_7d", "size"),
        new_recipients=("new_recipient_7d", "sum")
    )
)

summary["new_recipient_pct"] = (
    summary["new_recipients"] / summary["total"] * 100
)

print(summary.round(2).to_string())

In [ ]:
import pandas as pd

# Nauja imtis, neįtraukiant ankstesnių 200 operacijų
used_ids = set(validation_200["original_id"].astype(int))

available = validation_full[
    ~validation_full.index.isin(used_ids)
]

new_sample = pd.concat([
    available[available["Is Laundering"] == 0]
    .sample(n=100, random_state=2028),

    available[available["Is Laundering"] == 1]
    .sample(n=100, random_state=2028)
]).copy()

new_sample["original_id"] = new_sample.index

# Skaičiuojame tik ankstesnius mokėjimus tam pačiam gavėjui
results = []

for _, row in new_sample.iterrows():

    history = df.loc[
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] < row["Timestamp"]) &
        (df["Timestamp"] >= row["Timestamp"] - pd.Timedelta(days=7)) &
        (df["To Bank"] == row["To Bank"]) &
        (df["Account.1"] == row["Account.1"])
    ]

    results.append({
        "original_id": int(row["original_id"]),
        "y_true": int(row["Is Laundering"]),
        "same_recipient_7d_count": len(history),
        "new_recipient_7d": int(len(history) == 0)
    })

recipient_holdout = pd.DataFrame(results)

summary = recipient_holdout.groupby("y_true").agg(
    total=("new_recipient_7d", "size"),
    new_recipients=("new_recipient_7d", "sum")
)

summary["new_recipient_pct"] = (
    summary["new_recipients"] / summary["total"] * 100
)

print(summary.round(2).to_string())

recipient_holdout.to_csv(
    "aml_models/validation_recipient_holdout.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
from pathlib import Path
import hashlib

folder = Path("aml_models")

prompt_v3 = (
    folder / "gpt_context_prompt_v3.txt"
).read_text(encoding="utf-8")

recipient_instructions = """

Additional recipient-history feature:
- same_recipient_7d_count: Number of earlier outgoing transactions
  from the same sender bank-account pair to the same recipient
  bank-account pair during the preceding 7 days.

This feature excludes the current transaction and transactions
at the same or later timestamp.

Recipient-history interpretation:
- A value of 0 means that no payment to this recipient was observed
  in the preceding 7 days. It does not prove that the recipient
  has never received payments before.
- A positive value confirms previous payments to this recipient
  within the preceding 7 days.
- Validation analysis of this synthetic dataset found that
  laundering-labelled transactions more frequently involved
  recipients without payments in the preceding 7 days.
- This is a statistical association, not proof of laundering.
- Do not automatically classify transactions to new recipients
  as suspicious or transactions to known recipients as legitimate.
- Consider recipient history together with all other features.
- Do not claim that a recipient is new unless supported by
  the recipient-history feature.
"""

prompt_v4 = prompt_v3 + recipient_instructions

path_v4 = folder / "gpt_context_prompt_v4.txt"

if path_v4.exists():
    assert path_v4.read_text(encoding="utf-8") == prompt_v4, (
        "V4 failas jau egzistuoja, tačiau jo turinys skiriasi."
    )
else:
    path_v4.write_text(prompt_v4, encoding="utf-8")

print("V4 išsaugotas:", path_v4)
print("SHA-256:", hashlib.sha256(path_v4.read_bytes()).hexdigest())
print("Simbolių skaičius:", len(prompt_v4))
print("V3 išliko:", (folder / "gpt_context_prompt_v3.txt").exists())

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

# Neįtraukiame operacijų, kurias GPT jau vertino
previous_gpt = pd.concat([
    pd.read_csv(folder / "gpt_context_validation.csv"),
    pd.read_csv(folder / "gpt_context_validation_v3.csv")
], ignore_index=True)

used_gpt_ids = set(previous_gpt["original_id"].astype(int))

# Neįtraukiame ir operacijų, naudotų gavėjo požymiui tirti
used_feature_ids = (
    set(validation_200["original_id"].astype(int))
    | set(recipient_holdout["original_id"].astype(int))
)

excluded_ids = used_gpt_ids | used_feature_ids

available = validation_full.loc[
    ~validation_full.index.isin(excluded_ids)
]

comparison_sample = pd.concat([
    available[available["Is Laundering"] == 0]
    .sample(n=20, random_state=2029),

    available[available["Is Laundering"] == 1]
    .sample(n=20, random_state=2029)
]).copy()

comparison_sample["original_id"] = comparison_sample.index

# Paruošiame 5-ąjį istorinį požymį
# Istoriją tikriname pagal siuntėją ir gavėją
recipient_counts = []

for _, row in comparison_sample.iterrows():

    mask = (
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["To Bank"] == row["To Bank"]) &
        (df["Account.1"] == row["Account.1"]) &
        (df["Timestamp"] < row["Timestamp"]) &
        (
            df["Timestamp"] >=
            row["Timestamp"] - pd.Timedelta(days=7)
        )
    )

    recipient_counts.append(int(mask.sum()))

comparison_sample["same_recipient_7d_count"] = recipient_counts

# Patikriname imties sudėtį
print("Iš viso operacijų:", len(comparison_sample))
print("\nTikrosios klasės:")
print(comparison_sample["Is Laundering"].value_counts().sort_index())

print("\nGavėjo istorija pagal klasę:")
print(
    comparison_sample.groupby("Is Laundering")[
        "same_recipient_7d_count"
    ].agg(
        total="count",
        be_ankstesniu_mokejimu=lambda x: (x == 0).sum(),
        su_ankstesniais_mokejimais=lambda x: (x > 0).sum()
    ).to_string()
)

print("\nSutapimai su ankstesnėmis imtimis:",
      comparison_sample["original_id"].isin(excluded_ids).sum())

comparison_sample.to_csv(
    folder / "gpt_v3_v4_comparison_sample.csv",
    index=False
)

print("\nImtis išsaugota.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

comparison_sample = pd.read_csv(
    folder / "gpt_v3_v4_comparison_sample.csv",
    parse_dates=["Timestamp"]
)

# 1. Pridedame anksčiau apskaičiuotus aktyvumo rodiklius
activity_features = counts[
    ["From Bank", "Account", "Timestamp",
     "previous_24h_count", "previous_7d_count"]
].drop_duplicates(
    ["From Bank", "Account", "Timestamp"]
)

comparison_sample = comparison_sample.merge(
    activity_features,
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

# 2. Apskaičiuojame likusius istorinius rodiklius
amounts = []
unique_recipients = []

for _, row in comparison_sample.iterrows():

    history = df.loc[
        (df["From Bank"] == row["From Bank"]) &
        (df["Account"] == row["Account"]) &
        (df["Timestamp"] < row["Timestamp"]) &
        (df["Timestamp"] >= row["Timestamp"] - pd.Timedelta(days=7))
    ]

    history_24h = history[
        history["Timestamp"] >=
        row["Timestamp"] - pd.Timedelta(hours=24)
    ]

    same_currency = history_24h[
        history_24h["Payment Currency"] == row["Payment Currency"]
    ]

    amounts.append(
        float(same_currency["Amount Paid"].sum())
    )

    unique_recipients.append(
        int(
            history[["To Bank", "Account.1"]]
            .drop_duplicates()
            .shape[0]
        )
    )

comparison_sample["previous_24h_amount"] = amounts
comparison_sample["previous_7d_unique_recipients"] = unique_recipients

# 3. Patikriname visus penkis požymius
all_context_features = context_features + [
    "same_recipient_7d_count"
]

print("Operacijų skaičius:", len(comparison_sample))
print("\nTrūkstamos reikšmės:")
print(
    comparison_sample[all_context_features]
    .isna().sum().to_string()
)

print("\nPirmosios 5 operacijos:")
print(
    comparison_sample[
        ["original_id", "Is Laundering"] + all_context_features
    ].head().to_string(index=False)
)

# 4. Išsaugome galutinę palyginimo imtį
output_path = folder / "gpt_v3_v4_comparison_ready.csv"

comparison_sample.to_csv(output_path, index=False)

print("\nIšsaugota:", output_path)

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path

folder = Path("aml_models")

sample = pd.read_csv(
    folder / "gpt_v3_v4_comparison_ready.csv",
    parse_dates=["Timestamp"]
)

prompts = {
    "V3": (folder / "gpt_context_prompt_v3.txt").read_text(encoding="utf-8"),
    "V4": (folder / "gpt_context_prompt_v4.txt").read_text(encoding="utf-8")
}

output_path = folder / "gpt_v3_v4_comparison_results.csv"

columns = [
    "original_id", "y_true", "variant", "prediction",
    "reason", "input_tokens", "output_tokens", "seconds", "error"
]

if output_path.exists():
    results = pd.read_csv(output_path)
else:
    results = pd.DataFrame(columns=columns)

successful = results[
    results["error"].fillna("").eq("") &
    results["prediction"].isin([0, 1])
]

completed = set(
    zip(successful["original_id"].astype(int), successful["variant"])
)

new_requests = 0

for _, row in sample.iterrows():
    for variant, instructions in prompts.items():

        key = (int(row["original_id"]), variant)

        if key in completed:
            continue

        features = base_features + context_features

        if variant == "V4":
            features = features + ["same_recipient_7d_count"]

        transaction = prepare_transaction(row, features)

        start = time.time()

        try:
            response = client.responses.create(
                model="gpt-5.4-mini",
                instructions=instructions,
                input=json.dumps(transaction, ensure_ascii=False),
                max_output_tokens=150
            )

            answer = json.loads(response.output_text)
            prediction = int(answer["prediction"])

            if prediction not in (0, 1):
                raise ValueError("Prognozė turi būti 0 arba 1")

            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": prediction,
                "reason": answer.get("reason", ""),
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "seconds": round(time.time() - start, 2),
                "error": ""
            }

        except Exception as e:
            record = {
                "original_id": int(row["original_id"]),
                "y_true": int(row["Is Laundering"]),
                "variant": variant,
                "prediction": None,
                "reason": "",
                "input_tokens": None,
                "output_tokens": None,
                "seconds": round(time.time() - start, 2),
                "error": str(e)[:200]
            }

        results = pd.concat(
            [results, pd.DataFrame([record])],
            ignore_index=True
        )

        results.to_csv(output_path, index=False)
        new_requests += 1

        if new_requests % 10 == 0:
            print(f"Atlikta naujų užklausų: {new_requests}")

print("\nNaujų API užklausų:", new_requests)
print("Iš viso įrašų:", len(results))

print("\nRezultatai pagal klasę:")
print(
    results[
        results["error"].fillna("").eq("") &
        results["prediction"].isin([0, 1])
    ].groupby(["variant", "y_true"])["prediction"]
    .agg(["count", "sum"])
    .to_string()
)

print("\nKlaidų:", results["error"].fillna("").ne("").sum())
print("Išsaugota:", output_path)

In [ ]:
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v3_v4_comparison_results.csv"
)

comparison = results.pivot(
    index=["original_id", "y_true"],
    columns="variant",
    values="prediction"
).reset_index()

comparison["V3_correct"] = comparison["V3"] == comparison["y_true"]
comparison["V4_correct"] = comparison["V4"] == comparison["y_true"]

comparison["change"] = "Nepasikeitė"

comparison.loc[
    (~comparison["V3_correct"]) & comparison["V4_correct"],
    "change"
] = "V4 pagerino"

comparison.loc[
    comparison["V3_correct"] & (~comparison["V4_correct"]),
    "change"
] = "V4 pablogino"

print("Sprendimų pokyčiai:")
print(comparison["change"].value_counts().to_string())

print("\nPokyčiai pagal tikrąją klasę:")
print(
    pd.crosstab(
        comparison["y_true"],
        comparison["change"]
    ).to_string()
)

print("\nOperacijos, kurių klasifikacija pasikeitė:")
changed = comparison[comparison["V3"] != comparison["V4"]]

print(
    changed[
        ["original_id", "y_true", "V3", "V4", "change"]
    ].to_string(index=False)
)

In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import confusion_matrix

folder = Path("aml_models")

sample = pd.read_csv(
    folder / "gpt_v3_v4_comparison_ready.csv",
    parse_dates=["Timestamp"]
)

xgb_model = joblib.load(folder / "xgb_model.joblib")
preprocessor = joblib.load(folder / "preprocessor.joblib")

# Tie patys baziniai požymiai kaip mokant XGBoost
X = sample[base_features].copy()

# Patikriname, ar modelis priima šiuos duomenis
X_processed = preprocessor.transform(X)

probabilities = xgb_model.predict_proba(X_processed)[:, 1]

# Naudojame anksčiau validavimo metu pasirinktą slenkstį
threshold = 0.957565

sample["xgb_probability"] = probabilities
sample["xgb_prediction"] = (
    probabilities >= threshold
).astype(int)

# GPT V4 prognozės
gpt_results = pd.read_csv(
    folder / "gpt_v3_v4_comparison_results.csv"
)

gpt_v4 = gpt_results[
    (gpt_results["variant"] == "V4") &
    (gpt_results["error"].fillna("") == "")
][["original_id", "prediction"]].rename(
    columns={"prediction": "gpt_v4_prediction"}
)

comparison = sample.merge(
    gpt_v4,
    on="original_id",
    how="left",
    validate="one_to_one"
)

comparison["gpt_v4_prediction"] = (
    comparison["gpt_v4_prediction"].astype(int)
)

y_true = comparison["Is Laundering"]

for model_name, prediction_col in [
    ("XGBoost", "xgb_prediction"),
    ("GPT V4", "gpt_v4_prediction")
]:
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        comparison[prediction_col],
        labels=[0, 1]
    ).ravel()

    print(f"\n{model_name}")
    print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")

print("\nAML operacijos, kurias aptiko tik vienas modelis:")

aml = comparison[comparison["Is Laundering"] == 1]

print(
    pd.crosstab(
        aml["xgb_prediction"],
        aml["gpt_v4_prediction"],
        rownames=["XGBoost"],
        colnames=["GPT V4"]
    ).to_string()
)

comparison.to_csv(
    folder / "xgb_gpt_v4_validation_comparison.csv",
    index=False
)

print("\nPalyginimas išsaugotas.")

In [ ]:
import pandas as pd

# Stulpeliai, naudoti mokant XGBoost
expected_columns = list(preprocessor.feature_names_in_)

print("XGBoost tikisi šių stulpelių:")
print(expected_columns)

# Paruošiame duomenis
X = sample[base_features].copy()

# Laikas sekundėmis nuo vidurnakčio
X["Timestamp_sec"] = (
    X["Timestamp"].dt.hour * 3600
    + X["Timestamp"].dt.minute * 60
    + X["Timestamp"].dt.second
)

# Pašaliname Timestamp tik jei jo nereikia modeliui
X = X[expected_columns]

print("\nTrūkstamos reikšmės:")
print(X.isna().sum().to_string())

# Patikriname transformaciją
X_processed = preprocessor.transform(X)

print("\nTransformacija sėkminga!")
print("Duomenų forma:", X_processed.shape)

In [ ]:
import pandas as pd

print("Timestamp_sec statistika dabartinėje imtyje:")
print(X["Timestamp_sec"].describe().to_string())

print("\nPirmosios 5 laiko reikšmės:")
print(
    pd.DataFrame({
        "Timestamp": sample["Timestamp"].head().values,
        "Timestamp_sec": X["Timestamp_sec"].head().values
    }).to_string(index=False)
)

print("\nAr ankstesniuose duomenyse yra Timestamp_sec?")

for name, dataset in [
    ("train_full", train_full),
    ("validation_full", validation_full),
    ("test_full", test_full)
]:
    print(name, ":", "Timestamp_sec" in dataset.columns)

    if "Timestamp_sec" in dataset.columns:
        print(dataset["Timestamp_sec"].head().to_list())

In [ ]:
print("Timestamp_sec apdorojimas:")
print()

for name, transformer, columns in preprocessor.transformers_:
    if "Timestamp_sec" in list(columns):
        print("Transformatoriaus pavadinimas:", name)
        print("Transformatorius:", transformer)
        print("Stulpeliai:", columns)

print("\nMokymo metu naudotų laiko reikšmių patikra:")

if hasattr(preprocessor, "feature_names_in_"):
    print("Požymis buvo naudojamas:",
          "Timestamp_sec" in preprocessor.feature_names_in_)

# Patikriname, ar Notebook aplinkoje likęs pradinis
# XGBoost mokymo duomenų rinkinys
for variable_name in ["X_train", "X_train_full", "train_features"]:
    if variable_name in globals():
        obj = globals()[variable_name]
        print(f"\nRastas kintamasis: {variable_name}")

        if hasattr(obj, "columns") and "Timestamp_sec" in obj.columns:
            print(obj["Timestamp_sec"].describe().to_string())
            print("Pirmosios reikšmės:",
                  obj["Timestamp_sec"].head().to_list())

In [ ]:
import pandas as pd

print("Kintamieji, susiję su XGBoost mokymu:\n")

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, pd.DataFrame):
        if "Timestamp_sec" in obj.columns:
            print(f"\nKintamasis: {name}")
            print("Eilučių skaičius:", len(obj))
            print("Timestamp_sec reikšmės:")
            print(obj["Timestamp_sec"].head(5).to_list())
            print("Min:", obj["Timestamp_sec"].min())
            print("Max:", obj["Timestamp_sec"].max())

print("\nAnksčiau naudotų transformavimo funkcijų paieška:")

for name, obj in list(globals().items()):
    if (
        callable(obj)
        and not name.startswith("_")
        and any(word in name.lower()
                for word in ["feature", "prepare", "time", "preprocess"])
    ):
        print(name)

In [ ]:
import pandas as pd

print("Kintamojo check stulpeliai:")
print(check.columns.tolist())

print("\nPirmosios 5 eilutės:")
print(check.head(5).to_string(index=False))

print("\nDuomenų rinkinio pradžia:")
print(df["Timestamp"].min())

print("\nSekundės nuo duomenų rinkinio pradžios:")
origin = df["Timestamp"].min()

for _, row in sample.head(5).iterrows():
    seconds = (row["Timestamp"] - origin).total_seconds()
    print(row["Timestamp"], "→", seconds)


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# 1. Teisingai apskaičiuojame laiko požymį
X = sample[base_features].copy()

X["Timestamp_sec"] = (
    X["Timestamp"] - df["Timestamp"].min()
).dt.total_seconds()

X = X[preprocessor.feature_names_in_]

# 2. Patikriname transformaciją
X_processed = preprocessor.transform(X)

# 3. XGBoost prognozės
xgb_probabilities = xgb_model.predict_proba(X_processed)[:, 1]

sample["xgb_probability"] = xgb_probabilities
sample["xgb_prediction"] = (
    xgb_probabilities >= 0.957565
).astype(int)

# 4. Pridedame GPT V4 prognozes
gpt_results = pd.read_csv(
    "aml_models/gpt_v3_v4_comparison_results.csv"
)

gpt_v4 = gpt_results[
    (gpt_results["variant"] == "V4") &
    (gpt_results["error"].fillna("") == "")
][["original_id", "prediction"]].rename(
    columns={"prediction": "gpt_v4_prediction"}
)

comparison = sample.merge(
    gpt_v4,
    on="original_id",
    how="left",
    validate="one_to_one"
)

assert comparison["gpt_v4_prediction"].notna().all()

comparison["gpt_v4_prediction"] = (
    comparison["gpt_v4_prediction"].astype(int)
)

# 5. Palyginame abu modelius
for model_name, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT V4", "gpt_v4_prediction")
]:
    tn, fp, fn, tp = confusion_matrix(
        comparison["Is Laundering"],
        comparison[column],
        labels=[0, 1]
    ).ravel()

    print(f"\n{model_name}")
    print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# 6. Ar GPT aptinka XGBoost praleistus AML atvejus?
aml = comparison[comparison["Is Laundering"] == 1]

print("\nAML operacijų aptikimo palyginimas:")
print(
    pd.crosstab(
        aml["xgb_prediction"],
        aml["gpt_v4_prediction"],
        rownames=["XGBoost"],
        colnames=["GPT V4"]
    ).to_string()
)

comparison.to_csv(
    "aml_models/xgb_gpt_v4_validation_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

comparison = pd.read_csv(
    "aml_models/xgb_gpt_v4_validation_comparison.csv"
)

comparison["hybrid_OR"] = (
    (comparison["xgb_prediction"] == 1) |
    (comparison["gpt_v4_prediction"] == 1)
).astype(int)

comparison["hybrid_AND"] = (
    (comparison["xgb_prediction"] == 1) &
    (comparison["gpt_v4_prediction"] == 1)
).astype(int)

rows = []

for name, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT V4", "gpt_v4_prediction"),
    ("Hybrid OR", "hybrid_OR"),
    ("Hybrid AND", "hybrid_AND")
]:
    tn, fp, fn, tp = confusion_matrix(
        comparison["Is Laundering"],
        comparison[column],
        labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    fpr = fp / (fp + tn) if fp + tn else 0

    rows.append({
        "Modelis": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "FPR": round(fpr, 4)
    })

metrics = pd.DataFrame(rows)

print(metrics.to_string(index=False))

comparison.to_csv(
    "aml_models/hybrid_validation_comparison.csv",
    index=False
)

print("\nHibridinio metodo rezultatai išsaugoti.")

In [ ]:
import pandas as pd

comparison = pd.read_csv(
    "aml_models/hybrid_validation_comparison.csv"
)

comparison["group"] = "Kita"

comparison.loc[
    (comparison["Is Laundering"] == 1) &
    (comparison["gpt_v4_prediction"] == 1),
    "group"
] = "AML – GPT aptiko"

comparison.loc[
    (comparison["Is Laundering"] == 1) &
    (comparison["gpt_v4_prediction"] == 0),
    "group"
] = "AML – GPT praleido"

comparison.loc[
    (comparison["Is Laundering"] == 0) &
    (comparison["gpt_v4_prediction"] == 1),
    "group"
] = "Teisėta – GPT klaidingas įspėjimas"

comparison.loc[
    (comparison["Is Laundering"] == 0) &
    (comparison["gpt_v4_prediction"] == 0),
    "group"
] = "Teisėta – GPT teisingai atmetė"

summary = comparison.groupby("group")["xgb_probability"].agg(
    ["count", "min", "median", "mean", "max"]
)

print("XGBoost tikimybės pagal GPT V4 sprendimus:")
print(summary.round(6).to_string())

print("\nVisos 40 operacijų pagal XGBoost tikimybę:")

print(
    comparison[
        [
            "original_id",
            "Is Laundering",
            "xgb_probability",
            "xgb_prediction",
            "gpt_v4_prediction"
        ]
    ]
    .sort_values("xgb_probability", ascending=False)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

comparison = pd.read_csv(
    "aml_models/hybrid_validation_comparison.csv"
)

thresholds = [0.01, 0.05, 0.10, 0.30, 0.50, 0.70, 0.90, 0.95]

results = []

for threshold in thresholds:

    # GPT įspėjimas priimamas tik tada,
    # kai XGBoost tikimybė pasiekia slenkstį
    hybrid_prediction = (
        (comparison["gpt_v4_prediction"] == 1) &
        (comparison["xgb_probability"] >= threshold)
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        comparison["Is Laundering"],
        hybrid_prediction,
        labels=[0, 1]
    ).ravel()

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Recall": round(tp / (tp + fn), 3),
        "FPR": round(fp / (fp + tn), 3),
        "Precision_balanced": round(
            tp / (tp + fp), 3
        ) if tp + fp else 0
    })

threshold_analysis = pd.DataFrame(results)

print(threshold_analysis.to_string(index=False))

threshold_analysis.to_csv(
    "aml_models/hybrid_threshold_diagnostic.csv",
    index=False
)

print("\nDiagnostinė analizė išsaugota.")

In [ ]:
import pandas as pd

# Tikras AML dažnis visame validavimo rinkinyje
prevalence = validation_full["Is Laundering"].mean()

diagnostic = pd.read_csv(
    "aml_models/hybrid_threshold_diagnostic.csv"
)

# Precision apskaičiavimas atsižvelgiant į natūralų AML dažnį
diagnostic["estimated_precision_natural"] = (
    diagnostic["Recall"] * prevalence
) / (
    diagnostic["Recall"] * prevalence
    + diagnostic["FPR"] * (1 - prevalence)
)

# Tikėtinas įspėjimų skaičius 3000 operacijų
diagnostic["expected_alerts_3000"] = 3000 * (
    diagnostic["Recall"] * prevalence
    + diagnostic["FPR"] * (1 - prevalence)
)

diagnostic["expected_true_aml_alerts_3000"] = (
    3000 * diagnostic["Recall"] * prevalence
)

print("Validavimo AML dažnis:", round(prevalence * 100, 4), "%")

print("\nTeorinis įvertinimas esant natūraliam AML dažniui:")
print(
    diagnostic[
        [
            "threshold",
            "Recall",
            "FPR",
            "estimated_precision_natural",
            "expected_alerts_3000",
            "expected_true_aml_alerts_3000"
        ]
    ].round(4).to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np
import joblib

xgb_model = joblib.load("aml_models/xgb_model.joblib")
preprocessor = joblib.load("aml_models/preprocessor.joblib")

# Paruošiame visą validavimo rinkinį
X_val = validation_full.copy()

X_val["Timestamp_sec"] = (
    X_val["Timestamp"] - df["Timestamp"].min()
).dt.total_seconds()

X_val = X_val[list(preprocessor.feature_names_in_)]

# XGBoost tikimybės
X_val_processed = preprocessor.transform(X_val)

val_probabilities = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

y_val = validation_full["Is Laundering"].to_numpy()

thresholds = [
    0.01, 0.05, 0.10, 0.30,
    0.50, 0.70, 0.90, 0.95,
    0.957565
]

results = []

for threshold in thresholds:
    selected = val_probabilities >= threshold

    tp = int(np.sum(selected & (y_val == 1)))
    fp = int(np.sum(selected & (y_val == 0)))
    fn = int(np.sum(~selected & (y_val == 1)))
    tn = int(np.sum(~selected & (y_val == 0)))

    results.append({
        "threshold": threshold,
        "selected_total": tp + fp,
        "selected_percent": round(
            100 * (tp + fp) / len(y_val), 3
        ),
        "AML_detected": tp,
        "AML_missed": fn,
        "Recall": round(tp / (tp + fn), 4),
        "FPR": round(fp / (fp + tn), 4),
        "Precision": round(
            tp / (tp + fp), 4
        ) if tp + fp else 0
    })

selection_analysis = pd.DataFrame(results)

print("Validavimo operacijų:", len(y_val))
print("Tikros AML operacijos:", int(y_val.sum()))

print("\nXGBoost atrankos rezultatai:")
print(selection_analysis.to_string(index=False))

selection_analysis.to_csv(
    "aml_models/xgb_validation_selection_analysis.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score
)

# Naudojame jau apskaičiuotas validavimo prognozes
precision, recall, thresholds_pr = precision_recall_curve(
    y_val,
    val_probabilities
)

ap = average_precision_score(
    y_val,
    val_probabilities
)

roc_auc = roc_auc_score(
    y_val,
    val_probabilities
)

baseline = y_val.mean()

print(f"Average Precision (AP): {ap:.6f}")
print(f"ROC AUC: {roc_auc:.6f}")
print(f"AML dažnis (atsitiktinė bazinė reikšmė): {baseline:.6f}")

plt.figure(figsize=(9, 6))

plt.plot(
    recall,
    precision,
    label=f"XGBoost (AP = {ap:.4f})"
)

plt.axhline(
    baseline,
    linestyle="--",
    color="gray",
    label=f"AML dažnis = {baseline:.4f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("XGBoost Precision–Recall kreivė (validavimo rinkinys)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# XGBoost tikimybių intervalai
bins = [0, 0.05, 0.30, 0.50, 0.70, 0.90, 0.95, 1.0]

labels = [
    "0.00–0.05",
    "0.05–0.30",
    "0.30–0.50",
    "0.50–0.70",
    "0.70–0.90",
    "0.90–0.95",
    "0.95–1.00"
]

risk_analysis = pd.DataFrame({
    "xgb_probability": val_probabilities,
    "Is Laundering": y_val
})

risk_analysis["risk_group"] = pd.cut(
    risk_analysis["xgb_probability"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

summary = risk_analysis.groupby(
    "risk_group",
    observed=True
)["Is Laundering"].agg(
    total="count",
    aml="sum"
)

summary["legitimate"] = summary["total"] - summary["aml"]

summary["aml_rate_percent"] = (
    100 * summary["aml"] / summary["total"]
)

print("Operacijų pasiskirstymas pagal XGBoost riziką:")
print(summary.round(4).to_string())

print("\nIš viso:")
print("Operacijų:", summary["total"].sum())
print("AML:", summary["aml"].sum())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Validavimo operacijos su XGBoost tikimybėmis
candidate_pool = validation_full.copy()

candidate_pool["original_id"] = candidate_pool.index
candidate_pool["xgb_probability"] = val_probabilities

# Tik aukštos rizikos operacijos
candidate_pool = candidate_pool[
    candidate_pool["xgb_probability"] >= 0.90
].copy()

# Pašaliname ankstesniame GPT V3/V4 palyginime naudotas operacijas
used_ids = set()

previous_file = Path(
    "aml_models/gpt_v3_v4_comparison_ready.csv"
)

if previous_file.exists():
    previous = pd.read_csv(
        previous_file,
        usecols=["original_id"]
    )
    used_ids.update(previous["original_id"].tolist())

candidate_pool = candidate_pool[
    ~candidate_pool["original_id"].isin(used_ids)
]

# Atskirai atrenkame AML ir teisėtas operacijas
aml_sample = candidate_pool[
    candidate_pool["Is Laundering"] == 1
].sample(n=100, random_state=2026)

legit_sample = candidate_pool[
    candidate_pool["Is Laundering"] == 0
].sample(n=100, random_state=2026)

high_risk_200 = pd.concat(
    [aml_sample, legit_sample]
).sample(frac=1, random_state=2026).reset_index(drop=True)

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200.csv",
    index=False
)

print("Nauja aukštos rizikos imtis:")
print("Iš viso:", len(high_risk_200))
print("AML:", high_risk_200["Is Laundering"].sum())
print(
    "Teisėtos:",
    (high_risk_200["Is Laundering"] == 0).sum()
)

print("\nXGBoost tikimybių intervalas:")
print(
    high_risk_200["xgb_probability"].agg(
        ["min", "median", "max"]
    )
)

print("\nImtis išsaugota.")

In [ ]:
from pathlib import Path
import pandas as pd

print("Naujos imties stulpeliai:")
print(high_risk_200.columns.tolist())

print("\nIstorinių požymių būklė:")

required_context = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

for feature in required_context:
    if feature in high_risk_200.columns:
        print(
            feature,
            "– yra, trūkstamų reikšmių:",
            high_risk_200[feature].isna().sum()
        )
    else:
        print(feature, "– nėra")

print("\nAnksčiau išsaugoti kontekstiniai duomenys:")

for path in sorted(Path("aml_models").glob("*.csv")):
    name = path.name.lower()
    if any(word in name for word in ["context", "recipient", "comparison"]):
        print(path.name)

print("\nUnikalūs original_id:", high_risk_200["original_id"].nunique())
print("Operacijų skaičius:", len(high_risk_200))

In [ ]:
print("counts lentelės dydis:")
print(counts.shape)

print("\ncounts stulpeliai:")
print(counts.columns.tolist())

print("\nPirmosios 5 eilutės:")
print(counts.head(5).to_string(index=False))

print("\nPradinio df indeksas:")
print("Unikalus:", df.index.is_unique)

print("\nNaujos imties indeksų atitikimas:")
ids = high_risk_200["original_id"]

print("Visi original_id yra df:", ids.isin(df.index).all())

print(
    "Timestamp sutampa:",
    (
        df.loc[ids, "Timestamp"].to_numpy()
        == high_risk_200["Timestamp"].to_numpy()
    ).all()
)

In [ ]:
import pandas as pd

# Jei požymiai jau buvo prijungti, pašaliname juos prieš sujungimą
columns_to_add = [
    "previous_24h_count",
    "previous_7d_count"
]

high_risk_200 = high_risk_200.drop(
    columns=columns_to_add,
    errors="ignore"
)

high_risk_200 = high_risk_200.merge(
    counts[
        [
            "From Bank",
            "Account",
            "Timestamp",
            "previous_24h_count",
            "previous_7d_count"
        ]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų skaičius:", len(high_risk_200))
print("Unikalūs original_id:", high_risk_200["original_id"].nunique())

print("\nTrūkstamos reikšmės:")
print(
    high_risk_200[
        ["previous_24h_count", "previous_7d_count"]
    ].isna().sum()
)

assert len(high_risk_200) == 200
assert high_risk_200["original_id"].is_unique
assert high_risk_200[columns_to_add].notna().all().all()

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200_partial_context.csv",
    index=False
)

print("\n135 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# Siuntėjai, kurių istorija reikalinga
senders = high_risk_200[
    ["From Bank", "Account"]
].drop_duplicates()

# Atrenkame šių siuntėjų operacijas iš viso df
history = df.merge(
    senders,
    on=["From Bank", "Account"],
    how="inner"
)

history = history[
    ["From Bank", "Account", "Timestamp", "Amount Paid"]
].copy()

# Sujungiame vienu metu atliktų operacijų sumas
history = (
    history.groupby(
        ["From Bank", "Account", "Timestamp"],
        as_index=False
    )["Amount Paid"]
    .sum()
)

history = history.sort_values(
    ["From Bank", "Account", "Timestamp"]
)

# Kiekvienam siuntėjui apskaičiuojame ankstesnių 24 h sumą
def calculate_previous_amount(group):
    group = group.sort_values("Timestamp").copy()

    times = group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    amounts = group["Amount Paid"].to_numpy(dtype=float)

    cumulative = np.concatenate([[0.0], np.cumsum(amounts)])

    # Intervalas [Timestamp - 24 h, Timestamp)
    left = np.searchsorted(
        times,
        times - np.timedelta64(24, "h"),
        side="left"
    )

    positions = np.arange(len(group))

    group["previous_24h_amount"] = (
        cumulative[positions] - cumulative[left]
    )

    return group

history = (
    history.groupby(
        ["From Bank", "Account"],
        group_keys=False
    )
    .apply(calculate_previous_amount)
    .reset_index(drop=True)
)

# Prijungiame požymį prie 200 operacijų
high_risk_200 = high_risk_200.merge(
    history[
        [
            "From Bank",
            "Account",
            "Timestamp",
            "previous_24h_amount"
        ]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų skaičius:", len(high_risk_200))
print(
    "Trūkstamos previous_24h_amount reikšmės:",
    high_risk_200["previous_24h_amount"].isna().sum()
)

print("\nPožymio statistika:")
print(
    high_risk_200["previous_24h_amount"]
    .describe()
    .round(2)
)

assert len(high_risk_200) == 200
assert high_risk_200["previous_24h_amount"].notna().all()

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200_partial_context.csv",
    index=False
)

print("\n136 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# Pašaliname galimą ankstesnio bandymo stulpelį
high_risk_200 = high_risk_200.drop(
    columns=["previous_24h_amount"],
    errors="ignore"
)

# Atrenkame tik reikalingų siuntėjų istoriją
senders = high_risk_200[
    ["From Bank", "Account"]
].drop_duplicates()

history = df.merge(
    senders,
    on=["From Bank", "Account"],
    how="inner"
)

# Vienu metu atliktų operacijų sumos
history = (
    history.groupby(
        ["From Bank", "Account", "Timestamp"],
        as_index=False
    )["Amount Paid"]
    .sum()
)

history = history.sort_values(
    ["From Bank", "Account", "Timestamp"]
).reset_index(drop=True)

history["previous_24h_amount"] = 0.0

# Skaičiuojame atskirai kiekvienam siuntėjui
for _, positions in history.groupby(
    ["From Bank", "Account"],
    sort=False
).indices.items():

    group = history.iloc[positions]

    times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    amounts = group["Amount Paid"].to_numpy(
        dtype=float
    )

    cumulative = np.concatenate(
        [[0.0], np.cumsum(amounts)]
    )

    left = np.searchsorted(
        times,
        times - np.timedelta64(24, "h"),
        side="left"
    )

    previous_amounts = (
        cumulative[np.arange(len(group))]
        - cumulative[left]
    )

    history.loc[
        positions, "previous_24h_amount"
    ] = previous_amounts

# Prijungiame prie 200 operacijų
high_risk_200 = high_risk_200.merge(
    history[
        [
            "From Bank",
            "Account",
            "Timestamp",
            "previous_24h_amount"
        ]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų skaičius:", len(high_risk_200))

print(
    "Trūkstamos reikšmės:",
    high_risk_200["previous_24h_amount"].isna().sum()
)

print("\nPožymio statistika:")
print(
    high_risk_200["previous_24h_amount"]
    .describe()
    .round(2)
)

assert len(high_risk_200) == 200
assert high_risk_200["previous_24h_amount"].notna().all()

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200_partial_context.csv",
    index=False
)

print("\n136 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# Pašaliname stulpelį, jei kodas paleidžiamas pakartotinai
high_risk_200 = high_risk_200.drop(
    columns=["previous_7d_unique_recipients"],
    errors="ignore"
)

# Naudojame tik reikalingų siuntėjų operacijas
senders = high_risk_200[
    ["From Bank", "Account"]
].drop_duplicates()

history_recipients = df.merge(
    senders,
    on=["From Bank", "Account"],
    how="inner"
)[
    [
        "From Bank",
        "Account",
        "Timestamp",
        "To Bank",
        "Account.1"
    ]
].copy()

history_recipients = history_recipients.sort_values(
    ["From Bank", "Account", "Timestamp"]
)

# Kiekvienam siuntėjui apskaičiuojame unikalius gavėjus
results = []

for (bank, account), group in history_recipients.groupby(
    ["From Bank", "Account"],
    sort=False
):
    group = group.sort_values("Timestamp")

    times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    recipients = list(
        zip(
            group["To Bank"],
            group["Account.1"]
        )
    )

    # Skaičiuojame tik imtyje esančiais laiko momentais
    target_times = high_risk_200.loc[
        (high_risk_200["From Bank"] == bank) &
        (high_risk_200["Account"] == account),
        "Timestamp"
    ].unique()

    for target_time in target_times:
        t = np.datetime64(target_time, "ns")

        left = np.searchsorted(
            times,
            t - np.timedelta64(7, "D"),
            side="left"
        )

        right = np.searchsorted(
            times,
            t,
            side="left"
        )

        unique_count = len(
            set(recipients[left:right])
        )

        results.append({
            "From Bank": bank,
            "Account": account,
            "Timestamp": pd.Timestamp(t),
            "previous_7d_unique_recipients": unique_count
        })

recipient_counts = pd.DataFrame(results)

high_risk_200 = high_risk_200.merge(
    recipient_counts,
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

print("Operacijų skaičius:", len(high_risk_200))

print(
    "Trūkstamos reikšmės:",
    high_risk_200[
        "previous_7d_unique_recipients"
    ].isna().sum()
)

print("\nPožymio statistika:")
print(
    high_risk_200[
        "previous_7d_unique_recipients"
    ].describe().round(2)
)

assert len(high_risk_200) == 200
assert high_risk_200[
    "previous_7d_unique_recipients"
].notna().all()

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200_partial_context.csv",
    index=False
)

print("\n137 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# Kad kodą būtų galima saugiai paleisti pakartotinai
high_risk_200 = high_risk_200.drop(
    columns=["same_recipient_7d_count"],
    errors="ignore"
)

# Sugrupuojame istoriją pagal siuntėją ir gavėją
history_pairs = history_recipients.sort_values(
    [
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "Timestamp"
    ]
)

# Tikslinės operacijos
targets = high_risk_200[
    [
        "original_id",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "Timestamp"
    ]
].copy()

results = []

pair_columns = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1"
]

# Iš anksto paruošiame kiekvienos siuntėjo–gavėjo poros laikus
pair_times = {
    key: group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    for key, group in history_pairs.groupby(pair_columns)
}

for row in targets.itertuples(index=False, name=None):

    original_id, from_bank, account, to_bank, recipient_account, timestamp = row

    key = (from_bank, account, to_bank, recipient_account)

    times = pair_times.get(key)

    if times is None:
        count = 0
    else:
        t = np.datetime64(timestamp, "ns")

        left = np.searchsorted(
            times,
            t - np.timedelta64(7, "D"),
            side="left"
        )

        right = np.searchsorted(
            times,
            t,
            side="left"
        )

        count = int(right - left)

    results.append({
        "original_id": original_id,
        "same_recipient_7d_count": count
    })

recipient_repeat_counts = pd.DataFrame(results)

# Prijungiame pagal unikalų operacijos ID
high_risk_200 = high_risk_200.merge(
    recipient_repeat_counts,
    on="original_id",
    how="left",
    validate="one_to_one"
)

context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("Operacijų skaičius:", len(high_risk_200))

print("\nTrūkstamos reikšmės:")
print(high_risk_200[context_columns].isna().sum())

print("\nPaskutinio požymio statistika:")
print(
    high_risk_200["same_recipient_7d_count"]
    .describe()
    .round(2)
)

assert len(high_risk_200) == 200
assert high_risk_200["original_id"].is_unique
assert high_risk_200[context_columns].notna().all().all()

high_risk_200.to_csv(
    "aml_models/gpt_high_risk_200_full_context.csv",
    index=False
)

print("\n138 žingsnis sėkmingai atliktas!")
print("Visi 5 GPT V4 istoriniai požymiai paruošti.")

In [ ]:
import pandas as pd
from pathlib import Path

current_ids = set(
    high_risk_200["original_id"].astype(int)
)

files_to_check = [
    "gpt_context_pilot.csv",
    "gpt_context_validation.csv",
    "gpt_context_validation_v3.csv",
    "gpt_v3_v4_comparison_ready.csv",
    "gpt_v3_v4_comparison_sample.csv",
    "validation_100_context.csv",
    "validation_200_context.csv",
    "validation_200_recipient_features.csv",
    "validation_recipient_holdout.csv"
]

results = []

for filename in files_to_check:
    path = Path("aml_models") / filename

    if not path.exists():
        results.append((filename, "Failo nėra", None))
        continue

    columns = pd.read_csv(path, nrows=0).columns.tolist()

    id_column = next(
        (col for col in ["original_id", "test_row_id"]
         if col in columns),
        None
    )

    if id_column is None:
        results.append((filename, "Nėra tinkamo ID", None))
        continue

    previous_ids = set(
        pd.read_csv(path, usecols=[id_column])[id_column]
        .dropna()
        .astype(int)
    )

    overlap = current_ids.intersection(previous_ids)

    results.append(
        (filename, id_column, len(overlap))
    )

print("Ankstesnių eksperimentų patikrinimas:\n")

for filename, status, overlap in results:
    print(f"{filename}: {status}, sutapimų: {overlap}")

print("\nNaujos imties dydis:", len(current_ids))

print(
    "Failai, kuriuose aptikta sutapimų:",
    sum(
        overlap > 0
        for _, _, overlap in results
        if overlap is not None
    )
)

In [ ]:
import pandas as pd
from pathlib import Path

files_to_check = [
    "gpt_context_validation.csv",
    "gpt_context_validation_v3.csv",
    "gpt_v3_v4_comparison_ready.csv",
    "gpt_v3_v4_comparison_sample.csv",
    "validation_100_context.csv",
    "validation_200_context.csv",
    "validation_200_recipient_features.csv",
    "validation_recipient_holdout.csv"
]

used_ids = set()

for filename in files_to_check:
    path = Path("aml_models") / filename

    if not path.exists():
        continue

    columns = pd.read_csv(path, nrows=0).columns

    id_column = next(
        (col for col in ["original_id", "test_row_id"]
         if col in columns),
        None
    )

    if id_column is not None:
        ids = pd.read_csv(
            path,
            usecols=[id_column]
        )[id_column].dropna().astype(int)

        used_ids.update(ids.tolist())

overlap_mask = high_risk_200[
    "original_id"
].astype(int).isin(used_ids)

overlapping = high_risk_200.loc[overlap_mask]

print("Unikalių sutampančių operacijų:", len(overlapping))

print("\nSutapimų pasiskirstymas:")
print(
    overlapping["Is Laundering"]
    .value_counts()
    .rename(index={0: "Teisėtos", 1: "AML"})
    .to_string()
)

print("\nNaujoje imtyje lieka:")
print("Operacijų:", (~overlap_mask).sum())

print("\nReikės pakeisti:")
print("AML:", int(overlapping["Is Laundering"].sum()))
print(
    "Teisėtų:",
    int((overlapping["Is Laundering"] == 0).sum())
)

print("\nPastaba: failas gpt_context_pilot.csv neįtrauktas,")
print("nes ankstesnė patikra jame nerado tinkamo ID.")

In [ ]:
import pandas as pd

# 1. Paliekame 171 nesutampančią operaciją
clean_sample = high_risk_200[
    ~high_risk_200["original_id"].astype(int).isin(used_ids)
].copy()

# 2. Paruošiame galimų pakaitinių AML operacijų rinkinį
replacement_pool = validation_full.copy()

replacement_pool["original_id"] = replacement_pool.index
replacement_pool["xgb_probability"] = val_probabilities

replacement_pool = replacement_pool[
    (replacement_pool["Is Laundering"] == 1) &
    (replacement_pool["xgb_probability"] >= 0.90) &
    (~replacement_pool["original_id"].isin(used_ids)) &
    (~replacement_pool["original_id"].isin(clean_sample["original_id"]))
]

print("Galimų naujų AML operacijų:", len(replacement_pool))

assert len(replacement_pool) >= 29, (
    "Nepakanka naujų AML operacijų."
)

# 3. Atsitiktinai atrenkame 29 pakaitines operacijas
replacement_aml = replacement_pool.sample(
    n=29,
    random_state=2027
).copy()

# 4. Sujungiame su išsaugotomis operacijomis
high_risk_200_clean = pd.concat(
    [clean_sample, replacement_aml],
    ignore_index=True
)

# 5. Patikriname rezultatą
print("\nGalutinės imties dydis:", len(high_risk_200_clean))
print("Unikalūs ID:", high_risk_200_clean["original_id"].nunique())

print("\nKlasių pasiskirstymas:")
print(high_risk_200_clean["Is Laundering"].value_counts())

print(
    "\nSutapimai su ankstesniais eksperimentais:",
    high_risk_200_clean["original_id"].isin(used_ids).sum()
)

context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("\nTrūkstami istoriniai požymiai:")
print(high_risk_200_clean[context_columns].isna().sum())

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean["original_id"].is_unique
assert high_risk_200_clean["Is Laundering"].sum() == 100
assert not high_risk_200_clean["original_id"].isin(used_ids).any()

high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_partial.csv",
    index=False
)

print("\n141 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd

# Du požymiai, kuriuos papildysime
features = [
    "previous_24h_count",
    "previous_7d_count"
]

# Išsaugome originalią eilučių tvarką
high_risk_200_clean["_row_order"] = range(
    len(high_risk_200_clean)
)

# Prijungiame istorinius skaičiavimus
updated = high_risk_200_clean.merge(
    counts[
        [
            "From Bank",
            "Account",
            "Timestamp",
            "previous_24h_count",
            "previous_7d_count"
        ]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    suffixes=("", "_new"),
    validate="many_to_one"
)

# Užpildome tik trūkstamas reikšmes
for feature in features:
    updated[feature] = updated[feature].fillna(
        updated[f"{feature}_new"]
    )
    updated = updated.drop(columns=[f"{feature}_new"])

# Atkuriame eilučių tvarką
high_risk_200_clean = (
    updated.sort_values("_row_order")
    .drop(columns="_row_order")
    .reset_index(drop=True)
)

print("Operacijų skaičius:", len(high_risk_200_clean))

print("\nTrūkstamos reikšmės:")
print(
    high_risk_200_clean[
        [
            "previous_24h_count",
            "previous_7d_count",
            "previous_24h_amount",
            "previous_7d_unique_recipients",
            "same_recipient_7d_count"
        ]
    ].isna().sum()
)

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean["original_id"].is_unique
assert high_risk_200_clean[features].notna().all().all()

high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_partial.csv",
    index=False
)

print("\n142 žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# 1. Atrenkame tik operacijas, kurioms trūksta požymio
missing = high_risk_200_clean[
    high_risk_200_clean["previous_24h_amount"].isna()
].copy()

print("Operacijų, kurioms reikia skaičiavimo:", len(missing))

# 2. Atrenkame tik reikalingų siuntėjų istoriją
senders = missing[
    ["From Bank", "Account"]
].drop_duplicates()

history_amount = df.merge(
    senders,
    on=["From Bank", "Account"],
    how="inner"
)[
    ["From Bank", "Account", "Timestamp", "Amount Paid"]
].copy()

# 3. Susumuojame operacijas, atliktas tuo pačiu laiku
history_amount = (
    history_amount.groupby(
        ["From Bank", "Account", "Timestamp"],
        as_index=False
    )["Amount Paid"]
    .sum()
)

history_amount = history_amount.sort_values(
    ["From Bank", "Account", "Timestamp"]
).reset_index(drop=True)

# 4. Apskaičiuojame 24 valandų istoriją
history_amount["previous_24h_amount"] = 0.0

for _, positions in history_amount.groupby(
    ["From Bank", "Account"],
    sort=False
).indices.items():

    group = history_amount.iloc[positions]

    times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    amounts = group["Amount Paid"].to_numpy(dtype=float)

    cumulative = np.concatenate([
        [0.0],
        np.cumsum(amounts)
    ])

    left = np.searchsorted(
        times,
        times - np.timedelta64(24, "h"),
        side="left"
    )

    values = (
        cumulative[np.arange(len(group))]
        - cumulative[left]
    )

    history_amount.loc[
        positions, "previous_24h_amount"
    ] = values

# 5. Prijungiame apskaičiuotas reikšmes pagal siuntėją ir laiką
new_values = missing[
    ["original_id", "From Bank", "Account", "Timestamp"]
].merge(
    history_amount[
        [
            "From Bank",
            "Account",
            "Timestamp",
            "previous_24h_amount"
        ]
    ],
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

value_map = new_values.set_index(
    "original_id"
)["previous_24h_amount"]

high_risk_200_clean["previous_24h_amount"] = (
    high_risk_200_clean["previous_24h_amount"].fillna(
        high_risk_200_clean["original_id"].map(value_map)
    )
)

# 6. Patikriname ir išsaugome
print("\nTrūkstamos reikšmės:")
print(
    high_risk_200_clean[
        [
            "previous_24h_count",
            "previous_7d_count",
            "previous_24h_amount",
            "previous_7d_unique_recipients",
            "same_recipient_7d_count"
        ]
    ].isna().sum()
)

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean["previous_24h_amount"].notna().all()

high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_partial.csv",
    index=False
)

print("\n žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# 1. Atrenkame 29 operacijas su trūkstamu požymiu
missing = high_risk_200_clean[
    high_risk_200_clean["previous_7d_unique_recipients"].isna()
].copy()

print("Operacijų, kurioms reikia skaičiavimo:", len(missing))

# 2. Atrenkame reikalingų siuntėjų istoriją
senders = missing[
    ["From Bank", "Account"]
].drop_duplicates()

history_recipients_new = df.merge(
    senders,
    on=["From Bank", "Account"],
    how="inner"
)[
    ["From Bank", "Account", "Timestamp", "To Bank", "Account.1"]
].copy()

history_recipients_new = history_recipients_new.sort_values(
    ["From Bank", "Account", "Timestamp"]
)

# 3. Apskaičiuojame unikalius gavėjus per ankstesnes 7 dienas
results = []

for (bank, account), group in history_recipients_new.groupby(
    ["From Bank", "Account"],
    sort=False
):
    times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    recipients = list(
        zip(group["To Bank"], group["Account.1"])
    )

    targets = missing.loc[
        (missing["From Bank"] == bank) &
        (missing["Account"] == account),
        ["original_id", "Timestamp"]
    ]

    for original_id, timestamp in targets.itertuples(
        index=False, name=None
    ):
        t = np.datetime64(timestamp, "ns")

        left = np.searchsorted(
            times,
            t - np.timedelta64(7, "D"),
            side="left"
        )

        right = np.searchsorted(
            times,
            t,
            side="left"
        )

        unique_count = len(set(recipients[left:right]))

        results.append({
            "original_id": original_id,
            "previous_7d_unique_recipients": unique_count
        })

# 4. Užpildome trūkstamas reikšmes
new_values = pd.DataFrame(results)

assert len(new_values) == len(missing)
assert new_values["original_id"].is_unique

value_map = new_values.set_index(
    "original_id"
)["previous_7d_unique_recipients"]

high_risk_200_clean["previous_7d_unique_recipients"] = (
    high_risk_200_clean["previous_7d_unique_recipients"].fillna(
        high_risk_200_clean["original_id"].map(value_map)
    )
)

# 5. Patikriname ir išsaugome
context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("\nTrūkstamos reikšmės:")
print(high_risk_200_clean[context_columns].isna().sum())

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean[
    "previous_7d_unique_recipients"
].notna().all()

high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_partial.csv",
    index=False
)

print("\n žingsnis sėkmingai atliktas!")

In [ ]:
import pandas as pd
import numpy as np

# Atrenkame operacijas, kurioms trūksta paskutinio požymio
missing = high_risk_200_clean[
    high_risk_200_clean["same_recipient_7d_count"].isna()
].copy()

print("Operacijų, kurioms reikia skaičiavimo:", len(missing))

# Naudojame jau paruoštą operacijų istoriją
# history_recipients_new turi visų reikalingų siuntėjų operacijas

history_pairs_new = history_recipients_new[
    ["From Bank", "Account", "To Bank", "Account.1", "Timestamp"]
].copy()

history_pairs_new = history_pairs_new.sort_values("Timestamp")

# Sukuriame kiekvienos siuntėjo ir gavėjo poros operacijų laikų žodyną
pair_times = {}

for key, group in history_pairs_new.groupby(
    ["From Bank", "Account", "To Bank", "Account.1"],
    sort=False
):
    pair_times[key] = np.sort(
        group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    )

# Skaičiuojame tik ankstesnes operacijas per 7 dienas
results = []

for row in missing.itertuples(index=False):
    key = (
        row._asdict()["From Bank"],
        row._asdict()["Account"],
        row._asdict()["To Bank"],
        row._asdict()["Account.1"]
    )

    t = np.datetime64(row.Timestamp, "ns")
    times = pair_times.get(key, np.array([], dtype="datetime64[ns]"))

    left = np.searchsorted(
        times,
        t - np.timedelta64(7, "D"),
        side="left"
    )

    right = np.searchsorted(
        times,
        t,
        side="left"
    )

    results.append((row.original_id, int(right - left)))

# Papildome tik trūkstamas reikšmes
value_map = dict(results)

high_risk_200_clean["same_recipient_7d_count"] = (
    high_risk_200_clean["same_recipient_7d_count"].fillna(
        high_risk_200_clean["original_id"].map(value_map)
    )
)

# Galutinė patikra
context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("\nTrūkstamos reikšmės:")
print(high_risk_200_clean[context_columns].isna().sum())

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean[context_columns].notna().all().all()

high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_full_context.csv",
    index=False
)

print("\nVisi 5 istoriniai požymiai paruošti!")

In [ ]:
import pandas as pd
import numpy as np

missing = high_risk_200_clean[
    high_risk_200_clean["same_recipient_7d_count"].isna()
].copy()

print("Operacijų, kurioms reikia skaičiavimo:", len(missing))

# Naudojame jau paruoštą siuntėjų operacijų istoriją
history_pairs_new = history_recipients_new[
    ["From Bank", "Account", "To Bank", "Account.1", "Timestamp"]
].copy()

pair_times = {}

for key, group in history_pairs_new.groupby(
    ["From Bank", "Account", "To Bank", "Account.1"],
    sort=False
):
    pair_times[key] = np.sort(
        group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    )

# Apskaičiuojame ankstesnių pervedimų tam pačiam gavėjui skaičių
results = []

for _, row in missing.iterrows():
    key = (
        row["From Bank"],
        row["Account"],
        row["To Bank"],
        row["Account.1"]
    )

    t = np.datetime64(row["Timestamp"], "ns")

    times = pair_times.get(
        key,
        np.array([], dtype="datetime64[ns]")
    )

    left = np.searchsorted(
        times,
        t - np.timedelta64(7, "D"),
        side="left"
    )

    right = np.searchsorted(
        times,
        t,
        side="left"
    )

    results.append(
        (row["original_id"], int(right - left))
    )

# Užpildome trūkstamas reikšmes
value_map = dict(results)

high_risk_200_clean["same_recipient_7d_count"] = (
    high_risk_200_clean["same_recipient_7d_count"].fillna(
        high_risk_200_clean["original_id"].map(value_map)
    )
)

# Patikriname visus penkis požymius
context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("\nTrūkstamos reikšmės:")
print(high_risk_200_clean[context_columns].isna().sum())

assert len(high_risk_200_clean) == 200
assert high_risk_200_clean[context_columns].notna().all().all()

# Išsaugome galutinę imtį
high_risk_200_clean.to_csv(
    "aml_models/gpt_high_risk_200_clean_full_context.csv",
    index=False
)

print("\nVisi 5 istoriniai požymiai sėkmingai paruošti!")

In [ ]:
from pathlib import Path
import pandas as pd

# Įkeliame galutinę imtį
sample = pd.read_csv(
    "aml_models/gpt_high_risk_200_clean_full_context.csv",
    parse_dates=["Timestamp"]
)

# Įkeliame GPT V4 promptą
prompt_path = Path("aml_models/gpt_context_prompt_v4.txt")
prompt_v4 = prompt_path.read_text(encoding="utf-8")

# Patikriname imtį
context_columns = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("Operacijų skaičius:", len(sample))
print("AML operacijų:", int(sample["Is Laundering"].sum()))
print("Teisėtų operacijų:", int((sample["Is Laundering"] == 0).sum()))

print("\nTrūkstamos konteksto reikšmės:")
print(sample[context_columns].isna().sum())

print("\nGPT V4 promptas:")
print("-" * 60)
print(prompt_v4)
print("-" * 60)

print("\nPrompto ilgis (simboliais):", len(prompt_v4))

# Patikriname, ar visi reikalingi stulpeliai yra imtyje
required_columns = base_features + context_columns

missing_columns = [
    col for col in required_columns
    if col not in sample.columns
]

print("\nTrūkstami GPT įvesties stulpeliai:", missing_columns)

# Įsitikiname, kad modelio įvestyje nėra tikrųjų etikečių
# ar XGBoost prognozės
assert "Is Laundering" not in required_columns
assert "xgb_probability" not in required_columns

assert len(sample) == 200
assert sample[context_columns].notna().all().all()
assert not missing_columns

print("\nDuomenys paruošti GPT V4 vertinimui!")

In [ ]:
import json

# GPT V4 įvesties požymiai
input_features = base_features + context_columns

# Paruošiame vienos operacijos pavyzdį
example_transaction = prepare_transaction(
    sample.iloc[0],
    input_features
)

example_json = json.dumps(
    example_transaction,
    ensure_ascii=False
)

# Apytikris tokenų skaičius: 1 tokenas ≈ 4 simboliai
prompt_tokens = len(prompt_v4) / 4
transaction_tokens = len(example_json) / 4

estimated_input_tokens = prompt_tokens + transaction_tokens
estimated_output_tokens = 100

# Orientacinės kainų prielaidos USD už 1 mln. tokenų
# Tai nėra patvirtintas dabartinis GPT-5.4-mini kainoraštis
input_price_per_million = 1.00
output_price_per_million = 4.00

def estimate_cost(n):
    input_cost = (
        n * estimated_input_tokens
        / 1_000_000 * input_price_per_million
    )

    output_cost = (
        n * estimated_output_tokens
        / 1_000_000 * output_price_per_million
    )

    return input_cost + output_cost

print("Vienos operacijos JSON ilgis:", len(example_json))
print("Apytikriai įvesties tokenai:", round(estimated_input_tokens))
print("Numatyti išvesties tokenai:", estimated_output_tokens)

print("\nPreliminarios išlaidos pagal nurodytas prielaidas:")
for n in [5, 10, 20, 200]:
    print(f"{n} operacijų: ${estimate_cost(n):.4f}")

print("\nAPI užklausos nevykdomos.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Atrenkame 5 operacijas
pilot = pd.concat([
    sample[sample["Is Laundering"] == 1].sample(
        n=3, random_state=42
    ),
    sample[sample["Is Laundering"] == 0].sample(
        n=2, random_state=42
    )
]).sample(frac=1, random_state=42).reset_index(drop=True)

results = []

for i, row in pilot.iterrows():

    transaction = prepare_transaction(
        row,
        base_features + context_columns
    )

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=prompt_v4,
        input=json.dumps(transaction, ensure_ascii=False),
        max_output_tokens=150
    )

    raw_response = response.output_text.strip()

    try:
        parsed = json.loads(raw_response)
        prediction = parsed.get("prediction")
        reason = parsed.get("reason", "")

        if type(prediction) is not int or prediction not in (0, 1):
            raise ValueError("Netinkama prediction reikšmė")

        status = "OK"

    except (json.JSONDecodeError, ValueError) as error:
        prediction = None
        reason = ""
        status = f"KLAIDA: {error}"

    results.append({
        "original_id": row["original_id"],
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": row["xgb_probability"],
        "gpt_prediction": prediction,
        "gpt_reason": reason,
        "raw_response": raw_response,
        "status": status,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    })

    print(
        f"Operacija {i + 1}/5 | "
        f"Tikroji klasė: {int(row['Is Laundering'])} | "
        f"GPT: {prediction} | {status}"
    )

# Išsaugome rezultatus
pilot_results = pd.DataFrame(results)

output_path = Path("aml_models/gpt_v4_high_risk_pilot_5.csv")
pilot_results.to_csv(output_path, index=False)

print("\nBandomojo vertinimo rezultatai:")
print(
    pilot_results[
        ["true_label", "gpt_prediction", "status",
         "input_tokens", "output_tokens"]
    ].to_string(index=False)
)

print("\nSėkmingų JSON atsakymų:",
      (pilot_results["status"] == "OK").sum(), "/ 5")

print("Bendras įvesties tokenų skaičius:",
      pilot_results["input_tokens"].sum())

print("Bendras išvesties tokenų skaičius:",
      pilot_results["output_tokens"].sum())

print("\nRezultatai išsaugoti:", output_path)


In [ ]:
import pandas as pd

pilot_results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_pilot_5.csv"
)

print("GPT V4 bandomųjų operacijų analizė:\n")

for i, row in pilot_results.iterrows():
    print(f"Operacija {i + 1}")
    print(f"Tikroji klasė: {int(row['true_label'])}")
    print(f"XGBoost tikimybė: {row['xgb_probability']:.4f}")
    print(f"GPT prognozė: {int(row['gpt_prediction'])}")
    print(f"GPT paaiškinimas: {row['gpt_reason']}")
    print("-" * 60)

print("\nXGBoost tikimybių statistika:")
print(pilot_results["xgb_probability"].describe())

print("\nGPT prognozių pasiskirstymas:")
print(pilot_results["gpt_prediction"].value_counts())

In [ ]:
import pandas as pd

sample = pd.read_csv(
    "aml_models/gpt_high_risk_200_clean_full_context.csv"
)

pilot_results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_pilot_5.csv"
)

sample_ids = set(sample["original_id"].astype(int))
pilot_ids = set(pilot_results["original_id"].astype(int))

overlap = sample_ids.intersection(pilot_ids)

print("Galutinės imties dydis:", len(sample))
print("Bandomųjų operacijų:", len(pilot_results))
print("Bandomųjų operacijų galutinėje imtyje:", len(overlap))

print("\nBandomųjų operacijų ID:")
print(sorted(pilot_ids))

print("\nDar neįvertintų operacijų:", len(sample_ids - pilot_ids))

assert len(sample) == 200
assert len(pilot_results) == 5
assert len(overlap) == 5
assert pilot_results["original_id"].is_unique
assert (pilot_results["status"] == "OK").all()

print("\nVisos 5 bandomosios operacijos priklauso galutinei imčiai.")
print("Jų pakartotinai vertinti nereikės.")

In [ ]:
import json
import time
import pandas as pd
from pathlib import Path

# Galutinė 200 operacijų imtis
sample = pd.read_csv(
    "aml_models/gpt_high_risk_200_clean_full_context.csv",
    parse_dates=["Timestamp"]
)

# Jau įvertintos 5 operacijos
pilot_results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_pilot_5.csv"
)

# Bendras rezultatų failas
output_path = Path(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

# Jei eksperimentas jau buvo pradėtas, tęsiame jį
if output_path.exists():
    results = pd.read_csv(output_path)
else:
    results = pilot_results.copy()
    results.to_csv(output_path, index=False)

# Kartojame tik tas operacijas, kurioms dar nėra
# sėkmingos GPT prognozės
completed_ids = set(
    results.loc[
        (results["status"] == "OK") &
        (results["gpt_prediction"].isin([0, 1])),
        "original_id"
    ].astype(int)
)

remaining = sample[
    ~sample["original_id"].astype(int).isin(completed_ids)
].copy()

print("Jau įvertinta:", len(completed_ids))
print("Liko įvertinti:", len(remaining))

for i, (_, row) in enumerate(remaining.iterrows(), start=1):

    transaction = prepare_transaction(
        row,
        base_features + context_columns
    )

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt_v4,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            max_output_tokens=150
        )

        raw_response = response.output_text.strip()

        try:
            parsed = json.loads(raw_response)

            prediction = parsed.get("prediction")
            reason = parsed.get("reason", "")

            if (
                type(prediction) is not int
                or prediction not in (0, 1)
            ):
                raise ValueError(
                    "Netinkama prediction reikšmė"
                )

            status = "OK"

        except (json.JSONDecodeError, ValueError) as error:
            prediction = None
            reason = ""
            status = f"KLAIDA: {error}"

        result = {
            "original_id": int(row["original_id"]),
            "true_label": int(row["Is Laundering"]),
            "xgb_probability": row["xgb_probability"],
            "gpt_prediction": prediction,
            "gpt_reason": reason,
            "raw_response": raw_response,
            "status": status,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens
        }

        results = pd.concat(
            [results, pd.DataFrame([result])],
            ignore_index=True
        )

        # Išsaugome po kiekvienos užklausos
        results.to_csv(output_path, index=False)

        print(
            f"{i}/{len(remaining)} | "
            f"ID: {int(row['original_id'])} | "
            f"GPT: {prediction} | {status}"
        )

        # Nedidelė pauzė tarp užklausų
        time.sleep(0.2)

    except Exception as error:
        print(
            f"\nVykdymas sustabdytas ties "
            f"ID {int(row['original_id'])}: {error}"
        )
        print("Ankstesni rezultatai išsaugoti.")
        break

# Galutinė suvestinė
successful = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id")

print("\n--- GPT V4 VERTINIMO SUVESTINĖ ---")
print("Sėkmingai įvertinta:", len(successful), "/ 200")
print("Liko:", 200 - len(successful))

print("\nGPT prognozių pasiskirstymas:")
print(successful["gpt_prediction"].value_counts())

print("\nSunaudoti įvesties tokenai:",
      int(results["input_tokens"].sum()))

print("Sunaudoti išvesties tokenai:",
      int(results["output_tokens"].sum()))

print("\nRezultatai išsaugoti:", output_path)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

# Paliekame tik sėkmingus, unikalius rezultatus
results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id")

assert len(results) == 200

y_true = results["true_label"].astype(int)
y_gpt = results["gpt_prediction"].astype(int)

# XGBoost sprendimas pagal anksčiau pasirinktą slenkstį
xgb_threshold = 0.957565

y_xgb = (
    results["xgb_probability"] >= xgb_threshold
).astype(int)

def evaluate_model(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "Modelis": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "FPR": fp / (fp + tn) if fp + tn else 0,
        "Accuracy": accuracy_score(y_true, y_pred)
    }

comparison = pd.DataFrame([
    evaluate_model("GPT V4", y_true, y_gpt),
    evaluate_model("XGBoost", y_true, y_xgb)
])

print("GPT V4 ir XGBoost palyginimas:\n")
print(comparison.round(4).to_string(index=False))

print("\nGPT V4 klaidų matrica:")
print(
    confusion_matrix(
        y_true, y_gpt, labels=[0, 1]
    )
)

print("\nXGBoost klaidų matrica:")
print(
    confusion_matrix(
        y_true, y_xgb, labels=[0, 1]
    )
)

comparison.to_csv(
    "aml_models/gpt_v4_xgb_high_risk_comparison.csv",
    index=False
)

print("\nPalyginimo rezultatai išsaugoti!")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

results["gpt_pred"] = results["gpt_prediction"].astype(int)

results["xgb_pred"] = (
    results["xgb_probability"] >= 0.957565
).astype(int)

# Modelių prognozių sujungimas
results["OR"] = (
    (results["gpt_pred"] == 1) |
    (results["xgb_pred"] == 1)
).astype(int)

results["AND"] = (
    (results["gpt_pred"] == 1) &
    (results["xgb_pred"] == 1)
).astype(int)

# AML operacijų pasiskirstymas
aml = results[results["true_label"] == 1]

print("AML operacijų aptikimas:")
print(
    "Aptiko abu modeliai:",
    ((aml["gpt_pred"] == 1) &
     (aml["xgb_pred"] == 1)).sum()
)
print(
    "Aptiko tik GPT:",
    ((aml["gpt_pred"] == 1) &
     (aml["xgb_pred"] == 0)).sum()
)
print(
    "Aptiko tik XGBoost:",
    ((aml["gpt_pred"] == 0) &
     (aml["xgb_pred"] == 1)).sum()
)
print(
    "Nepastebėjo abu:",
    ((aml["gpt_pred"] == 0) &
     (aml["xgb_pred"] == 0)).sum()
)

# Įvertiname sujungimo strategijas
rows = []

for model in ["gpt_pred", "xgb_pred", "OR", "AND"]:
    tn, fp, fn, tp = confusion_matrix(
        results["true_label"],
        results[model],
        labels=[0, 1]
    ).ravel()

    rows.append({
        "Modelis": model,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Recall": tp / (tp + fn),
        "FPR": fp / (fp + tn),
        "Precision": tp / (tp + fp) if tp + fp else 0
    })

hybrid_comparison = pd.DataFrame(rows)

print("\nModelių sujungimo rezultatai:")
print(
    hybrid_comparison.round(4).to_string(index=False)
)

hybrid_comparison.to_csv(
    "aml_models/gpt_xgb_hybrid_high_risk_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

y_true = results["true_label"].astype(int)
gpt_pred = results["gpt_prediction"].astype(int)
xgb_prob = results["xgb_probability"]

thresholds = [
    0.90, 0.91, 0.92, 0.93,
    0.94, 0.95, 0.957565,
    0.96, 0.97
]

rows = []

for threshold in thresholds:

    hybrid_pred = (
        (gpt_pred == 1) &
        (xgb_prob >= threshold)
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        hybrid_pred,
        labels=[0, 1]
    ).ravel()

    rows.append({
        "XGB_threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Recall": tp / (tp + fn),
        "FPR": fp / (fp + tn),
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "Alerts": tp + fp
    })

threshold_comparison = pd.DataFrame(rows)

print("Hibridinio modelio slenksčių palyginimas:\n")
print(
    threshold_comparison.round(4).to_string(index=False)
)

threshold_comparison.to_csv(
    "aml_models/gpt_xgb_hybrid_thresholds.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200

y_true = results["true_label"].astype(int)
gpt_pred = results["gpt_prediction"].astype(int)
xgb_prob = results["xgb_probability"]

thresholds = [
    0.90, 0.91, 0.92, 0.93,
    0.94, 0.95, 0.957565,
    0.96, 0.97
]

rows = []

for threshold in thresholds:

    xgb_pred = (xgb_prob >= threshold).astype(int)

    hybrid_pred = (
        (xgb_pred == 1) &
        (gpt_pred == 1)
    ).astype(int)

    for model_name, predictions in [
        ("XGBoost", xgb_pred),
        ("XGBoost + GPT V4", hybrid_pred)
    ]:
        tn, fp, fn, tp = confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1]
        ).ravel()

        rows.append({
            "Threshold": threshold,
            "Model": model_name,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Recall": tp / (tp + fn),
            "FPR": fp / (fp + tn),
            "Precision": (
                tp / (tp + fp) if tp + fp else 0
            ),
            "Alerts": tp + fp
        })

fair_comparison = pd.DataFrame(rows)

print("XGBoost ir hibridinio modelio palyginimas:\n")

print(
    fair_comparison.round(4).to_string(index=False)
)

fair_comparison.to_csv(
    "aml_models/xgb_vs_hybrid_same_thresholds.csv",
    index=False
)

print("\nPalyginimas išsaugotas!")

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200

results["true_label"] = results["true_label"].astype(int)
results["gpt_prediction"] = results["gpt_prediction"].astype(int)

# XGBoost reitingas pagal tikimybę
xgb_ranked = results.sort_values(
    "xgb_probability",
    ascending=False
)

# Hibridinis reitingas:
# pirmiausia GPT teigiamos prognozės,
# jų viduje – didžiausia XGBoost tikimybė
hybrid_ranked = results.sort_values(
    ["gpt_prediction", "xgb_probability"],
    ascending=[False, False]
)

budgets = [10, 20, 35, 50, 71, 100, 150, 178]

rows = []

for budget in budgets:
    for name, ranked in [
        ("XGBoost", xgb_ranked),
        ("Hybrid", hybrid_ranked)
    ]:
        selected = ranked.head(budget)

        tp = int(selected["true_label"].sum())
        fp = budget - tp

        rows.append({
            "Alerts": budget,
            "Model": name,
            "TP": tp,
            "FP": fp,
            "Recall": tp / 100,
            "Precision": tp / budget
        })

budget_comparison = pd.DataFrame(rows)

print("Modelių palyginimas pagal perspėjimų biudžetą:\n")
print(
    budget_comparison.round(4).to_string(index=False)
)

# Parodome, kiek papildomų AML aptinka hibridinis modelis
pivot = budget_comparison.pivot(
    index="Alerts",
    columns="Model",
    values="TP"
)

pivot["Hybrid_minus_XGBoost"] = (
    pivot["Hybrid"] - pivot["XGBoost"]
)

print("\nHibridinio modelio papildomai aptiktos AML operacijos:")
print(pivot.to_string())

budget_comparison.to_csv(
    "aml_models/xgb_hybrid_equal_alert_budget.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert results["true_label"].sum() == 100

# XGBoost reitingas
xgb_ranked = results.sort_values(
    ["xgb_probability", "original_id"],
    ascending=[False, True]
)

# Hibridinis reitingas
hybrid_ranked = results.sort_values(
    ["gpt_prediction", "xgb_probability", "original_id"],
    ascending=[False, False, True]
)

# Sukauptas aptiktų AML operacijų skaičius
xgb_tp = np.cumsum(
    xgb_ranked["true_label"].to_numpy(dtype=int)
)

hybrid_tp = np.cumsum(
    hybrid_ranked["true_label"].to_numpy(dtype=int)
)

budgets = np.arange(1, 201)

ranking_comparison = pd.DataFrame({
    "Alerts": budgets,
    "XGBoost_TP": xgb_tp,
    "Hybrid_TP": hybrid_tp,
    "Difference": hybrid_tp - xgb_tp
})

print("Vidutinis aptiktų AML skaičius per visus biudžetus:")
print("XGBoost:", round(xgb_tp.mean(), 2))
print("Hybrid:", round(hybrid_tp.mean(), 2))

print("\nBiudžetų skaičius, kai:")
print(
    "Hybrid geresnis:",
    int((ranking_comparison["Difference"] > 0).sum())
)
print(
    "XGBoost geresnis:",
    int((ranking_comparison["Difference"] < 0).sum())
)
print(
    "Rezultatai vienodi:",
    int((ranking_comparison["Difference"] == 0).sum())
)

print("\nDidžiausias hibridinio metodo pranašumas:")
print(
    ranking_comparison.loc[
        ranking_comparison["Difference"].idxmax()
    ].to_string()
)

print("\nDidžiausias XGBoost pranašumas:")
print(
    ranking_comparison.loc[
        ranking_comparison["Difference"].idxmin()
    ].to_string()
)

ranking_comparison.to_csv(
    "aml_models/xgb_hybrid_full_budget_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200

rng = np.random.default_rng(42)

aml = results[results["true_label"] == 1]
legit = results[results["true_label"] == 0]

def mean_detected_difference(data):
    xgb_ranked = data.sort_values(
        "xgb_probability",
        ascending=False,
        kind="stable"
    )

    hybrid_ranked = data.sort_values(
        ["gpt_prediction", "xgb_probability"],
        ascending=[False, False],
        kind="stable"
    )

    xgb_mean = np.cumsum(
        xgb_ranked["true_label"].to_numpy(dtype=int)
    ).mean()

    hybrid_mean = np.cumsum(
        hybrid_ranked["true_label"].to_numpy(dtype=int)
    ).mean()

    return hybrid_mean - xgb_mean

observed_difference = mean_detected_difference(results)

bootstrap_differences = []

for _ in range(2000):
    aml_sample = aml.iloc[
        rng.integers(0, len(aml), len(aml))
    ]

    legit_sample = legit.iloc[
        rng.integers(0, len(legit), len(legit))
    ]

    bootstrap_sample = pd.concat(
        [aml_sample, legit_sample],
        ignore_index=True
    )

    bootstrap_differences.append(
        mean_detected_difference(bootstrap_sample)
    )

bootstrap_differences = np.array(bootstrap_differences)

ci_lower, ci_upper = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)

print("Stebimas vidutinis skirtumas:",
      round(observed_difference, 4))

print("95 % bootstrap pasikliautinasis intervalas:")
print(
    f"[{ci_lower:.4f}, {ci_upper:.4f}]"
)

print(
    "Bootstrap imčių, kuriose hibridinis metodas geresnis:",
    f"{(bootstrap_differences > 0).mean():.2%}"
)

print(
    "Ar intervalas apima nulį?",
    ci_lower <= 0 <= ci_upper
)

pd.DataFrame({
    "difference": bootstrap_differences
}).to_csv(
    "aml_models/xgb_hybrid_bootstrap.csv",
    index=False
)

print("\nBootstrap rezultatai išsaugoti!")

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

# Patikriname vienodas tikimybes
prob_counts = results["xgb_probability"].value_counts()

print("Unikalių XGBoost tikimybių:", results["xgb_probability"].nunique())
print("Pasikartojančių tikimybių grupių:", (prob_counts > 1).sum())
print("Operacijų pasikartojančiose grupėse:", int(prob_counts[prob_counts > 1].sum()))

# Vienoda rūšiavimo taisyklė abiem metodams
xgb_ranked = results.sort_values(
    ["xgb_probability", "original_id"],
    ascending=[False, True]
)

hybrid_ranked = results.sort_values(
    ["gpt_prediction", "xgb_probability", "original_id"],
    ascending=[False, False, True]
)

xgb_cumulative = np.cumsum(
    xgb_ranked["true_label"].astype(int).to_numpy()
)

hybrid_cumulative = np.cumsum(
    hybrid_ranked["true_label"].astype(int).to_numpy()
)

difference = hybrid_cumulative - xgb_cumulative

print("\nVidutinis XGBoost aptikimas:", round(xgb_cumulative.mean(), 4))
print("Vidutinis hibridinis aptikimas:", round(hybrid_cumulative.mean(), 4))
print("Vidutinis skirtumas:", round(difference.mean(), 4))

print("\nBiudžetų skaičius:")
print("Hibridinis geresnis:", int((difference > 0).sum()))
print("XGBoost geresnis:", int((difference < 0).sum()))
print("Vienodi:", int((difference == 0).sum()))

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert (results["true_label"] == 1).sum() == 100
assert (results["true_label"] == 0).sum() == 100

def ranking_difference(data):
    xgb_ranked = data.sort_values(
        ["xgb_probability", "original_id"],
        ascending=[False, True],
        kind="stable"
    )

    hybrid_ranked = data.sort_values(
        ["gpt_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True],
        kind="stable"
    )

    xgb_tp = np.cumsum(
        xgb_ranked["true_label"].to_numpy(dtype=int)
    )

    hybrid_tp = np.cumsum(
        hybrid_ranked["true_label"].to_numpy(dtype=int)
    )

    return (hybrid_tp - xgb_tp).mean()

observed = ranking_difference(results)

aml = results[results["true_label"] == 1]
legit = results[results["true_label"] == 0]

rng = np.random.default_rng(42)
differences = []

for _ in range(2000):
    aml_sample = aml.iloc[
        rng.integers(0, len(aml), len(aml))
    ]

    legit_sample = legit.iloc[
        rng.integers(0, len(legit), len(legit))
    ]

    bootstrap_sample = pd.concat(
        [aml_sample, legit_sample],
        ignore_index=True
    )

    differences.append(
        ranking_difference(bootstrap_sample)
    )

differences = np.array(differences)

lower, upper = np.percentile(
    differences, [2.5, 97.5]
)

print("Stebimas skirtumas:", round(observed, 4))
print(
    "95 % bootstrap intervalas:",
    f"[{lower:.4f}, {upper:.4f}]"
)
print(
    "Hibridinis metodas geresnis bootstrap imtyse:",
    f"{(differences > 0).mean():.2%}"
)
print(
    "Intervalas apima nulį:",
    lower <= 0 <= upper
)

pd.DataFrame({
    "difference": differences
}).to_csv(
    "aml_models/xgb_hybrid_bootstrap_consistent.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import os
import pandas as pd

files_to_check = [
    "aml_models/test_3000_natural.csv",
    "aml_models/test_3000_context.csv",
    "aml_models/comparison_3000_context.csv",
    "aml_models/gpt_v4_high_risk_200_results.csv"
]

for path in files_to_check:
    print(f"\nFailas: {path}")

    if not os.path.exists(path):
        print("NERASTAS")
        continue

    data = pd.read_csv(path)

    print("Eilučių skaičius:", len(data))
    print("Stulpeliai:", list(data.columns))

    if "Is Laundering" in data.columns:
        print(
            "AML operacijų skaičius:",
            int(data["Is Laundering"].sum())
        )

    elif "true_label" in data.columns:
        print(
            "AML operacijų skaičius:",
            int(data["true_label"].sum())
        )

    elif "label" in data.columns:
        print(
            "AML operacijų skaičius:",
            int(data["label"].sum())
        )

    if "gpt_prediction" in data.columns:
        print(
            "GPT prognozių skaičius:",
            int(data["gpt_prediction"].notna().sum())
        )

print("\nPatikra baigta.")

In [ ]:
import numpy as np
import pandas as pd
import joblib

model = joblib.load("aml_models/xgb_model.joblib")
preprocessor = joblib.load("aml_models/preprocessor.joblib")

test_data = test_full.copy()

test_data["Timestamp"] = pd.to_datetime(
    test_data["Timestamp"]
)

test_data["Timestamp_sec"] = (
    test_data["Timestamp"] -
    pd.to_datetime(df["Timestamp"]).min()
).dt.total_seconds()

feature_columns = list(preprocessor.feature_names_in_)

X_test = preprocessor.transform(
    test_data[feature_columns]
)

test_data["xgb_probability"] = model.predict_proba(
    X_test
)[:, 1]

thresholds = [0.50, 0.70, 0.90, 0.95]

print("Nepriklausoma testavimo imtis:", len(test_data))
print("Visos AML operacijos:", int(test_data["Is Laundering"].sum()))

for threshold in thresholds:
    selected = test_data[
        test_data["xgb_probability"] >= threshold
    ]

    aml_count = int(selected["Is Laundering"].sum())
    legit_count = len(selected) - aml_count

    print(
        f"\nXGBoost >= {threshold:.2f}:"
        f"\n  Visos operacijos: {len(selected)}"
        f"\n  AML: {aml_count}"
        f"\n  Teisėtos: {legit_count}"
    )

print("\nTestavimo laikotarpis:")
print("Nuo:", test_data["Timestamp"].min())
print("Iki:", test_data["Timestamp"].max())

In [ ]:
import pandas as pd

# Tik nepriklausomo testavimo laikotarpio aukštos rizikos operacijos
high_risk_test = test_data[
    test_data["xgb_probability"] >= 0.90
].copy()

# Atskirai atrenkame AML ir teisėtas operacijas
aml_sample = high_risk_test[
    high_risk_test["Is Laundering"] == 1
].sample(n=100, random_state=42)

legit_sample = high_risk_test[
    high_risk_test["Is Laundering"] == 0
].sample(n=100, random_state=42)

# Išsaugome originalų duomenų indeksą
test_200 = pd.concat(
    [aml_sample, legit_sample]
).sample(frac=1, random_state=42)

test_200["original_id"] = test_200.index

test_200 = test_200.reset_index(drop=True)

assert len(test_200) == 200
assert test_200["original_id"].is_unique
assert test_200["Is Laundering"].sum() == 100

test_200.to_csv(
    "aml_models/gpt_v4_independent_test_200.csv",
    index=False
)

print("Nepriklausoma GPT V4 testavimo imtis sukurta.")
print("Operacijų:", len(test_200))
print("AML:", int(test_200["Is Laundering"].sum()))
print("Teisėtų:", int((test_200["Is Laundering"] == 0).sum()))
print("Unikalūs ID:", test_200["original_id"].nunique())
print("Mažiausia XGBoost tikimybė:", round(test_200["xgb_probability"].min(), 6))
print("Didžiausia XGBoost tikimybė:", round(test_200["xgb_probability"].max(), 6))
print("Laikotarpis:", test_200["Timestamp"].min(), "–", test_200["Timestamp"].max())
print("\nFailas: aml_models/gpt_v4_independent_test_200.csv")

In [ ]:
import pandas as pd
import numpy as np
import os

test_200 = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200.csv"
)

test_200["Timestamp"] = pd.to_datetime(test_200["Timestamp"])

history = df[
    ["Timestamp", "From Bank", "Account", "To Bank", "Account.1"]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

sender_cols = ["From Bank", "Account"]
recipient_cols = ["To Bank", "Account.1"]

# Skaičiuojame tik reikalingų siuntėjo–gavėjo porų istoriją
pairs = test_200[
    sender_cols + recipient_cols
].drop_duplicates()

relevant_history = history.merge(
    pairs,
    on=sender_cols + recipient_cols,
    how="inner"
)

# Kiekvienai operacijai skaičiuojame ankstesnius
# tos pačios poros pervedimus per 7 dienas
test_200["same_recipient_7d_count"] = 0

for keys, group in test_200.groupby(
    sender_cols + recipient_cols,
    dropna=False
):
    if not isinstance(keys, tuple):
        keys = (keys,)

    mask = np.ones(len(relevant_history), dtype=bool)

    for col, value in zip(
        sender_cols + recipient_cols, keys
    ):
        if pd.isna(value):
            mask &= relevant_history[col].isna().to_numpy()
        else:
            mask &= relevant_history[col].eq(value).to_numpy()

    timestamps = np.sort(
        relevant_history.loc[
            mask, "Timestamp"
        ].to_numpy(dtype="datetime64[ns]")
    )

    current_times = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    left = np.searchsorted(
        timestamps,
        current_times - np.timedelta64(7, "D"),
        side="left"
    )

    right = np.searchsorted(
        timestamps,
        current_times,
        side="left"
    )

    test_200.loc[
        group.index, "same_recipient_7d_count"
    ] = right - left

output_path = (
    "aml_models/gpt_v4_independent_test_200_recipient_context.csv"
)

test_200.to_csv(output_path, index=False)

print("Operacijų:", len(test_200))
print(
    "Trūkstamų reikšmių:",
    test_200["same_recipient_7d_count"].isna().sum()
)
print(
    "Operacijų be ankstesnių pervedimų tam pačiam gavėjui:",
    int((test_200["same_recipient_7d_count"] == 0).sum())
)

print("\nPožymio statistika pagal tikrąją klasę:")
print(
    test_200.groupby("Is Laundering")[
        "same_recipient_7d_count"
    ].describe().round(2).to_string()
)

print("\nFailas išsaugotas:", output_path)

In [ ]:
import pandas as pd

names_to_check = [
    "counts",
    "context_features",
    "high_risk_200_clean",
    "test_200",
    "test_200_context",
    "test_200_full_context"
]

for name in names_to_check:
    print(f"\n{name}:")

    if name not in globals():
        print("Nėra atmintyje")
        continue

    obj = globals()[name]

    if isinstance(obj, pd.DataFrame):
        print("Eilučių:", len(obj))
        print("Stulpeliai:", list(obj.columns))
        print(obj.head(2).to_string(index=False))

    else:
        print("Tipas:", type(obj).__name__)
        print("Reikšmė:", str(obj)[:500])

print("\nPatikra baigta.")

In [ ]:
import pandas as pd
import numpy as np

test_200 = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_recipient_context.csv"
)

test_200["Timestamp"] = pd.to_datetime(test_200["Timestamp"])

# Prijungiame jau apskaičiuotus istorinius skaičius
counts_ready = counts[
    ["From Bank", "Account", "Timestamp",
     "previous_24h_count", "previous_7d_count"]
].copy()

counts_ready["Timestamp"] = pd.to_datetime(
    counts_ready["Timestamp"]
)

test_200 = test_200.merge(
    counts_ready,
    on=["From Bank", "Account", "Timestamp"],
    how="left",
    validate="many_to_one"
)

# Visa operacijų istorija reikalingiems siuntėjams
sender_keys = test_200[
    ["From Bank", "Account"]
].drop_duplicates()

history = df[
    ["Timestamp", "From Bank", "Account",
     "To Bank", "Account.1", "Amount Paid"]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

history = history.merge(
    sender_keys,
    on=["From Bank", "Account"],
    how="inner"
)

test_200["previous_24h_amount"] = np.nan
test_200["previous_7d_unique_recipients"] = np.nan

# Grupavimas pagal siuntėją
history_groups = {
    key: group.sort_values("Timestamp")
    for key, group in history.groupby(
        ["From Bank", "Account"], sort=False
    )
}

for key, group in test_200.groupby(
    ["From Bank", "Account"], sort=False
):
    sender_history = history_groups.get(key)

    if sender_history is None:
        continue

    timestamps = sender_history["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    amounts = sender_history["Amount Paid"].to_numpy(
        dtype=float
    )

    recipients = list(zip(
        sender_history["To Bank"],
        sender_history["Account.1"]
    ))

    cumulative_amount = np.concatenate([
        [0.0], np.cumsum(amounts)
    ])

    for idx, row in group.iterrows():
        current_time = np.datetime64(row["Timestamp"])

        end = np.searchsorted(
            timestamps, current_time, side="left"
        )

        start_24h = np.searchsorted(
            timestamps,
            current_time - np.timedelta64(24, "h"),
            side="left"
        )

        start_7d = np.searchsorted(
            timestamps,
            current_time - np.timedelta64(7, "D"),
            side="left"
        )

        test_200.at[idx, "previous_24h_amount"] = (
            cumulative_amount[end] -
            cumulative_amount[start_24h]
        )

        test_200.at[
            idx, "previous_7d_unique_recipients"
        ] = len(set(recipients[start_7d:end]))

context_cols = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

print("Operacijų:", len(test_200))
print("\nTrūkstamos reikšmės:")
print(test_200[context_cols].isna().sum().to_string())

print("\nPožymių statistika:")
print(test_200[context_cols].describe().round(2).to_string())

if test_200[context_cols].isna().sum().sum() == 0:
    test_200.to_csv(
        "aml_models/gpt_v4_independent_test_200_full_context.csv",
        index=False
    )
    print("\nVisi 5 požymiai paruošti ir failas išsaugotas!")
else:
    print("\nYra trūkstamų reikšmių – failas neišsaugotas.")

In [ ]:
import pandas as pd

test_check = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_full_context.csv"
)

columns = [
    "original_id",
    "Timestamp",
    "Is Laundering",
    "Amount Paid",
    "Payment Currency",
    "previous_24h_count",
    "previous_24h_amount",
    "previous_7d_count"
]

print("10 didžiausių ankstesnių 24 val. sumų:\n")

print(
    test_check.nlargest(
        10, "previous_24h_amount"
    )[columns].to_string(index=False)
)

print("\nLoginės patikros:")

print(
    "Suma > 0, nors ankstesnių operacijų nėra:",
    int((
        (test_check["previous_24h_count"] == 0) &
        (test_check["previous_24h_amount"].abs() > 0.01)
    ).sum())
)

print(
    "24 val. operacijų daugiau nei 7 dienų:",
    int((
        test_check["previous_24h_count"] >
        test_check["previous_7d_count"]
    ).sum())
)

print(
    "Neigiamos istorinės sumos:",
    int((test_check["previous_24h_amount"] < -0.01).sum())
)

In [ ]:
import pandas as pd

test_check = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_full_context.csv"
)

test_check["Timestamp"] = pd.to_datetime(
    test_check["Timestamp"]
)

history = df[
    ["Timestamp", "From Bank", "Account",
     "Amount Paid", "Payment Currency"]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

sender_keys = test_check[
    ["From Bank", "Account"]
].drop_duplicates()

history = history.merge(
    sender_keys,
    on=["From Bank", "Account"],
    how="inner"
)

history_groups = {
    key: group.sort_values("Timestamp")
    for key, group in history.groupby(
        ["From Bank", "Account"], sort=False
    )
}

rows = []

for _, row in test_check.iterrows():
    key = (row["From Bank"], row["Account"])
    group = history_groups[key]

    start = row["Timestamp"] - pd.Timedelta(hours=24)

    previous = group[
        (group["Timestamp"] >= start) &
        (group["Timestamp"] < row["Timestamp"])
    ]

    currencies = previous["Payment Currency"].dropna().unique()

    rows.append({
        "original_id": row["original_id"],
        "true_label": row["Is Laundering"],
        "previous_24h_count": len(previous),
        "currency_count": len(currencies),
        "mixed_currencies": len(currencies) > 1,
        "different_from_current": (
            len(currencies) > 0 and
            any(c != row["Payment Currency"] for c in currencies)
        ),
        "currencies": ", ".join(map(str, currencies))
    })

currency_check = pd.DataFrame(rows)

print("Operacijų su mišriomis ankstesnių mokėjimų valiutomis:",
      int(currency_check["mixed_currencies"].sum()))

print("Operacijų, kurių ankstesnių mokėjimų valiuta skiriasi nuo dabartinės:",
      int(currency_check["different_from_current"].sum()))

print("\nPagal tikrąją klasę:")
print(
    currency_check.groupby("true_label")[
        ["mixed_currencies", "different_from_current"]
    ].sum().to_string()
)

print("\nMišrių valiutų pavyzdžiai:")
print(
    currency_check[
        currency_check["mixed_currencies"]
    ].head(10).to_string(index=False)
)

currency_check.to_csv(
    "aml_models/gpt_v4_test_currency_check.csv",
    index=False
)

print("\nPatikra išsaugota!")

In [ ]:
import os
import hashlib

prompt_path = "aml_models/gpt_context_prompt_v4.txt"

if os.path.exists(prompt_path):
    with open(prompt_path, "r", encoding="utf-8") as f:
        saved_prompt = f.read()

    print("GPT V4 instrukcijos ilgis:", len(saved_prompt))
    print(
        "SHA-256:",
        hashlib.sha256(saved_prompt.encode("utf-8")).hexdigest()
    )

    print("\nInstrukcijos eilutės, susijusios su istorine suma:\n")

    lines = saved_prompt.splitlines()

    keywords = [
        "previous_24h_amount",
        "currency",
        "currencies",
        "amount",
        "valiut"
    ]

    matching_indices = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()

    for i in matching_indices:
        for j in range(max(0, i - 1), min(len(lines), i + 2)):
            if j not in shown:
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)
else:
    print("GPT V4 instrukcijos failas nerastas.")

print("\nAr atmintyje yra prompt_v4?", "prompt_v4" in globals())

if "prompt_v4" in globals():
    print(
        "Ar failo turinys sutampa su atmintyje esančia instrukcija?",
        saved_prompt == prompt_v4 if os.path.exists(prompt_path) else "Nežinoma"
    )

In [ ]:
import pandas as pd
import numpy as np

path = "aml_models/gpt_v4_independent_test_200_full_context.csv"

test_ready = pd.read_csv(path)
test_ready["Timestamp"] = pd.to_datetime(test_ready["Timestamp"])

history = df[
    [
        "Timestamp", "From Bank", "Account",
        "Amount Paid", "Payment Currency"
    ]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Tik testavimo imčiai reikalingos siuntėjo ir valiutos poros
keys = test_ready[
    ["From Bank", "Account", "Payment Currency"]
].drop_duplicates()

history = history.merge(
    keys,
    on=["From Bank", "Account", "Payment Currency"],
    how="inner"
)

history_groups = {
    key: group.sort_values("Timestamp")
    for key, group in history.groupby(
        ["From Bank", "Account", "Payment Currency"],
        sort=False
    )
}

corrected_amounts = []

for _, row in test_ready.iterrows():
    key = (
        row["From Bank"],
        row["Account"],
        row["Payment Currency"]
    )

    group = history_groups.get(key)

    if group is None:
        corrected_amounts.append(0.0)
        continue

    start = row["Timestamp"] - pd.Timedelta(hours=24)

    previous = group[
        (group["Timestamp"] >= start) &
        (group["Timestamp"] < row["Timestamp"])
    ]

    corrected_amounts.append(
        float(previous["Amount Paid"].sum())
    )

test_ready["previous_24h_amount_original"] = (
    test_ready["previous_24h_amount"]
)

test_ready["previous_24h_amount"] = corrected_amounts

difference = (
    test_ready["previous_24h_amount_original"] -
    test_ready["previous_24h_amount"]
).abs()

print("Operacijų:", len(test_ready))
print("Pakeistų sumų skaičius:", int((difference > 0.01).sum()))

print("\nDidžiausi pakeitimai:")
print(
    test_ready.assign(difference=difference)
    .nlargest(10, "difference")[
        [
            "original_id",
            "Is Laundering",
            "Payment Currency",
            "previous_24h_amount_original",
            "previous_24h_amount",
            "difference"
        ]
    ].to_string(index=False)
)

print("\nTrūkstamų reikšmių:")
print(
    test_ready[
        [
            "previous_24h_count",
            "previous_7d_count",
            "previous_24h_amount",
            "previous_7d_unique_recipients",
            "same_recipient_7d_count"
        ]
    ].isna().sum().to_string()
)

output_path = (
    "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
)

test_ready.to_csv(output_path, index=False)

print("\nIšsaugota:", output_path)

In [ ]:
import pandas as pd
import hashlib

test_ready = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
)

validation_results = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

context_cols = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

overlap = set(test_ready["original_id"]) & set(
    validation_results["original_id"]
)

with open(
    "aml_models/gpt_context_prompt_v4.txt",
    "r",
    encoding="utf-8"
) as f:
    saved_prompt = f.read()

prompt_hash = hashlib.sha256(
    saved_prompt.encode("utf-8")
).hexdigest()

checks = {
    "Tiksliai 200 operacijų": len(test_ready) == 200,
    "100 AML operacijų": int(test_ready["Is Laundering"].sum()) == 100,
    "100 teisėtų operacijų": int((test_ready["Is Laundering"] == 0).sum()) == 100,
    "Unikalūs original_id": test_ready["original_id"].is_unique,
    "Nėra sutapimų su validacijos imtimi": len(overlap) == 0,
    "Visi 5 požymiai užpildyti": test_ready[context_cols].notna().all().all(),
    "Visi kontekstiniai požymiai neneigiami": (
        test_ready[context_cols] >= 0
    ).all().all(),
    "XGBoost tikimybės >= 0.90": (
        test_ready["xgb_probability"] >= 0.90
    ).all(),
    "GPT V4 instrukcija nepakeista": (
        prompt_hash ==
        "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
    )
}

print("GALUTINĖ TESTAVIMO DUOMENŲ PATIKRA\n")

for name, passed in checks.items():
    print(f"{'OK' if passed else 'KLAIDA'} – {name}")

print("\nSutampančių ID:", len(overlap))
print("GPT V4 instrukcijos SHA-256:", prompt_hash)

print(
    "\nBendra išvada:",
    "DUOMENYS PARUOŠTI" if all(checks.values())
    else "REIKIA PAPILDOMOS PATIKROS"
)

In [ ]:
import pandas as pd

previous = pd.read_csv(
    "aml_models/gpt_v4_high_risk_200_results.csv"
)

previous = previous[previous["status"] == "OK"]

input_tokens = int(previous["input_tokens"].sum())
output_tokens = int(previous["output_tokens"].sum())

estimated_input = input_tokens
estimated_output = output_tokens

print("Ankstesnio eksperimento užklausų:", len(previous))
print("Įvesties žetonai:", input_tokens)
print("Išvesties žetonai:", output_tokens)

print("\nNumatomas naujo 200 operacijų eksperimento sunaudojimas:")
print("Įvesties žetonai:", estimated_input)
print("Išvesties žetonai:", estimated_output)

print("\nVidutiniškai vienai operacijai:")
print("Įvestis:", round(input_tokens / len(previous), 1))
print("Išvestis:", round(output_tokens / len(previous), 1))

In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI

client = OpenAI()

input_path = "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
output_path = "aml_models/gpt_v4_independent_test_200_results.csv"
prompt_path = "aml_models/gpt_context_prompt_v4.txt"

data = pd.read_csv(input_path)

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt_v4_test = f.read()

base_features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

context_features_v4 = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

features = base_features + context_features_v4

assert len(data) == 200
assert data["original_id"].is_unique
assert data[features].notna().all().all()

def make_transaction(row):
    transaction = {}

    for col in features:
        value = row[col]

        if hasattr(value, "item"):
            value = value.item()

        transaction[col] = value

    return transaction

columns = [
    "original_id",
    "true_label",
    "xgb_probability",
    "gpt_prediction",
    "gpt_reason",
    "raw_response",
    "status",
    "input_tokens",
    "output_tokens"
]

if os.path.exists(output_path):
    results = pd.read_csv(output_path)
    results = results[columns].copy()
    results = results.drop_duplicates(
        "original_id", keep="last"
    )
else:
    results = pd.DataFrame(columns=columns)

completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

print("Iš viso operacijų:", len(data))
print("Jau sėkmingai apdorota:", len(completed))
print("Liko:", len(data) - len(completed))
print()

for i, (_, row) in enumerate(data.iterrows(), start=1):
    original_id = int(row["original_id"])

    if original_id in completed:
        continue

    transaction = make_transaction(row)

    record = {
        "original_id": original_id,
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt_v4_test,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            max_output_tokens=150
        )

        raw = response.output_text.strip()
        record["raw_response"] = raw

        record["input_tokens"] = (
            response.usage.input_tokens
            if response.usage else 0
        )
        record["output_tokens"] = (
            response.usage.output_tokens
            if response.usage else 0
        )

        parsed = json.loads(raw)

        prediction = parsed.get("prediction")

        if type(prediction) is int and prediction in (0, 1):
            record["gpt_prediction"] = prediction
            record["gpt_reason"] = str(
                parsed.get("reason", "")
            )
            record["status"] = "OK"
        else:
            record["status"] = "INVALID_PREDICTION"

    except Exception as e:
        record["status"] = "ERROR"
        record["raw_response"] = str(e)[:500]

    results = results[
        results["original_id"] != original_id
    ]

    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )

    # Išsaugome po kiekvienos užklausos
    results.to_csv(output_path, index=False)

    if record["status"] == "OK":
        completed.add(original_id)

    if i % 10 == 0 or record["status"] != "OK":
        print(
            f"Operacija {i}/200 | "
            f"Būsena: {record['status']} | "
            f"Sėkmingai: {len(completed)}/200"
        )

    # Sustabdome, jei API grąžina klaidą,
    # kad nekartotume mokamų užklausų be priežiūros
    if record["status"] == "ERROR":
        print("Vykdymas sustabdytas. Patikrink klaidą.")
        break

print("\nGALUTINĖ SUVESTINĖ")
print(results["status"].value_counts().to_string())

print(
    "\nĮvesties žetonai:",
    int(pd.to_numeric(
        results["input_tokens"], errors="coerce"
    ).sum())
)

print(
    "Išvesties žetonai:",
    int(pd.to_numeric(
        results["output_tokens"], errors="coerce"
    ).sum())
)

print("\nRezultatai išsaugoti:", output_path)

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Įvesk savo OpenAI API raktą (įvedimas bus paslėptas): "
    ).strip()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("OpenAI klientas sėkmingai sukurtas!")
print("API raktas nustatytas:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI

client = OpenAI()

input_path = "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
output_path = "aml_models/gpt_v4_independent_test_200_results.csv"
prompt_path = "aml_models/gpt_context_prompt_v4.txt"

data = pd.read_csv(input_path)

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt_v4_test = f.read()

base_features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

context_features_v4 = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

features = base_features + context_features_v4

assert len(data) == 200
assert data["original_id"].is_unique
assert data[features].notna().all().all()

def make_transaction(row):
    transaction = {}

    for col in features:
        value = row[col]

        if hasattr(value, "item"):
            value = value.item()

        transaction[col] = value

    return transaction

columns = [
    "original_id",
    "true_label",
    "xgb_probability",
    "gpt_prediction",
    "gpt_reason",
    "raw_response",
    "status",
    "input_tokens",
    "output_tokens"
]

if os.path.exists(output_path):
    results = pd.read_csv(output_path)
    results = results[columns].copy()
    results = results.drop_duplicates(
        "original_id", keep="last"
    )
else:
    results = pd.DataFrame(columns=columns)

completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

print("Iš viso operacijų:", len(data))
print("Jau sėkmingai apdorota:", len(completed))
print("Liko:", len(data) - len(completed))
print()

for i, (_, row) in enumerate(data.iterrows(), start=1):
    original_id = int(row["original_id"])

    if original_id in completed:
        continue

    transaction = make_transaction(row)

    record = {
        "original_id": original_id,
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt_v4_test,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            max_output_tokens=150
        )

        raw = response.output_text.strip()
        record["raw_response"] = raw

        record["input_tokens"] = (
            response.usage.input_tokens
            if response.usage else 0
        )
        record["output_tokens"] = (
            response.usage.output_tokens
            if response.usage else 0
        )

        parsed = json.loads(raw)

        prediction = parsed.get("prediction")

        if type(prediction) is int and prediction in (0, 1):
            record["gpt_prediction"] = prediction
            record["gpt_reason"] = str(
                parsed.get("reason", "")
            )
            record["status"] = "OK"
        else:
            record["status"] = "INVALID_PREDICTION"

    except Exception as e:
        record["status"] = "ERROR"
        record["raw_response"] = str(e)[:500]

    results = results[
        results["original_id"] != original_id
    ]

    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )

    # Išsaugome po kiekvienos užklausos
    results.to_csv(output_path, index=False)

    if record["status"] == "OK":
        completed.add(original_id)

    if i % 10 == 0 or record["status"] != "OK":
        print(
            f"Operacija {i}/200 | "
            f"Būsena: {record['status']} | "
            f"Sėkmingai: {len(completed)}/200"
        )

    # Sustabdome, jei API grąžina klaidą,
    # kad nekartotume mokamų užklausų be priežiūros
    if record["status"] == "ERROR":
        print("Vykdymas sustabdytas. Patikrink klaidą.")
        break

print("\nGALUTINĖ SUVESTINĖ")
print(results["status"].value_counts().to_string())

print(
    "\nĮvesties žetonai:",
    int(pd.to_numeric(
        results["input_tokens"], errors="coerce"
    ).sum())
)

print(
    "Išvesties žetonai:",
    int(pd.to_numeric(
        results["output_tokens"], errors="coerce"
    ).sum())
)

print("\nRezultatai išsaugoti:", output_path)

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert results["true_label"].sum() == 100

y_true = results["true_label"].astype(int)

gpt_pred = results["gpt_prediction"].astype(int)

# Anksčiau validacijos metu nustatytas XGBoost slenkstis
xgb_threshold = 0.957565

xgb_pred = (
    results["xgb_probability"] >= xgb_threshold
).astype(int)

def evaluate_model(name, predictions):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Model": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "Recall": tp / (tp + fn) if tp + fn else 0,
        "F1": 2 * tp / (2 * tp + fp + fn)
              if 2 * tp + fp + fn else 0,
        "FPR": fp / (fp + tn) if fp + tn else 0,
        "Accuracy": (tp + tn) / len(y_true)
    }

comparison = pd.DataFrame([
    evaluate_model("XGBoost", xgb_pred),
    evaluate_model("GPT V4", gpt_pred)
])

print("NEPRIKLAUSOMO TESTAVIMO REZULTATAI\n")
print(comparison.round(4).to_string(index=False))

print("\nAML aptikimo sutapimai:")

aml_mask = y_true == 1

print(
    "Aptiko abu modeliai:",
    int((aml_mask & (gpt_pred == 1) & (xgb_pred == 1)).sum())
)

print(
    "Aptiko tik GPT:",
    int((aml_mask & (gpt_pred == 1) & (xgb_pred == 0)).sum())
)

print(
    "Aptiko tik XGBoost:",
    int((aml_mask & (gpt_pred == 0) & (xgb_pred == 1)).sum())
)

print(
    "Neaptiko nė vienas:",
    int((aml_mask & (gpt_pred == 0) & (xgb_pred == 0)).sum())
)

comparison.to_csv(
    "aml_models/gpt_v4_independent_test_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200

y_true = results["true_label"].astype(int)

gpt_pred = results["gpt_prediction"].astype(int)

xgb_pred = (
    results["xgb_probability"] >= 0.957565
).astype(int)

strategies = {
    "XGBoost": xgb_pred,
    "GPT V4": gpt_pred,
    "OR": ((xgb_pred == 1) | (gpt_pred == 1)).astype(int),
    "AND": ((xgb_pred == 1) & (gpt_pred == 1)).astype(int)
}

rows = []

for name, predictions in strategies.items():
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    rows.append({
        "Model": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Recall": tp / (tp + fn),
        "FPR": fp / (fp + tn),
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "F1": (
            2 * tp / (2 * tp + fp + fn)
            if 2 * tp + fp + fn else 0
        ),
        "Alerts": tp + fp
    })

hybrid_comparison = pd.DataFrame(rows)

print("NEPRIKLAUSOMAS HIBRIDINIŲ STRATEGIJŲ PALYGINIMAS\n")
print(hybrid_comparison.round(4).to_string(index=False))

hybrid_comparison.to_csv(
    "aml_models/gpt_v4_independent_hybrid_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert results["true_label"].sum() == 100

# XGBoost: rikiuojame pagal rizikos tikimybę
xgb_ranked = results.sort_values(
    ["xgb_probability", "original_id"],
    ascending=[False, True]
)

# Hibridinis metodas:
# pirmiausia GPT pažymėtos operacijos,
# tada pagal XGBoost tikimybę
hybrid_ranked = results.sort_values(
    ["gpt_prediction", "xgb_probability", "original_id"],
    ascending=[False, False, True]
)

xgb_tp = np.cumsum(
    xgb_ranked["true_label"].astype(int).to_numpy()
)

hybrid_tp = np.cumsum(
    hybrid_ranked["true_label"].astype(int).to_numpy()
)

budgets = [10, 20, 35, 50, 71, 100, 150, 178]

rows = []

for budget in budgets:
    rows.append({
        "Alerts": budget,
        "XGBoost_TP": int(xgb_tp[budget - 1]),
        "Hybrid_TP": int(hybrid_tp[budget - 1]),
        "Difference": int(
            hybrid_tp[budget - 1] - xgb_tp[budget - 1]
        )
    })

budget_comparison = pd.DataFrame(rows)

print("NEPRIKLAUSOMAS PRIORITETIZAVIMO PALYGINIMAS\n")
print(budget_comparison.to_string(index=False))

print("\nBendras palyginimas per visus 200 biudžetų:")
print("XGBoost vidurkis:", round(xgb_tp.mean(), 4))
print("Hybrid vidurkis:", round(hybrid_tp.mean(), 4))
print(
    "Vidutinis skirtumas:",
    round((hybrid_tp - xgb_tp).mean(), 4)
)

print("\nBiudžetų skaičius:")
print("Hybrid geresnis:", int((hybrid_tp > xgb_tp).sum()))
print("XGBoost geresnis:", int((hybrid_tp < xgb_tp).sum()))
print("Vienodi:", int((hybrid_tp == xgb_tp).sum()))

budget_comparison.to_csv(
    "aml_models/gpt_v4_independent_budget_comparison.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import numpy as np
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert (results["true_label"] == 1).sum() == 100
assert (results["true_label"] == 0).sum() == 100

def ranking_difference(sample):
    xgb = sample.sort_values(
        ["xgb_probability", "original_id"],
        ascending=[False, True],
        kind="mergesort"
    )

    hybrid = sample.sort_values(
        ["gpt_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True],
        kind="mergesort"
    )

    xgb_cumulative = np.cumsum(
        xgb["true_label"].to_numpy()
    )

    hybrid_cumulative = np.cumsum(
        hybrid["true_label"].to_numpy()
    )

    return float(
        (hybrid_cumulative - xgb_cumulative).mean()
    )

observed = ranking_difference(results)

aml = results[results["true_label"] == 1]
legit = results[results["true_label"] == 0]

rng = np.random.default_rng(42)
bootstrap_differences = []

for _ in range(2000):
    aml_indices = rng.integers(0, len(aml), len(aml))
    legit_indices = rng.integers(0, len(legit), len(legit))

    sample = pd.concat([
        aml.iloc[aml_indices],
        legit.iloc[legit_indices]
    ], ignore_index=True)

    bootstrap_differences.append(
        ranking_difference(sample)
    )

bootstrap_differences = np.array(bootstrap_differences)

ci_low, ci_high = np.percentile(
    bootstrap_differences, [2.5, 97.5]
)

print("NEPRIKLAUSOMO TESTO BOOTSTRAP ANALIZĖ\n")
print(f"Stebėtas vidutinis skirtumas: {observed:.4f}")
print(f"95 % pasikliautinasis intervalas: [{ci_low:.4f}; {ci_high:.4f}]")
print(
    "Bootstrap imčių, kuriose hibridinis metodas geresnis:",
    f"{(bootstrap_differences > 0).mean() * 100:.2f}%"
)

pd.DataFrame({
    "bootstrap_difference": bootstrap_differences
}).to_csv(
    "aml_models/gpt_v4_independent_bootstrap.csv",
    index=False
)

print("\nRezultatai išsaugoti!")

In [ ]:
import pandas as pd

# Tikrasis AML paplitimas aukštos rizikos testavimo grupėje
total_high_risk = 34395
aml_high_risk = 1148
legit_high_risk = total_high_risk - aml_high_risk

prevalence = aml_high_risk / total_high_risk

models = {
    "XGBoost": {"recall": 0.23, "fpr": 0.12},
    "GPT V4": {"recall": 0.93, "fpr": 0.85},
    "OR": {"recall": 0.94, "fpr": 0.87},
    "AND": {"recall": 0.22, "fpr": 0.10}
}

rows = []

for name, metrics in models.items():
    recall = metrics["recall"]
    fpr = metrics["fpr"]

    estimated_tp = recall * aml_high_risk
    estimated_fp = fpr * legit_high_risk
    estimated_alerts = estimated_tp + estimated_fp

    estimated_precision = (
        estimated_tp / estimated_alerts
        if estimated_alerts > 0 else 0
    )

    rows.append({
        "Model": name,
        "Estimated_TP": round(estimated_tp),
        "Estimated_FP": round(estimated_fp),
        "Estimated_Alerts": round(estimated_alerts),
        "Estimated_Precision": estimated_precision,
        "Alert_Rate": estimated_alerts / total_high_risk
    })

adjusted_results = pd.DataFrame(rows)

print("AML paplitimas aukštos rizikos grupėje:")
print(f"{prevalence:.4%}")

print("\nPAPILDOMAI ĮVERTINTI PRAKTINIAI RODIKLIAI\n")
print(adjusted_results.round(4).to_string(index=False))

adjusted_results.to_csv(
    "aml_models/gpt_v4_prevalence_adjusted_estimates.csv",
    index=False
)

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Visa nepriklausomo testavimo imtis
# test_data ir preprocessor / xgb_model turi būti jau atmintyje

threshold = 0.957565

high_risk = test_data[
    test_data["xgb_probability"] >= 0.90
].copy()

y_true = high_risk["Is Laundering"].astype(int)

y_pred = (
    high_risk["xgb_probability"] >= threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("FAKTINIAI XGBOOST REZULTATAI AUKŠTOS RIZIKOS GRUPĖJE\n")
print("Operacijų:", len(high_risk))
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print("\nPrecision:", round(tp / (tp + fp), 4))
print("Recall:", round(tp / (tp + fn), 4))
print("FPR:", round(fp / (fp + tn), 4))
print("Perspėjimų:", tp + fp)

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

results = results[
    (results["status"] == "OK") &
    (results["gpt_prediction"].isin([0, 1]))
].drop_duplicates("original_id").copy()

assert len(results) == 200
assert (results["true_label"] == 1).sum() == 100
assert (results["true_label"] == 0).sum() == 100

aml = results[results["true_label"] == 1]
legit = results[results["true_label"] == 0]

tp = int((aml["gpt_prediction"] == 1).sum())
fp = int((legit["gpt_prediction"] == 1).sum())

recall = tp / len(aml)
fpr = fp / len(legit)

recall_ci = proportion_confint(
    tp, len(aml), alpha=0.05, method="wilson"
)
fpr_ci = proportion_confint(
    fp, len(legit), alpha=0.05, method="wilson"
)

prevalence = 1148 / 34395

def adjusted_precision(tpr, false_positive_rate):
    numerator = prevalence * tpr
    denominator = (
        numerator + (1 - prevalence) * false_positive_rate
    )
    return numerator / denominator if denominator else np.nan

rng = np.random.default_rng(42)
bootstrap_precision = []

for _ in range(10000):
    aml_sample = rng.choice(
        aml["gpt_prediction"].to_numpy(), size=100, replace=True
    )
    legit_sample = rng.choice(
        legit["gpt_prediction"].to_numpy(), size=100, replace=True
    )

    bootstrap_precision.append(
        adjusted_precision(
            aml_sample.mean(),
            legit_sample.mean()
        )
    )

precision_ci = np.percentile(
    bootstrap_precision, [2.5, 97.5]
)

print("GPT V4 NEAPIBRĖŽTUMO ANALIZĖ\n")

print(
    f"Recall: {recall:.2%} "
    f"(95 % PI: {recall_ci[0]:.2%}–{recall_ci[1]:.2%})"
)

print(
    f"FPR: {fpr:.2%} "
    f"(95 % PI: {fpr_ci[0]:.2%}–{fpr_ci[1]:.2%})"
)

print(
    f"Koreguotas Precision: "
    f"{adjusted_precision(recall, fpr):.2%} "
    f"(95 % bootstrap PI: "
    f"{precision_ci[0]:.2%}–{precision_ci[1]:.2%})"
)

print("\nFaktinis XGBoost Precision visoje grupėje: 11.39 %")

In [ ]:
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

context = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
)

data = results.merge(
    context[
        ["original_id", "same_recipient_7d_count"]
    ],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

data = data[
    (data["status"] == "OK") &
    (data["true_label"] == 1)
].copy()

data["recipient_group"] = data[
    "same_recipient_7d_count"
].apply(
    lambda x: "Naujas gavėjas" if x == 0
    else "Anksčiau naudotas gavėjas"
)

data["xgb_detected"] = (
    data["xgb_probability"] >= 0.957565
).astype(int)

summary = data.groupby(
    "recipient_group"
).agg(
    AML_operacijos=("true_label", "size"),
    GPT_aptikta=("gpt_prediction", "sum"),
    XGBoost_aptikta=("xgb_detected", "sum")
)

summary["GPT_Recall"] = (
    summary["GPT_aptikta"] / summary["AML_operacijos"]
)

summary["XGBoost_Recall"] = (
    summary["XGBoost_aptikta"] / summary["AML_operacijos"]
)

print("AML APTIKIMAS PAGAL GAVĖJO ISTORIJĄ\n")
print(summary.round(4).to_string())

In [ ]:
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

context = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
)

data = results[
    results["status"] == "OK"
].merge(
    context[["original_id", "same_recipient_7d_count"]],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

assert len(data) == 200

data["recipient_group"] = data[
    "same_recipient_7d_count"
].apply(
    lambda x: "Naujas gavėjas" if x == 0
    else "Anksčiau naudotas gavėjas"
)

data["transaction_type"] = data["true_label"].map({
    0: "Teisėta",
    1: "AML"
})

data["xgb_prediction"] = (
    data["xgb_probability"] >= 0.957565
).astype(int)

summary = data.groupby(
    ["transaction_type", "recipient_group"]
).agg(
    Operaciju_skaicius=("true_label", "size"),
    GPT_pazymeta=("gpt_prediction", "sum"),
    XGBoost_pazymeta=("xgb_prediction", "sum")
)

summary["GPT_pazymeta_pct"] = (
    100 * summary["GPT_pazymeta"] /
    summary["Operaciju_skaicius"]
)

summary["XGBoost_pazymeta_pct"] = (
    100 * summary["XGBoost_pazymeta"] /
    summary["Operaciju_skaicius"]
)

print("MODELIŲ SPRENDIMAI PAGAL GAVĖJO ISTORIJĄ\n")
print(summary.round(2).to_string())

summary.to_csv(
    "aml_models/gpt_v4_recipient_group_analysis.csv"
)

In [ ]:
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_results.csv"
)

context = pd.read_csv(
    "aml_models/gpt_v4_independent_test_200_corrected_context.csv"
)

data = results[results["status"] == "OK"].merge(
    context[["original_id", "same_recipient_7d_count"]],
    on="original_id",
    validate="one_to_one"
)

assert len(data) == 200

data["recipient_group"] = data["same_recipient_7d_count"].apply(
    lambda x: "Naujas gavėjas" if x == 0
    else "Anksčiau naudotas gavėjas"
)

data["xgb_prediction"] = (
    data["xgb_probability"] >= 0.957565
).astype(int)

rows = []

for group, subset in data.groupby("recipient_group"):
    for model, prediction_column in [
        ("GPT V4", "gpt_prediction"),
        ("XGBoost", "xgb_prediction")
    ]:
        predictions = subset[prediction_column].astype(int)
        labels = subset["true_label"].astype(int)

        tp = int(((predictions == 1) & (labels == 1)).sum())
        fp = int(((predictions == 1) & (labels == 0)).sum())

        rows.append({
            "Gavėjo grupė": group,
            "Modelis": model,
            "TP": tp,
            "FP": fp,
            "Precision": tp / (tp + fp) if tp + fp else 0,
            "Perspėjimai": tp + fp
        })

precision_by_recipient = pd.DataFrame(rows)

print("PRECISION PAGAL GAVĖJO ISTORIJĄ\n")
print(precision_by_recipient.round(4).to_string(index=False))

precision_by_recipient.to_csv(
    "aml_models/gpt_v4_precision_by_recipient.csv",
    index=False
)

In [ ]:
import os
import pandas as pd

files = [
    "aml_models/test_3000_natural.csv",
    "aml_models/test_3000_context.csv",
    "aml_models/comparison_3000_context.csv",
    "aml_models/gpt_context_prompt_v4.txt"
]

for path in files:
    print(f"\nFailas: {path}")
    print("Egzistuoja:", os.path.exists(path))

    if os.path.exists(path) and path.endswith(".csv"):
        data = pd.read_csv(path)
        print("Eilučių skaičius:", len(data))
        print("Stulpeliai:", data.columns.tolist())

        if "Is Laundering" in data.columns:
            print(
                "AML operacijų:",
                int(data["Is Laundering"].sum())
            )

In [ ]:
import pandas as pd

for name in ["df", "test_data", "counts"]:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}: {obj.shape}")
        print("Stulpeliai:", obj.columns.tolist())
        print("Indekso unikalumas:", obj.index.is_unique)
    else:
        print(f"\n{name}: nėra Jupyter atmintyje")

comparison = pd.read_csv(
    "aml_models/comparison_3000_context.csv"
)

print("\n3 000 OPERACIJŲ IDENTIFIKATORIAI")
print(
    comparison[
        ["test_row_id", "original_id", "old_test_row_id"]
    ].head(5).to_string(index=False)
)

print("\nUnikalūs original_id:", comparison["original_id"].nunique())
print("Trūkstami original_id:", comparison["original_id"].isna().sum())
print("Unikalūs test_row_id:", comparison["test_row_id"].nunique())

In [ ]:
import pandas as pd
import numpy as np

comparison = pd.read_csv(
    "aml_models/comparison_3000_context.csv"
)

history = df[
    ["Timestamp", "From Bank", "Account", "To Bank", "Account.1"]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])
comparison["Timestamp"] = pd.to_datetime(comparison["Timestamp"])

# Kiekviena siuntėjo ir gavėjo pora turi atskirą istoriją
pair_columns = [
    "From Bank", "Account", "To Bank", "Account.1"
]

history = history.sort_values(
    pair_columns + ["Timestamp"],
    kind="mergesort"
)

# Išsaugome pradinį operacijos indeksą
history["original_id"] = history.index

# Skaičiuojame ankstesnes operacijas per 7 dienas.
# Vienodo laiko operacijos neįtraukiamos.
history["same_recipient_7d_count"] = (
    history.groupby(pair_columns, sort=False, dropna=False)
    .rolling(
        "7D",
        on="Timestamp",
        closed="left"
    )["original_id"]
    .count()
    .reset_index(level=list(range(len(pair_columns))), drop=True)
    .to_numpy()
)

recipient_counts = history[
    ["original_id", "same_recipient_7d_count"]
]

comparison = comparison.merge(
    recipient_counts,
    on="original_id",
    how="left",
    validate="one_to_one"
)

print("Operacijų skaičius:", len(comparison))
print(
    "Trūkstamų reikšmių:",
    comparison["same_recipient_7d_count"].isna().sum()
)
print(
    "\nGavėjo istorijos statistika:\n",
    comparison["same_recipient_7d_count"].describe()
)

comparison.to_csv(
    "aml_models/comparison_3000_context_v4.csv",
    index=False
)

print("\nFailas išsaugotas!")

In [ ]:
import pandas as pd
import numpy as np

comparison = pd.read_csv(
    "aml_models/comparison_3000_context.csv"
)

comparison["Timestamp"] = pd.to_datetime(
    comparison["Timestamp"]
)

pair_cols = [
    "From Bank", "Account",
    "To Bank", "Account.1"
]

history = df[pair_cols + ["Timestamp"]].copy()
history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Tikrasis pradinės operacijos identifikatorius
history["original_id"] = history.index

# Skaičiuojame pagal unikalias siuntėjo, gavėjo ir laiko grupes.
# Taip vienodo laiko operacijos nepatenka į viena kitos istoriją.
time_groups = (
    history.groupby(
        pair_cols + ["Timestamp"],
        dropna=False,
        sort=False
    )
    .size()
    .rename("transactions_at_time")
    .reset_index()
)

time_groups = time_groups.sort_values(
    pair_cols + ["Timestamp"],
    kind="mergesort"
)

def calculate_previous_7d(group):
    timestamps = group["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    ).astype("int64")

    counts = group["transactions_at_time"].to_numpy()

    cumulative = np.concatenate([
        [0], np.cumsum(counts)
    ])

    window_start = np.searchsorted(
        timestamps,
        timestamps - np.timedelta64(7, "D").astype(
            "timedelta64[ns]"
        ).astype("int64"),
        side="left"
    )

    positions = np.arange(len(group))

    group = group.copy()
    group["same_recipient_7d_count"] = (
        cumulative[positions] - cumulative[window_start]
    )

    return group

time_groups = (
    time_groups.groupby(
        pair_cols,
        dropna=False,
        group_keys=False,
        sort=False
    )
    .apply(calculate_previous_7d)
    .reset_index(drop=True)
)

comparison = comparison.merge(
    time_groups[
        pair_cols + [
            "Timestamp",
            "same_recipient_7d_count"
        ]
    ],
    on=pair_cols + ["Timestamp"],
    how="left",
    validate="many_to_one"
)

assert len(comparison) == 3000
assert comparison["same_recipient_7d_count"].notna().all()
assert (comparison["same_recipient_7d_count"] >= 0).all()

comparison.to_csv(
    "aml_models/comparison_3000_context_v4.csv",
    index=False
)

print("Operacijų skaičius:", len(comparison))
print(
    "Trūkstamų reikšmių:",
    comparison["same_recipient_7d_count"].isna().sum()
)
print(
    "Operacijų be ankstesnių pervedimų:",
    int((comparison["same_recipient_7d_count"] == 0).sum())
)
print("\nStatistika:")
print(comparison["same_recipient_7d_count"].describe())
print("\nPataisytas failas išsaugotas.")

In [ ]:
import pandas as pd
import numpy as np

comparison = pd.read_csv(
    "aml_models/comparison_3000_context.csv"
)

comparison["Timestamp"] = pd.to_datetime(comparison["Timestamp"])

pair_cols = ["From Bank", "Account", "To Bank", "Account.1"]

# Imame tik istorijai reikalingus stulpelius
history = df[pair_cols + ["Timestamp"]].copy()
history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Unikalios siuntėjo ir gavėjo poros mūsų imtyje
selected_pairs = comparison[pair_cols].drop_duplicates()

# Paliekame tik aktualių porų istoriją
history = history.merge(
    selected_pairs,
    on=pair_cols,
    how="inner"
)

# Grupuojame istoriją pagal siuntėjo ir gavėjo porą
history_groups = {
    key: np.sort(group["Timestamp"].to_numpy(dtype="datetime64[ns]"))
    for key, group in history.groupby(pair_cols, sort=False)
}

seven_days = np.timedelta64(7, "D")
recipient_counts = []

for row in comparison.itertuples(index=False, name=None):
    record = dict(zip(comparison.columns, row))

    key = tuple(record[col] for col in pair_cols)
    timestamp = np.datetime64(record["Timestamp"], "ns")

    times = history_groups.get(key)

    if times is None:
        recipient_counts.append(0)
        continue

    start = np.searchsorted(
        times, timestamp - seven_days, side="left"
    )
    end = np.searchsorted(
        times, timestamp, side="left"
    )

    recipient_counts.append(int(end - start))

comparison["same_recipient_7d_count"] = recipient_counts

assert len(comparison) == 3000
assert comparison["same_recipient_7d_count"].notna().all()
assert (comparison["same_recipient_7d_count"] >= 0).all()

comparison.to_csv(
    "aml_models/comparison_3000_context_v4.csv",
    index=False
)

print("Operacijų skaičius:", len(comparison))
print(
    "Trūkstamų reikšmių:",
    comparison["same_recipient_7d_count"].isna().sum()
)
print(
    "Operacijų be ankstesnių pervedimų:",
    int((comparison["same_recipient_7d_count"] == 0).sum())
)
print("\nStatistika:")
print(comparison["same_recipient_7d_count"].describe())
print("\nFailas išsaugotas!")

In [ ]:
import pandas as pd
import numpy as np

comparison = pd.read_csv(
    "aml_models/comparison_3000_context_v4.csv"
)
comparison["Timestamp"] = pd.to_datetime(comparison["Timestamp"])

# Istorija tik reikalingiems siuntėjams
sender_cols = ["From Bank", "Account"]

selected_senders = comparison[sender_cols].drop_duplicates()

history = df[
    sender_cols + ["Timestamp", "Payment Currency", "Amount Paid"]
].copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

history = history.merge(
    selected_senders,
    on=sender_cols,
    how="inner"
)

# Atskira istorija kiekvienai siuntėjo sąskaitai ir valiutai
history_groups = {}

for key, group in history.groupby(
    sender_cols + ["Payment Currency"],
    sort=False,
    dropna=False
):
    group = group.sort_values("Timestamp")

    times = group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    amounts = group["Amount Paid"].to_numpy(dtype=float)

    cumulative = np.concatenate([[0.0], np.cumsum(amounts)])

    history_groups[key] = (times, cumulative)

one_day = np.timedelta64(24, "h")
corrected_amounts = []

for row in comparison.itertuples(index=False):
    key = (
        row._asdict()["From_Bank"],
        row.Account,
        row._asdict()["Payment_Currency"]
    ) if False else None

    # Naudojame stulpelių pavadinimus tiesiogiai
    break

for _, row in comparison.iterrows():
    key = (
        row["From Bank"],
        row["Account"],
        row["Payment Currency"]
    )

    timestamp = np.datetime64(row["Timestamp"], "ns")
    group = history_groups.get(key)

    if group is None:
        corrected_amounts.append(0.0)
        continue

    times, cumulative = group

    start = np.searchsorted(
        times, timestamp - one_day, side="left"
    )
    end = np.searchsorted(
        times, timestamp, side="left"
    )

    corrected_amounts.append(
        float(cumulative[end] - cumulative[start])
    )

comparison["previous_24h_amount_original"] = (
    comparison["previous_24h_amount"]
)

comparison["previous_24h_amount"] = corrected_amounts

changed = ~np.isclose(
    comparison["previous_24h_amount"],
    comparison["previous_24h_amount_original"],
    rtol=1e-8,
    atol=0.01
)

assert len(comparison) == 3000
assert comparison["previous_24h_amount"].notna().all()

comparison.to_csv(
    "aml_models/comparison_3000_context_v4_corrected.csv",
    index=False
)

print("Operacijų skaičius:", len(comparison))
print("Patikslintų sumų skaičius:", int(changed.sum()))
print("AML tarp patikslintų:", int(
    comparison.loc[changed, "Is Laundering"].sum()
))
print("Trūkstamų sumų:", comparison["previous_24h_amount"].isna().sum())
print("\nFailas išsaugotas!")

In [ ]:
import pandas as pd
import hashlib
import os

path = "aml_models/comparison_3000_context_v4_corrected.csv"
data = pd.read_csv(path)

features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

prompt_path = "aml_models/gpt_context_prompt_v4.txt"

with open(prompt_path, "rb") as f:
    prompt_hash = hashlib.sha256(f.read()).hexdigest()

checks = {
    "Tiksliai 3000 operacijų": len(data) == 3000,
    "7 AML operacijos": data["Is Laundering"].sum() == 7,
    "Unikalūs operacijų ID": data["original_id"].is_unique,
    "Visi 15 požymių yra": all(c in data.columns for c in features),
    "Nėra trūkstamų požymių": data[features].notna().all().all(),
    "Istoriniai skaičiai neneigiami": (
        data[
            [
                "previous_24h_count",
                "previous_7d_count",
                "previous_24h_amount",
                "previous_7d_unique_recipients",
                "same_recipient_7d_count"
            ]
        ] >= 0
    ).all().all(),
    "XGBoost prognozės yra": data["xgb_probability"].notna().all(),
    "GPT V4 instrukcija nepakeista": (
        prompt_hash ==
        "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
    )
}

print("GALUTINIS DUOMENŲ PATIKRINIMAS\n")

for name, passed in checks.items():
    print(f"{'OK' if passed else 'KLAIDA'} | {name}")

print("\nGPT gaus tik šiuos stulpelius:")
print(features)

print("\nVisi patikrinimai sėkmingi:", all(checks.values()))

print("\nAr yra ankstesnių GPT rezultatų failas:",
      os.path.exists("aml_models/gpt_v4_natural_3000_results.csv"))

In [ ]:
import os
import json
import time
import pandas as pd
from getpass import getpass
from openai import OpenAI

# API prisijungimas
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Įvesk OpenAI API raktą: "
    ).strip()

client = OpenAI(max_retries=0)

input_path = "aml_models/comparison_3000_context_v4_corrected.csv"
output_path = "aml_models/gpt_v4_natural_3000_results.csv"
prompt_path = "aml_models/gpt_context_prompt_v4.txt"

data = pd.read_csv(input_path)

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt = f.read()

features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

assert len(data) == 3000
assert data["original_id"].is_unique
assert data[features].notna().all().all()
assert int(data["Is Laundering"].sum()) == 7

columns = [
    "original_id",
    "true_label",
    "xgb_probability",
    "gpt_prediction",
    "gpt_reason",
    "raw_response",
    "status",
    "input_tokens",
    "output_tokens"
]

if os.path.exists(output_path):
    results = pd.read_csv(output_path)
    results = results[columns].copy()
    results = results.drop_duplicates(
        "original_id", keep="last"
    )
else:
    results = pd.DataFrame(columns=columns)

completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

print("Iš viso operacijų:", len(data))
print("Jau sėkmingai apdorota:", len(completed))
print("Liko:", len(data) - len(completed))
print()

for i, (_, row) in enumerate(data.iterrows(), start=1):

    original_id = int(row["original_id"])

    if original_id in completed:
        continue

    transaction = {}

    for col in features:
        value = row[col]

        if hasattr(value, "item"):
            value = value.item()

        transaction[col] = value

    record = {
        "original_id": original_id,
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            max_output_tokens=150
        )

        raw = response.output_text.strip()

        record["raw_response"] = raw

        if response.usage:
            record["input_tokens"] = (
                response.usage.input_tokens
            )
            record["output_tokens"] = (
                response.usage.output_tokens
            )

        parsed = json.loads(raw)
        prediction = parsed.get("prediction")

        if type(prediction) is int and prediction in (0, 1):
            record["gpt_prediction"] = prediction
            record["gpt_reason"] = str(
                parsed.get("reason", "")
            )
            record["status"] = "OK"
        else:
            record["status"] = "INVALID_PREDICTION"

    except Exception as e:
        record["status"] = "ERROR"
        record["raw_response"] = str(e)[:500]

    # Atnaujiname tik šios operacijos rezultatą
    results = results[
        results["original_id"] != original_id
    ]

    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )

    # Išsaugome po kiekvienos užklausos
    results.to_csv(output_path, index=False)

    if record["status"] == "OK":
        completed.add(original_id)

    if i % 50 == 0 or record["status"] != "OK":
        print(
            f"Operacija {i}/3000 | "
            f"Būsena: {record['status']} | "
            f"Sėkmingai: {len(completed)}/3000",
            flush=True
        )

    if record["status"] != "OK":
        print(
            "\nVykdymas sustabdytas. "
            "Patikrink paskutinį rezultatą."
        )
        print("Būsena:", record["status"])
        print("Atsakymas:", record["raw_response"][:500])
        break

print("\nGALUTINĖ SUVESTINĖ")
print(results["status"].value_counts().to_string())

input_tokens = int(
    pd.to_numeric(
        results["input_tokens"],
        errors="coerce"
    ).sum()
)

output_tokens = int(
    pd.to_numeric(
        results["output_tokens"],
        errors="coerce"
    ).sum()
)

estimated_cost = (
    input_tokens * 0.75 / 1_000_000
    + output_tokens * 4.50 / 1_000_000
)

print("\nĮvesties žetonai:", input_tokens)
print("Išvesties žetonai:", output_tokens)

print(
    "Preliminari kaina (USD, be mokesčių):",
    round(estimated_cost, 4)
)

print("\nRezultatai išsaugoti:", output_path)

In [ ]:
from openai import OpenAI
import pandas as pd
import os

# Padidiname vienos API užklausos laukimo laiką
client = OpenAI(
    timeout=180.0,
    max_retries=0
)

# Patikriname išsaugotą progresą
output_path = "aml_models/gpt_v4_natural_3000_results.csv"

results = pd.read_csv(output_path)

completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

print("Sėkmingai atlikta:", len(completed))
print("Liko atlikti:", 3000 - len(completed))
print("Klaidų įrašų:", (results["status"] != "OK").sum())
print("\nAPI laukimo laikas padidintas iki 180 sekundžių.")
print("Galima tęsti eksperimentą.")

In [ ]:
import os
import json
import time
import pandas as pd

input_path = "aml_models/comparison_3000_context_v4_corrected.csv"
output_path = "aml_models/gpt_v4_natural_3000_results.csv"
prompt_path = "aml_models/gpt_context_prompt_v4.txt"

data = pd.read_csv(input_path)
results = pd.read_csv(output_path)

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt = f.read()

features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

remaining = data[
    ~data["original_id"].isin(completed)
].copy()

print("Jau atlikta:", len(completed))
print("Liko:", len(remaining))
print("Pradedamas tęsinys...\n", flush=True)

for i, (_, row) in enumerate(remaining.iterrows(), start=1):

    original_id = int(row["original_id"])

    transaction = {}

    for col in features:
        value = row[col]

        if hasattr(value, "item"):
            value = value.item()

        transaction[col] = value

    record = {
        "original_id": original_id,
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            max_output_tokens=150
        )

        raw = response.output_text.strip()
        record["raw_response"] = raw

        if response.usage:
            record["input_tokens"] = response.usage.input_tokens
            record["output_tokens"] = response.usage.output_tokens

        parsed = json.loads(raw)
        prediction = parsed.get("prediction")

        if type(prediction) is int and prediction in (0, 1):
            record["gpt_prediction"] = prediction
            record["gpt_reason"] = str(parsed.get("reason", ""))
            record["status"] = "OK"
        else:
            record["status"] = "INVALID_PREDICTION"

    except Exception as e:
        record["raw_response"] = str(e)[:500]

    # Pašaliname ankstesnį šios operacijos ERROR įrašą
    results = results[
        results["original_id"] != original_id
    ]

    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )

    # Išsaugome po kiekvienos operacijos
    results.to_csv(output_path, index=False)

    if record["status"] == "OK":
        completed.add(original_id)

    if i % 25 == 0 or record["status"] != "OK":
        print(
            f"Tęsinys {i}/{len(remaining)} | "
            f"Iš viso atlikta: {len(completed)}/3000 | "
            f"Būsena: {record['status']}",
            flush=True
        )

    if record["status"] != "OK":
        print("\nEksperimentas sustabdytas.")
        print("Klaida:", record["raw_response"][:500])
        break

print("\nGALUTINĖ SUVESTINĖ")
print(results["status"].value_counts().to_string())

input_tokens = pd.to_numeric(
    results["input_tokens"], errors="coerce"
).sum()

output_tokens = pd.to_numeric(
    results["output_tokens"], errors="coerce"
).sum()

estimated_cost = (
    input_tokens * 0.75 / 1_000_000
    + output_tokens * 4.50 / 1_000_000
)

print("\nĮvesties žetonai:", int(input_tokens))
print("Išvesties žetonai:", int(output_tokens))
print("Preliminari kaina (USD):", round(estimated_cost, 4))
print("Sėkmingai atlikta:", len(completed), "/ 3000")
print("Liko:", 3000 - len(completed))
print("\nRezultatų failas:", output_path)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

results = pd.read_csv(
    "aml_models/gpt_v4_natural_3000_results.csv"
)

assert len(results) == 3000
assert results["status"].eq("OK").all()
assert results["original_id"].is_unique
assert results["gpt_prediction"].notna().all()

y_true = results["true_label"].astype(int)

gpt_pred = results["gpt_prediction"].astype(int)

xgb_pred = (
    results["xgb_probability"].astype(float) >= 0.957565
).astype(int)

print("NATŪRALIOS 3000 OPERACIJŲ IMTIES REZULTATAI")
print("=" * 55)

print("Visos operacijos:", len(results))
print("AML operacijos:", int(y_true.sum()))
print("Teisėtos operacijos:", int((y_true == 0).sum()))
print()

for name, predictions in [
    ("XGBoost", xgb_pred),
    ("GPT V4", gpt_pred)
]:
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true, predictions, zero_division=0
    )

    recall = recall_score(
        y_true, predictions, zero_division=0
    )

    f1 = f1_score(
        y_true, predictions, zero_division=0
    )

    accuracy = accuracy_score(y_true, predictions)

    fpr = fp / (fp + tn) if (fp + tn) else 0

    print(f"--- {name} ---")
    print(f"TP (aptiktos AML): {tp}")
    print(f"FP (klaidingi perspėjimai): {fp}")
    print(f"FN (praleistos AML): {fn}")
    print(f"TN (teisingai atpažintos teisėtos): {tn}")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall: {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"FPR: {fpr:.4f} ({fpr*100:.2f}%)")
    print(f"Iš viso perspėjimų: {int(predictions.sum())}")
    print()


In [ ]:
import pandas as pd

results = pd.read_csv(
    "aml_models/gpt_v4_natural_3000_results.csv"
)

results["xgb_prediction"] = (
    results["xgb_probability"] >= 0.957565
).astype(int)

results["gpt_prediction"] = (
    results["gpt_prediction"].astype(int)
)

aml = results[results["true_label"] == 1].copy()

print("VISŲ 7 AML OPERACIJŲ PALYGINIMAS")
print("=" * 65)

print(
    aml[
        [
            "original_id",
            "xgb_probability",
            "xgb_prediction",
            "gpt_prediction"
        ]
    ].to_string(index=False)
)

both = (
    (aml["xgb_prediction"] == 1) &
    (aml["gpt_prediction"] == 1)
).sum()

only_xgb = (
    (aml["xgb_prediction"] == 1) &
    (aml["gpt_prediction"] == 0)
).sum()

only_gpt = (
    (aml["xgb_prediction"] == 0) &
    (aml["gpt_prediction"] == 1)
).sum()

neither = (
    (aml["xgb_prediction"] == 0) &
    (aml["gpt_prediction"] == 0)
).sum()

print("\nMODELIO SPRENDIMŲ SUTAPIMAS")
print("Abu aptiko:", both)
print("Tik XGBoost aptiko:", only_xgb)
print("Tik GPT aptiko:", only_gpt)
print("Nė vienas neaptiko:", neither)
print("Bendrai aptikta:", both + only_xgb + only_gpt, "iš 7")


In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

results = pd.read_csv(
    "aml_models/gpt_v4_natural_3000_results.csv"
)

y = results["true_label"].astype(int)

xgb = (
    results["xgb_probability"] >= 0.957565
).astype(int)

gpt = results["gpt_prediction"].astype(int)

or_pred = ((xgb == 1) | (gpt == 1)).astype(int)
and_pred = ((xgb == 1) & (gpt == 1)).astype(int)

print("HIBRIDINIŲ MODELIŲ PALYGINIMAS")
print("=" * 55)

for name, pred in [
    ("XGBoost", xgb),
    ("GPT V4", gpt),
    ("OR (bent vienas modelis)", or_pred),
    ("AND (abu modeliai)", and_pred)
]:
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    print(f"\n{name}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
    print(f"Precision: {precision_score(y, pred, zero_division=0):.4f}")
    print(f"Recall: {recall_score(y, pred, zero_division=0):.4f}")
    print(f"F1: {f1_score(y, pred, zero_division=0):.4f}")
    print(f"Perspėjimų skaičius: {int(pred.sum())}")

In [ ]:
import os
import pandas as pd

print("DUOMENŲ PATIKRINIMAS")
print("=" * 55)

files = [
    "HI-Small_Trans.csv",
    "aml_models/test_3000_natural.csv",
    "aml_models/comparison_3000_context_v4_corrected.csv",
    "aml_models/gpt_v4_natural_3000_results.csv",
    "aml_models/xgb_model.joblib",
    "aml_models/preprocessor.joblib",
    "aml_models/gpt_context_prompt_v4.txt"
]

for path in files:
    print(
        f"{'RASTA' if os.path.exists(path) else 'NERASTA'} | {path}"
    )

print("\nANKSTESNIO EKSPERIMENTO PATIKRA")

path = "aml_models/gpt_v4_natural_3000_results.csv"

if os.path.exists(path):
    previous = pd.read_csv(path)

    print("Išsaugotų operacijų:", len(previous))
    print("Unikalių ID:", previous["original_id"].nunique())
    print("Sėkmingų GPT atsakymų:",
          (previous["status"] == "OK").sum())

print("\nNOTEBOOK ATMINTYJE ESANTYS OBJEKTAI")

for name in ["df", "train", "validation", "test", "model", "preprocessor"]:
    if name in globals():
        obj = globals()[name]
        print(name, "| RASTAS", "| shape:", getattr(obj, "shape", "—"))
    else:
        print(name, "| NĖRA")

In [ ]:
import pandas as pd

print("NAUJOS IMTIES ATRANKOS PATIKRA")
print("=" * 55)

# Ankstesniame tyrime naudota testavimo laikotarpio pradžia
test_start = pd.Timestamp("2022-09-09")

# Originalaus duomenų rinkinio indeksas naudojamas
# kaip operacijos identifikatorius
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

test_mask = df["Timestamp"] >= test_start

# Ankstesnės natūralios imties operacijų ID
previous = pd.read_csv(
    "aml_models/gpt_v4_natural_3000_results.csv"
)

used_ids = set(previous["original_id"].astype(int))

# Patikriname, ar original_id atitinka df indeksą
print("df indeksas unikalus:", df.index.is_unique)
print("df indekso pradžia:", df.index[:5].tolist())
print("Ankstesnių ID rasta df indekse:",
      len(used_ids.intersection(set(df.index))))

# Atrenkame tik dar nenaudotas testavimo operacijas
available = df.loc[
    test_mask & ~df.index.isin(used_ids)
]

aml_available = available[
    available["Is Laundering"] == 1
]

legit_available = available[
    available["Is Laundering"] == 0
]

print("\nTESTAVIMO LAIKOTARPIO LIKUTIS")
print("Iš viso:", len(available))
print("AML:", len(aml_available))
print("Teisėtos:", len(legit_available))

print("\nAR GALIMA SUDARYTI 100 AML + 900 TEISĖTŲ IMTĮ?")
print(
    len(aml_available) >= 100
    and len(legit_available) >= 900
)

print("\nPastaba: dar reikės patikrinti ankstesnių")
print("200 operacijų eksperimentų ID, jei jie išsaugoti.")

In [ ]:
import os
import pandas as pd

print("ANKSTESNIŲ EKSPERIMENTŲ FAILŲ PAIEŠKA")
print("=" * 65)

for root, dirs, files in os.walk("aml_models"):
    for filename in sorted(files):
        if filename.lower().endswith(".csv"):
            path = os.path.join(root, filename)

            try:
                columns = pd.read_csv(path, nrows=0).columns.tolist()

                id_columns = [
                    c for c in columns
                    if c.lower() in [
                        "original_id",
                        "original_index",
                        "row_id",
                        "transaction_id"
                    ]
                ]

                print(f"\nFailas: {path}")
                print("ID stulpeliai:", id_columns or "Nerasta")

                if id_columns:
                    ids = pd.read_csv(
                        path,
                        usecols=id_columns
                    )

                    print("Eilučių:", len(ids))
                    print(
                        "Unikalių ID:",
                        ids[id_columns[0]].nunique()
                    )

            except Exception as e:
                print(f"Nepavyko perskaityti {path}: {e}")

In [ ]:
import os
import pandas as pd

# Ankstesnių GPT eksperimentų failai
previous_files = [
    "gpt_v4_natural_3000_results.csv",
    "gpt_v4_independent_test_200_results.csv",
    "gpt_v4_high_risk_200_results.csv",
    "gpt_v3_v4_comparison_results.csv",
    "gpt_v4_high_risk_pilot_5.csv",
    "gpt_context_validation.csv",
    "gpt_context_validation_v3.csv",
    "gpt_validation_v2_20.csv",
    "gpt_3000_results.csv",
    "gpt_3000_pilot_predictions.csv"
]

used_ids = set()
files_without_ids = []

for filename in previous_files:
    path = os.path.join("aml_models", filename)

    if not os.path.exists(path):
        continue

    temp = pd.read_csv(path)

    if "original_id" in temp.columns:
        ids = pd.to_numeric(
            temp["original_id"], errors="coerce"
        ).dropna().astype(int)

        used_ids.update(ids.tolist())
    else:
        files_without_ids.append(filename)

print("Žinomų anksčiau naudotų ID:", len(used_ids))
print("Failai be original_id:", files_without_ids)

# Tik nepriklausomo testavimo laikotarpis
test_data = df.loc[
    (df["Timestamp"] >= pd.Timestamp("2022-09-09"))
    & (~df.index.isin(used_ids))
].copy()

aml_pool = test_data[
    test_data["Is Laundering"] == 1
]

legit_pool = test_data[
    test_data["Is Laundering"] == 0
]

assert len(aml_pool) >= 100
assert len(legit_pool) >= 900

# Atsitiktinė, atkuriama atranka
aml_sample = aml_pool.sample(n=100, random_state=42)
legit_sample = legit_pool.sample(n=900, random_state=42)

new_sample = pd.concat(
    [aml_sample, legit_sample]
).sample(frac=1, random_state=42)

new_sample.insert(0, "original_id", new_sample.index)

# Išsaugome atrinktą imtį
path = "aml_models/test_1000_enriched.csv"
new_sample.to_csv(path, index=False)

print("\nNAUJA IMTIS")
print("Operacijų:", len(new_sample))
print("AML:", int(new_sample["Is Laundering"].sum()))
print("Teisėtų:", int((new_sample["Is Laundering"] == 0).sum()))
print("Pasikartojimų su žinomais ID:",
      new_sample["original_id"].isin(used_ids).sum())
print("Failas:", path)

In [ ]:
import pandas as pd
import os

files_to_check = [
    "aml_models/gpt_validation_v2_20.csv",
    "aml_models/gpt_3000_results.csv",
    "aml_models/gpt_3000_pilot_predictions.csv"
]

new_sample = pd.read_csv(
    "aml_models/test_1000_enriched.csv"
)

transaction_cols = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1",
    "Amount Received", "Receiving Currency",
    "Amount Paid", "Payment Currency",
    "Payment Format"
]

print("ANKSTESNIŲ FAILŲ PATIKRA")
print("=" * 60)

for path in files_to_check:
    if not os.path.exists(path):
        continue

    old = pd.read_csv(path)

    print("\nFailas:", path)
    print("Eilučių:", len(old))
    print("Stulpeliai:", old.columns.tolist())

    common = [
        col for col in transaction_cols
        if col in old.columns
    ]

    print("Bendri operacijų požymiai:", len(common))

    if len(common) == len(transaction_cols):
        old_keys = old[transaction_cols].drop_duplicates()

        matches = new_sample.merge(
            old_keys,
            on=transaction_cols,
            how="inner"
        )

        print(
            "Galimų pasikartojimų:",
            len(matches)
        )
    else:
        print(
            "Nepakanka požymių patikimam "
            "operacijų sutapimui nustatyti."
        )

In [ ]:
import pandas as pd

new_sample = pd.read_csv(
    "aml_models/test_1000_enriched.csv"
)

old_test = pd.read_csv(
    "aml_models/test_3000_natural.csv"
)

print("ANKSTESNIŲ TESTAVIMO ID PATIKRA")
print("=" * 60)

print("Seno testavimo failo stulpeliai:")
print(old_test.columns.tolist())

for filename in [
    "gpt_3000_results.csv",
    "gpt_3000_pilot_predictions.csv"
]:
    path = "aml_models/" + filename
    old_results = pd.read_csv(path)

    print("\nFailas:", filename)
    print("old_test_row_id reikšmės:")
    print(
        old_results["old_test_row_id"]
        .dropna()
        .unique()
        .tolist()
    )

print("\nNaujos imties ID skaičius:",
      new_sample["original_id"].nunique())

In [ ]:
import pandas as pd

new_sample = pd.read_csv(
    "aml_models/test_1000_enriched.csv"
)

old_test = pd.read_csv(
    "aml_models/test_3000_natural.csv"
)

old_results = pd.read_csv(
    "aml_models/gpt_3000_results.csv"
)

old_ids = old_results["old_test_row_id"].dropna().unique()

previous_transactions = old_test[
    old_test["test_row_id"].isin(old_ids)
].copy()

transaction_cols = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1",
    "Amount Received", "Receiving Currency",
    "Amount Paid", "Payment Currency",
    "Payment Format"
]

# Suvienodiname datos formatą
for frame in [new_sample, previous_transactions]:
    frame["Timestamp"] = pd.to_datetime(
        frame["Timestamp"]
    ).dt.strftime("%Y-%m-%d %H:%M:%S")

# Lyginame visas 10 operacijos charakteristikų
matches = new_sample.merge(
    previous_transactions[transaction_cols].drop_duplicates(),
    on=transaction_cols,
    how="inner"
)

print("GALUTINĖ PATIKRA")
print("=" * 50)
print("Ankstesnių unikalių ID:", len(old_ids))
print("Surastų ankstesnių operacijų:",
      len(previous_transactions))
print("Pasikartojimų naujoje imtyje:", len(matches))

if len(matches) == 0:
    print("\nŽinomų pasikartojimų nenustatyta.")
    print("Galime pereiti prie istorinių požymių.")
else:
    print("\nRasta pasikartojimų – juos reikės pakeisti.")
    print(matches["original_id"].tolist())

In [ ]:
import pandas as pd
import numpy as np

sample = pd.read_csv("aml_models/test_1000_enriched.csv")
sample["Timestamp"] = pd.to_datetime(sample["Timestamp"])

pair_cols = ["From Bank", "Account", "To Bank", "Account.1"]

# Visų operacijų istorija
history = df[pair_cols + ["Timestamp"]].copy()
history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Paliekame tik naujos imties siuntėjo–gavėjo poras
selected_pairs = sample[pair_cols].drop_duplicates()

history = history.merge(
    selected_pairs,
    on=pair_cols,
    how="inner"
)

# Sukuriame laiko masyvus kiekvienai porai
history_groups = {}

for key, group in history.groupby(pair_cols, sort=False):
    history_groups[key] = np.sort(
        group["Timestamp"].to_numpy(dtype="datetime64[ns]")
    )

seven_days = np.timedelta64(7, "D")
recipient_counts = []

for row in sample.itertuples(index=False, name=None):
    record = dict(zip(sample.columns, row))

    key = tuple(record[col] for col in pair_cols)
    timestamp = np.datetime64(record["Timestamp"], "ns")

    times = history_groups.get(key)

    if times is None:
        recipient_counts.append(0)
        continue

    start = np.searchsorted(
        times, timestamp - seven_days, side="left"
    )

    end = np.searchsorted(
        times, timestamp, side="left"
    )

    recipient_counts.append(int(end - start))

sample["same_recipient_7d_count"] = recipient_counts

assert len(sample) == 1000
assert sample["same_recipient_7d_count"].notna().all()
assert (sample["same_recipient_7d_count"] >= 0).all()

sample.to_csv(
    "aml_models/test_1000_recipient_context.csv",
    index=False
)

print("GAVĖJO ISTORIJOS PATIKRA")
print("=" * 50)
print("Operacijų:", len(sample))
print("Be ankstesnių pervedimų:",
      (sample["same_recipient_7d_count"] == 0).sum())
print("\nStatistika:")
print(sample["same_recipient_7d_count"].describe())
print("\nFailas išsaugotas!")

In [ ]:
import pandas as pd
import numpy as np

sample = pd.read_csv(
    "aml_models/test_1000_recipient_context.csv"
)
sample["Timestamp"] = pd.to_datetime(sample["Timestamp"])

sender_cols = ["From Bank", "Account"]
recipient_cols = ["To Bank", "Account.1"]

history_cols = sender_cols + recipient_cols + [
    "Timestamp", "Payment Currency", "Amount Paid"
]

# Atrenkame tik naujoje imtyje esančių siuntėjų istoriją
selected_senders = sample[sender_cols].drop_duplicates()

history = df[history_cols].merge(
    selected_senders,
    on=sender_cols,
    how="inner"
).copy()

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Istoriją sugrupuojame pagal siuntėją
sender_history = {}

for key, group in history.groupby(sender_cols, sort=False):
    group = group.sort_values("Timestamp")

    sender_history[key] = {
        "times": group["Timestamp"].to_numpy(
            dtype="datetime64[ns]"
        ),
        "recipients": list(zip(
            group["To Bank"],
            group["Account.1"]
        )),
        "currencies": group["Payment Currency"].to_numpy(),
        "amounts": group["Amount Paid"].to_numpy(dtype=float)
    }

day = np.timedelta64(24, "h")
week = np.timedelta64(7, "D")

count_24h = []
count_7d = []
amount_24h = []
unique_recipients_7d = []

for row in sample.itertuples(index=False, name=None):

    record = dict(zip(sample.columns, row))
    key = tuple(record[col] for col in sender_cols)

    timestamp = np.datetime64(
        record["Timestamp"], "ns"
    )

    group = sender_history.get(key)

    if group is None:
        count_24h.append(0)
        count_7d.append(0)
        amount_24h.append(0.0)
        unique_recipients_7d.append(0)
        continue

    times = group["times"]

    # side="left" neįtraukia operacijų tuo pačiu laiku
    end = np.searchsorted(
        times, timestamp, side="left"
    )

    start_24h = np.searchsorted(
        times, timestamp - day, side="left"
    )

    start_7d = np.searchsorted(
        times, timestamp - week, side="left"
    )

    count_24h.append(int(end - start_24h))
    count_7d.append(int(end - start_7d))

    # Sumuojamos tik tos pačios mokėjimo valiutos sumos
    currencies = group["currencies"][start_24h:end]
    amounts = group["amounts"][start_24h:end]

    same_currency = (
        currencies == record["Payment Currency"]
    )

    amount_24h.append(
        float(amounts[same_currency].sum())
    )

    recipients = group["recipients"][start_7d:end]

    unique_recipients_7d.append(
        len(set(recipients))
    )

sample["previous_24h_count"] = count_24h
sample["previous_7d_count"] = count_7d
sample["previous_24h_amount"] = amount_24h
sample["previous_7d_unique_recipients"] = unique_recipients_7d

context_cols = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

assert sample[context_cols].notna().all().all()
assert (sample[context_cols] >= 0).all().all()
assert (
    sample["previous_24h_count"] <=
    sample["previous_7d_count"]
).all()

output_path = "aml_models/test_1000_full_context.csv"

sample.to_csv(output_path, index=False)

print("VISŲ PENKIŲ ISTORINIŲ POŽYMIŲ PATIKRA")
print("=" * 60)

print("Operacijų:", len(sample))
print("AML:", int(sample["Is Laundering"].sum()))
print("Trūkstamų reikšmių:",
      int(sample[context_cols].isna().sum().sum()))

print("\nPožymių statistika:")
print(sample[context_cols].describe().round(2).to_string())

print("\nFailas išsaugotas:", output_path)

In [ ]:
import pandas as pd
import numpy as np

sample = pd.read_csv("aml_models/test_1000_full_context.csv")

print("XGBOOST PARUOŠIMO PATIKRA")
print("=" * 60)

print("\nModelio tipas:", type(model).__name__)
print("Preprocessor tipas:", type(preprocessor).__name__)

print("\nModelio požymių skaičius:")
print(getattr(model, "n_features_in_", "Nenurodytas"))

print("\nPreprocessor įvesties stulpeliai:")
if hasattr(preprocessor, "feature_names_in_"):
    print(preprocessor.feature_names_in_.tolist())
else:
    print("feature_names_in_ nėra")

print("\nPreprocessor transformacijos:")
if hasattr(preprocessor, "transformers_"):
    for name, transformer, columns in preprocessor.transformers_:
        print(f"{name}: {columns}")

print("\nNaujos imties duomenų tipai:")
print(sample.dtypes.to_string())

print("\nPirmosios trys datos:")
print(sample["Timestamp"].head(3).tolist())

In [ ]:
import pandas as pd
import numpy as np
import joblib

# Įkeliame išsaugotą XGBoost modelį
xgb_model = joblib.load("aml_models/xgb_model.joblib")
xgb_preprocessor = joblib.load("aml_models/preprocessor.joblib")

print("Modelio tipas:", type(xgb_model).__name__)

# Nauja imtis su visais penkiais istoriniais požymiais
sample = pd.read_csv("aml_models/test_1000_full_context.csv")

# Datos konvertavimas į sekundes
sample["Timestamp"] = pd.to_datetime(sample["Timestamp"])
sample["Timestamp_sec"] = (
    sample["Timestamp"].astype("int64") / 1e9
)

# Tik tie požymiai, kurie buvo naudojami mokant XGBoost
feature_cols = list(xgb_preprocessor.feature_names_in_)

X = sample[feature_cols].copy()
X_transformed = xgb_preprocessor.transform(X)

# XGBoost tikimybės
xgb_prob = xgb_model.predict_proba(X_transformed)[:, 1]

# Anksčiau validavimo imtyje nustatytas slenkstis
threshold = 0.957565

sample["xgb_probability"] = xgb_prob
sample["xgb_prediction"] = (
    sample["xgb_probability"] >= threshold
).astype(int)

# Rezultatų išsaugojimas
output_path = "aml_models/test_1000_xgb_context.csv"
sample.to_csv(output_path, index=False)

# Pagrindiniai rezultatai
y_true = sample["Is Laundering"]
y_pred = sample["xgb_prediction"]

tp = int(((y_true == 1) & (y_pred == 1)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("\nXGBOOST REZULTATAI – 1000 OPERACIJŲ")
print("=" * 55)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("Įspėjimų skaičius:", int(y_pred.sum()))
print("\nFailas:", output_path)

In [ ]:
import pandas as pd
import hashlib
from pathlib import Path

data_path = Path("aml_models/test_1000_xgb_context.csv")
prompt_path = Path("aml_models/gpt_context_prompt_v4.txt")

sample = pd.read_csv(data_path)
prompt = prompt_path.read_text(encoding="utf-8")

expected_hash = (
    "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
)

actual_hash = hashlib.sha256(
    prompt.encode("utf-8")
).hexdigest()

required_cols = [
    "original_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count",
    "xgb_probability",
    "xgb_prediction"
]

missing = [c for c in required_cols if c not in sample.columns]

print("GPT V4 PARUOŠIMO PATIKRA")
print("=" * 55)
print("Operacijų:", len(sample))
print("Unikalių ID:", sample["original_id"].nunique())
print("Trūkstami stulpeliai:", missing)
print("Trūkstamos reikšmės:",
      int(sample[required_cols].isna().sum().sum()) if not missing else "Netikrinta")
print("GPT V4 instrukcijos SHA256:", actual_hash)
print("Instrukcija sutampa su ankstesne:", actual_hash == expected_hash)
print("Instrukcijos ilgis:", len(prompt), "simbolių")

print("\nAr paruošta GPT eksperimentui:",
      len(sample) == 1000
      and sample["original_id"].is_unique
      and not missing
      and actual_hash == expected_hash)

In [ ]:
import os
import pandas as pd
import importlib.util

print("GPT API APLINKOS PATIKRA")
print("=" * 55)

print("OpenAI biblioteka:",
      importlib.util.find_spec("openai") is not None)

print("OPENAI_API_KEY nustatytas:",
      bool(os.environ.get("OPENAI_API_KEY")))

print("OpenAI klientas atmintyje:",
      "client" in globals())

results_path = "aml_models/gpt_v4_natural_3000_results.csv"

if os.path.exists(results_path):
    previous = pd.read_csv(results_path)

    print("\nAnkstesnio GPT eksperimento stulpeliai:")
    print(previous.columns.tolist())

    if "status" in previous.columns:
        print("\nStatusų pasiskirstymas:")
        print(previous["status"].value_counts().to_string())

    print("\nAnkstesnių rezultatų skaičius:", len(previous))

print("\nNaujas GPT eksperimentas dar nepaleistas.")

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path
from openai import OpenAI

client = OpenAI()

sample = pd.read_csv("aml_models/test_1000_xgb_context.csv")

prompt = Path(
    "aml_models/gpt_context_prompt_v4.txt"
).read_text(encoding="utf-8")

# Parenkame 2 AML ir 3 teisėtas operacijas
pilot = pd.concat([
    sample[sample["Is Laundering"] == 1].head(2),
    sample[sample["Is Laundering"] == 0].head(3)
])

features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

results = []

for _, row in pilot.iterrows():

    transaction = {
        col: row[col].item()
        if hasattr(row[col], "item")
        else row[col]
        for col in features
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False,
                default=str
            ),
            temperature=0
        )

        raw = response.output_text
        parsed = json.loads(raw)

        prediction = int(parsed["prediction"])
        reason = str(parsed.get("reason", ""))

        if prediction not in (0, 1):
            raise ValueError("Neteisinga prognozė")

        status = "OK"

    except Exception as e:
        raw = locals().get("raw", "")
        prediction = None
        reason = ""
        status = f"ERROR: {str(e)}"

    results.append({
        "original_id": int(row["original_id"]),
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": prediction,
        "gpt_reason": reason,
        "raw_response": raw,
        "status": status,
        "input_tokens": (
            response.usage.input_tokens
            if status == "OK" else None
        ),
        "output_tokens": (
            response.usage.output_tokens
            if status == "OK" else None
        )
    })

    print(
        f"ID {int(row['original_id'])}: "
        f"GPT={prediction}, "
        f"Tikroji žyma={int(row['Is Laundering'])}, "
        f"Statusas={status}"
    )

    time.sleep(0.2)

results_df = pd.DataFrame(results)

output_path = "aml_models/gpt_v4_enriched_1000_pilot.csv"
results_df.to_csv(output_path, index=False)

print("\nBANDOMOJO TESTO REZULTATAI")
print("=" * 55)
print(results_df["status"].value_counts())
print("\nFailas:", output_path)

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path
from openai import OpenAI

client = OpenAI()

sample = pd.read_csv("aml_models/test_1000_xgb_context.csv")

prompt = Path(
    "aml_models/gpt_context_prompt_v4.txt"
).read_text(encoding="utf-8")

output_path = Path("aml_models/gpt_v4_enriched_1000_results.csv")
pilot_path = Path("aml_models/gpt_v4_enriched_1000_pilot.csv")

features = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1",
    "Amount Received", "Receiving Currency",
    "Amount Paid", "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

# Tęsiame ankstesnį vykdymą arba pradedame nuo pilotinių rezultatų
if output_path.exists():
    results_df = pd.read_csv(output_path)
else:
    results_df = pd.read_csv(pilot_path)

# Kartojame tik nepavykusias arba dar nevertintas operacijas
completed_ids = set(
    results_df.loc[
        results_df["status"] == "OK", "original_id"
    ].astype(int)
)

remaining = sample[
    ~sample["original_id"].isin(completed_ids)
].copy()

print("Jau įvertinta:", len(completed_ids))
print("Liko įvertinti:", len(remaining))

results = results_df.to_dict("records")

for i, (_, row) in enumerate(remaining.iterrows(), start=1):

    transaction = {
        col: row[col].item()
        if hasattr(row[col], "item")
        else row[col]
        for col in features
    }

    response = None
    raw = ""
    prediction = None
    reason = ""
    input_tokens = None
    output_tokens = None

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False,
                default=str
            ),
            temperature=0
        )

        raw = response.output_text
        parsed = json.loads(raw)

        prediction = int(parsed["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Prognozė turi būti 0 arba 1")

        reason = str(parsed.get("reason", ""))
        status = "OK"

        if response.usage:
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens

    except Exception as e:
        status = f"ERROR: {str(e)}"

    # Jei šios operacijos ankstesnis bandymas nepavyko,
    # pašaliname jo seną įrašą
    results = [
        r for r in results
        if int(r["original_id"]) != int(row["original_id"])
    ]

    results.append({
        "original_id": int(row["original_id"]),
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": prediction,
        "gpt_reason": reason,
        "raw_response": raw,
        "status": status,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    })

    if i % 20 == 0 or i == len(remaining):
        pd.DataFrame(results).to_csv(output_path, index=False)

        successful = sum(
            r["status"] == "OK" for r in results
        )

        print(
            f"Apdorota šiame paleidime: {i}/{len(remaining)} | "
            f"Iš viso sėkmingai: {successful}/1000"
        )

    time.sleep(0.2)

final_results = pd.DataFrame(results)
final_results.to_csv(output_path, index=False)

print("\nGALUTINĖ BŪSENA")
print("=" * 50)
print(final_results["status"].value_counts().to_string())
print("\nFailas:", output_path)

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("aml_models/gpt_v4_enriched_1000_results.csv")

if path.exists():
    results = pd.read_csv(path)

    successful = results[
        results["status"] == "OK"
    ]["original_id"].nunique()

    print("Išsaugotų įrašų:", len(results))
    print("Sėkmingai įvertintų operacijų:", successful)
    print("Liko įvertinti:", 1000 - successful)

    print("\nStatusai:")
    print(results["status"].value_counts())
else:
    print("Rezultatų failas nerastas.")

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path
from openai import OpenAI

# Ribojame vienos užklausos laukimo laiką
client = OpenAI(timeout=30.0, max_retries=2)

sample = pd.read_csv("aml_models/test_1000_xgb_context.csv")

output_path = Path("aml_models/gpt_v4_enriched_1000_results.csv")
results_df = pd.read_csv(output_path)

prompt = Path(
    "aml_models/gpt_context_prompt_v4.txt"
).read_text(encoding="utf-8")

features = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1",
    "Amount Received", "Receiving Currency",
    "Amount Paid", "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

completed_ids = set(
    results_df.loc[
        results_df["status"] == "OK", "original_id"
    ].astype(int)
)

remaining = sample[
    ~sample["original_id"].isin(completed_ids)
].copy()

print(f"Liko įvertinti: {len(remaining)}")
print("=" * 55, flush=True)

for i, (_, row) in enumerate(remaining.iterrows(), start=1):

    transaction = {
        col: row[col].item()
        if hasattr(row[col], "item")
        else row[col]
        for col in features
    }

    raw = ""
    prediction = None
    reason = ""
    input_tokens = None
    output_tokens = None

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False,
                default=str
            ),
            temperature=0
        )

        raw = response.output_text
        parsed = json.loads(raw)

        prediction = int(parsed["prediction"])

        if prediction not in (0, 1):
            raise ValueError("Neteisinga prognozė")

        reason = str(parsed.get("reason", ""))
        status = "OK"

        if response.usage:
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens

    except Exception as e:
        status = f"ERROR: {type(e).__name__}: {e}"

    new_result = {
        "original_id": int(row["original_id"]),
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": prediction,
        "gpt_reason": reason,
        "raw_response": raw,
        "status": status,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

    # Pašaliname galimą ankstesnį nesėkmingą įrašą
    results_df = results_df[
        results_df["original_id"] != int(row["original_id"])
    ]

    results_df = pd.concat(
        [results_df, pd.DataFrame([new_result])],
        ignore_index=True
    )

    # Išsaugome iš karto po kiekvienos užklausos
    results_df.to_csv(output_path, index=False)

    successful = int((results_df["status"] == "OK").sum())

    print(
        f"{i}/{len(remaining)} | "
        f"ID={int(row['original_id'])} | "
        f"{status} | "
        f"Iš viso: {successful}/1000",
        flush=True
    )

    time.sleep(0.2)

print("\nGALUTINĖ BŪSENA")
print("=" * 55)
print(results_df["status"].value_counts().to_string())

print("\nRezultatų failas:", output_path)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

sample = pd.read_csv("aml_models/test_1000_xgb_context.csv")
gpt = pd.read_csv("aml_models/gpt_v4_enriched_1000_results.csv")

assert len(gpt) == 1000
assert gpt["original_id"].is_unique
assert (gpt["status"] == "OK").all()

data = sample.merge(
    gpt[["original_id", "gpt_prediction", "true_label"]],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

assert len(data) == 1000
assert (
    data["Is Laundering"].astype(int)
    == data["true_label"].astype(int)
).all()

y = data["Is Laundering"].astype(int)
xgb_pred = data["xgb_prediction"].astype(int)
gpt_pred = data["gpt_prediction"].astype(int)

data["or_prediction"] = (
    (xgb_pred == 1) | (gpt_pred == 1)
).astype(int)

data["and_prediction"] = (
    (xgb_pred == 1) & (gpt_pred == 1)
).astype(int)

models = {
    "XGBoost": xgb_pred,
    "GPT-5.4-mini": gpt_pred,
    "OR hibridas": data["or_prediction"],
    "AND hibridas": data["and_prediction"]
}

results = []

for name, pred in models.items():
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Modelis": name,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": round(
            precision_score(y, pred, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y, pred, zero_division=0), 4
        ),
        "F1": round(
            f1_score(y, pred, zero_division=0), 4
        ),
        "FPR": round(fp / (fp + tn), 4),
        "Įspėjimai": int(pred.sum())
    })

comparison = pd.DataFrame(results)

print("MODELIŲ PALYGINIMAS – 1000 OPERACIJŲ")
print("=" * 90)
print(comparison.to_string(index=False))

aml = data[y == 1]

both = int(
    ((aml["xgb_prediction"] == 1) &
     (aml["gpt_prediction"] == 1)).sum()
)

only_xgb = int(
    ((aml["xgb_prediction"] == 1) &
     (aml["gpt_prediction"] == 0)).sum()
)

only_gpt = int(
    ((aml["xgb_prediction"] == 0) &
     (aml["gpt_prediction"] == 1)).sum()
)

neither = int(
    ((aml["xgb_prediction"] == 0) &
     (aml["gpt_prediction"] == 0)).sum()
)

print("\nAML APTIKIMO PERSIDENGIMAS")
print("=" * 55)
print("Aptiko abu modeliai:", both)
print("Aptiko tik XGBoost:", only_xgb)
print("Aptiko tik GPT:", only_gpt)
print("Neaptiko nė vienas:", neither)

assert both + only_xgb + only_gpt + neither == 100

comparison.to_csv(
    "aml_models/gpt_v4_enriched_1000_comparison.csv",
    index=False
)

data.to_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

data["risk_band"] = pd.cut(
    data["xgb_probability"],
    bins=[-0.001, 0.5, 0.8, 0.9, 0.95, 1.0],
    labels=[
        "0–0.50",
        "0.50–0.80",
        "0.80–0.90",
        "0.90–0.95",
        "0.95–1.00"
    ]
)

summary = data.groupby(
    ["risk_band", "gpt_prediction"],
    observed=True
).agg(
    operacijos=("original_id", "count"),
    aml=("Is Laundering", "sum")
).reset_index()

summary["aml_dalis"] = (
    summary["aml"] / summary["operacijos"]
).round(4)

print("GPT SPRENDIMAI PAGAL XGBOOST RIZIKOS LYGĮ")
print("=" * 70)
print(summary.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

assert len(data) == 1000
assert data["original_id"].is_unique

data["or_prediction"] = (
    (data["xgb_prediction"] == 1) |
    (data["gpt_prediction"] == 1)
).astype(int)

budgets = [25, 50, 100, 200]

strategies = {
    "XGBoost": data.sort_values(
        ["xgb_probability", "original_id"],
        ascending=[False, True]
    ),
    "GPT + XGBoost": data.sort_values(
        ["gpt_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True]
    ),
    "OR + XGBoost": data.sort_values(
        ["or_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, False, True]
    )
}

results = []

total_aml = int(data["Is Laundering"].sum())

for budget in budgets:
    for name, ranked in strategies.items():

        selected = ranked.head(budget)

        tp = int(selected["Is Laundering"].sum())
        fp = budget - tp

        results.append({
            "Biudzetas": budget,
            "Strategija": name,
            "Aptikta_AML": tp,
            "Klaidingi_ispejimai": fp,
            "Precision": round(tp / budget, 4),
            "Recall": round(tp / total_aml, 4)
        })

comparison = pd.DataFrame(results)

print("VIENODO ĮSPĖJIMŲ BIUDŽETO PALYGINIMAS")
print("=" * 85)
print(comparison.to_string(index=False))

print("\nPAPILDOMAI APTIKTI AML, PALYGINTI SU XGBOOST")
print("=" * 65)

pivot = comparison.pivot(
    index="Biudzetas",
    columns="Strategija",
    values="Aptikta_AML"
)

pivot["GPT_pranasumas"] = (
    pivot["GPT + XGBoost"] - pivot["XGBoost"]
)

pivot["OR_pranasumas"] = (
    pivot["OR + XGBoost"] - pivot["XGBoost"]
)

print(pivot.to_string())

comparison.to_csv(
    "aml_models/hybrid_1000_equal_alert_budget.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

assert len(data) == 1000
assert data["original_id"].is_unique

data["or_prediction"] = (
    (data["xgb_prediction"] == 1) |
    (data["gpt_prediction"] == 1)
).astype(int)

budgets = [25, 50, 100, 200]

strategies = {
    "XGBoost": data.sort_values(
        ["xgb_probability", "original_id"],
        ascending=[False, True]
    ),
    "GPT + XGBoost": data.sort_values(
        ["gpt_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True]
    ),
    "OR + XGBoost": data.sort_values(
        ["or_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True]
    )
}

results = []
total_aml = int(data["Is Laundering"].sum())

for budget in budgets:
    for name, ranked in strategies.items():

        selected = ranked.head(budget)

        tp = int(selected["Is Laundering"].sum())
        fp = budget - tp

        results.append({
            "Biudzetas": budget,
            "Strategija": name,
            "Aptikta_AML": tp,
            "Klaidingi_ispejimai": fp,
            "Precision": round(tp / budget, 4),
            "Recall": round(tp / total_aml, 4)
        })

comparison = pd.DataFrame(results)

print("VIENODO ĮSPĖJIMŲ BIUDŽETO PALYGINIMAS")
print("=" * 85)
print(comparison.to_string(index=False))

print("\nPAPILDOMAI APTIKTI AML, PALYGINTI SU XGBOOST")
print("=" * 65)

pivot = comparison.pivot(
    index="Biudzetas",
    columns="Strategija",
    values="Aptikta_AML"
)

pivot["GPT_pranasumas"] = (
    pivot["GPT + XGBoost"] - pivot["XGBoost"]
)

pivot["OR_pranasumas"] = (
    pivot["OR + XGBoost"] - pivot["XGBoost"]
)

print(pivot.to_string())

comparison.to_csv(
    "aml_models/hybrid_1000_equal_alert_budget.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
from statsmodels.stats.proportion import proportion_confint

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

y = data["Is Laundering"].astype(int)
xgb = data["xgb_prediction"].astype(int)
gpt = data["gpt_prediction"].astype(int)

predictions = {
    "XGBoost": xgb,
    "GPT-5.4-mini": gpt,
    "OR hibridas": (xgb | gpt).astype(int),
    "AND hibridas": (xgb & gpt).astype(int)
}

rows = []

for name, pred in predictions.items():

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    measures = {
        "Recall": (tp, tp + fn),
        "FPR": (fp, fp + tn),
        "Precision": (tp, tp + fp)
    }

    for metric, (successes, total) in measures.items():

        if total == 0:
            estimate, lower, upper = float("nan"), float("nan"), float("nan")
        else:
            estimate = successes / total
            lower, upper = proportion_confint(
                successes,
                total,
                alpha=0.05,
                method="wilson"
            )

        rows.append({
            "Modelis": name,
            "Rodiklis": metric,
            "Ivertis": round(estimate, 4),
            "CI_95_nuo": round(lower, 4),
            "CI_95_iki": round(upper, 4),
            "Skaitiklis": int(successes),
            "Vardiklis": int(total)
        })

ci_results = pd.DataFrame(rows)

print("95 % PASIKLIAUTINIEJI INTERVALAI")
print("=" * 90)
print(ci_results.to_string(index=False))

ci_results.to_csv(
    "aml_models/hybrid_1000_confidence_intervals.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

data["or_prediction"] = (
    (data["xgb_prediction"] == 1) |
    (data["gpt_prediction"] == 1)
).astype(int)

comparisons = [
    ("AML", 1, "XGBoost", "GPT-5.4-mini",
     "xgb_prediction", "gpt_prediction"),

    ("Teisėtos", 0, "XGBoost", "GPT-5.4-mini",
     "xgb_prediction", "gpt_prediction"),

    ("AML", 1, "GPT-5.4-mini", "OR hibridas",
     "gpt_prediction", "or_prediction")
]

results = []

for group_name, true_label, name_a, name_b, col_a, col_b in comparisons:

    subset = data[
        data["Is Laundering"] == true_label
    ].copy()

    a = subset[col_a].astype(int)
    b = subset[col_b].astype(int)

    both_0 = int(((a == 0) & (b == 0)).sum())
    a0_b1 = int(((a == 0) & (b == 1)).sum())
    a1_b0 = int(((a == 1) & (b == 0)).sum())
    both_1 = int(((a == 1) & (b == 1)).sum())

    table = [
        [both_0, a0_b1],
        [a1_b0, both_1]
    ]

    # Tikslusis McNemar testas
    test = mcnemar(
        table,
        exact=True
    )

    results.append({
        "Grupe": group_name,
        "Modelis_A": name_a,
        "Modelis_B": name_b,
        "A0_B1": a0_b1,
        "A1_B0": a1_b0,
        "p_reiksme": test.pvalue,
        "Statistiskai_reiksminga_0_05": test.pvalue < 0.05
    })

comparison = pd.DataFrame(results)

print("McNEMAR TESTO REZULTATAI")
print("=" * 100)
print(comparison.to_string(index=False))

comparison.to_csv(
    "aml_models/hybrid_1000_mcnemar.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "aml_models/gpt_v4_enriched_1000_merged.csv"
)

required = [
    "Is Laundering", "xgb_probability",
    "xgb_prediction", "gpt_prediction"
]
assert all(c in data.columns for c in required)

data["or_prediction"] = (
    (data["xgb_prediction"] == 1) |
    (data["gpt_prediction"] == 1)
).astype(int)

aml = data[data["Is Laundering"] == 1].copy()
legit = data[data["Is Laundering"] == 0].copy()

assert len(aml) == 100
assert len(legit) == 900

rng = np.random.default_rng(42)
n_bootstrap = 2000
budgets = [25, 50, 100, 200]

def detected_aml(sample, budget, strategy):
    if strategy == "XGBoost":
        ranked = sample.sort_values(
            "xgb_probability", ascending=False,
            kind="stable"
        )
    else:
        ranked = sample.sort_values(
            ["or_prediction", "xgb_probability"],
            ascending=[False, False],
            kind="stable"
        )

    return int(
        ranked.head(budget)["Is Laundering"].sum()
    )

results = []

for budget in budgets:
    differences = []

    observed = (
        detected_aml(data, budget, "OR")
        - detected_aml(data, budget, "XGBoost")
    )

    for _ in range(n_bootstrap):
        sampled_aml = aml.iloc[
            rng.integers(0, len(aml), len(aml))
        ]

        sampled_legit = legit.iloc[
            rng.integers(0, len(legit), len(legit))
        ]

        sample = pd.concat(
            [sampled_aml, sampled_legit],
            ignore_index=True
        )

        xgb_tp = detected_aml(
            sample, budget, "XGBoost"
        )
        or_tp = detected_aml(
            sample, budget, "OR"
        )

        differences.append(or_tp - xgb_tp)

    lower, upper = np.percentile(
        differences, [2.5, 97.5]
    )

    results.append({
        "Biudzetas": budget,
        "Stebetas_OR_minus_XGB": observed,
        "Bootstrap_vidurkis": round(
            np.mean(differences), 3
        ),
        "CI_95_nuo": round(lower, 3),
        "CI_95_iki": round(upper, 3),
        "P_OR_geresnis": round(
            np.mean(np.array(differences) > 0), 4
        )
    })

bootstrap_results = pd.DataFrame(results)

print("VIENODO ĮSPĖJIMŲ BIUDŽETO BOOTSTRAP ANALIZĖ")
print("=" * 95)
print(bootstrap_results.to_string(index=False))

bootstrap_results.to_csv(
    "aml_models/hybrid_1000_budget_bootstrap.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

existing = pd.read_csv(
    folder / "test_1000_full_context.csv"
)

print("ESAMA IMTIS")
print("Operacijų:", len(existing))
print("AML:", int(existing["Is Laundering"].sum()))
print("Teisėtų:", int((existing["Is Laundering"] == 0).sum()))

assert len(existing) == 1000
assert existing["original_id"].is_unique

# Atkuriame chronologinio testavimo rinkinio indeksus
df_sorted = df.sort_values(
    "Timestamp", kind="stable"
).reset_index()

test_start = 3248921 + 965524

test_pool = df_sorted.iloc[test_start:].copy()

# original_id turi reikšti pradinį df indeksą
test_pool = test_pool.rename(
    columns={"index": "original_id"}
)

# Pašaliname jau naudotas operacijas
known_ids = set(existing["original_id"])

# Papildomi ankstesnių eksperimentų ID
for filename in [
    "comparison_3000_context_v4_corrected.csv",
    "gpt_v4_natural_3000_results.csv",
    "test_1000_enriched.csv"
]:
    path = folder / filename
    if path.exists():
        old = pd.read_csv(path)
        if "original_id" in old.columns:
            known_ids.update(old["original_id"].dropna())

available = test_pool[
    ~test_pool["original_id"].isin(known_ids)
]

print("\nNAUJOMS OPERACIJOMS PRIEINAMI DUOMENYS")
print("Iš viso:", len(available))
print("AML:", int(available["Is Laundering"].sum()))
print("Teisėtų:", int((available["Is Laundering"] == 0).sum()))

print("\nREIKALINGA PAPILDOMAI")
print("AML: 200")
print("Teisėtų: 1800")

print("\nAR PAKANKA DUOMENŲ?")
print(
    "AML:",
    int(available["Is Laundering"].sum()) >= 200
)
print(
    "Teisėtų:",
    int((available["Is Laundering"] == 0).sum()) >= 1800
)

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

existing = pd.read_csv(folder / "test_1000_full_context.csv")

df_sorted = df.sort_values(
    "Timestamp", kind="stable"
).reset_index().rename(columns={"index": "original_id"})

test_start = 3248921 + 965524
test_pool = df_sorted.iloc[test_start:].copy()

print("IDENTIFIKATORIŲ PATIKRA")
print("=" * 65)

print("Esamos imties ID unikalūs:", existing["original_id"].is_unique)
print("Testavimo rinkinio ID unikalūs:", test_pool["original_id"].is_unique)

matched = existing.merge(
    test_pool,
    on="original_id",
    how="left",
    suffixes=("_saved", "_source"),
    indicator=True
)

print("Sutampantys ID:", int((matched["_merge"] == "both").sum()))
print("Nerasti ID:", int((matched["_merge"] != "both").sum()))

for column in ["Timestamp", "Is Laundering", "Amount Paid"]:
    saved = column + "_saved"
    source = column + "_source"

    if saved in matched.columns and source in matched.columns:
        if column == "Timestamp":
            a = pd.to_datetime(matched[saved], errors="coerce")
            b = pd.to_datetime(matched[source], errors="coerce")
        elif column == "Amount Paid":
            a = pd.to_numeric(matched[saved], errors="coerce").round(6)
            b = pd.to_numeric(matched[source], errors="coerce").round(6)
        else:
            a = matched[saved]
            b = matched[source]

        matches = int((a.notna() & b.notna() & (a == b)).sum())
        print(f"{column} reikšmių sutapimai: {matches}/1000")

print("\nANKSTESNIŲ FAILŲ PATIKRA")
print("=" * 65)

files = [
    "comparison_3000_context_v4_corrected.csv",
    "gpt_v4_natural_3000_results.csv",
    "test_1000_enriched.csv"
]

for filename in files:
    path = folder / filename

    if not path.exists():
        print(filename, "- failas nerastas")
        continue

    old = pd.read_csv(path)

    if "original_id" not in old.columns:
        print(filename, "- nėra original_id")
        continue

    ids = set(old["original_id"].dropna())
    overlap = len(ids & set(existing["original_id"]))

    print(
        filename,
        "| unikalių ID:", len(ids),
        "| sutampa su 1000 imtimi:", overlap
    )

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

existing = pd.read_csv(
    folder / "test_1000_full_context.csv"
)

df_sorted = (
    df.sort_values("Timestamp", kind="stable")
      .reset_index()
      .rename(columns={"index": "original_id"})
)

test_start = 3248921 + 965524
test_pool = df_sorted.iloc[test_start:].copy()

# Surenkame ankstesnių eksperimentų identifikatorius
excluded_ids = set(existing["original_id"])

for filename in [
    "comparison_3000_context_v4_corrected.csv",
    "gpt_v4_natural_3000_results.csv",
    "test_1000_enriched.csv"
]:
    path = folder / filename

    if path.exists():
        previous = pd.read_csv(path)

        if "original_id" in previous.columns:
            excluded_ids.update(
                previous["original_id"].dropna().tolist()
            )

available = test_pool[
    ~test_pool["original_id"].isin(excluded_ids)
].copy()

# Atsitiktinė, atkuriama atranka pagal tikrąją klasę
new_aml = available[
    available["Is Laundering"] == 1
].sample(n=200, random_state=42)

new_legit = available[
    available["Is Laundering"] == 0
].sample(n=1800, random_state=42)

new_sample = pd.concat(
    [new_aml, new_legit],
    ignore_index=True
)

new_sample = new_sample.sort_values(
    ["Timestamp", "original_id"],
    kind="stable"
).reset_index(drop=True)

# Patikriname nepersidengimą
assert len(new_sample) == 2000
assert new_sample["original_id"].is_unique
assert new_sample["Is Laundering"].sum() == 200
assert len(
    set(new_sample["original_id"]) & excluded_ids
) == 0

new_sample.to_csv(
    folder / "test_2000_additional_raw.csv",
    index=False
)

print("NAUJA PAPILDOMA IMTIS")
print("=" * 55)
print("Operacijų:", len(new_sample))
print("AML:", int(new_sample["Is Laundering"].sum()))
print("Teisėtų:", int((new_sample["Is Laundering"] == 0).sum()))
print("Unikalūs ID:", new_sample["original_id"].is_unique)
print("Persidengimas su ankstesnėmis imtimis: 0")

print("\nGALUTINĖ PLANUOJAMA IMTIS")
print("Operacijų: 3000")
print("AML: 300")
print("Teisėtų: 2700")

print("\nFailas išsaugotas: aml_models/test_2000_additional_raw.csv")

In [ ]:
import pandas as pd
import numpy as np
from collections import deque, Counter
from pathlib import Path

folder = Path("aml_models")
sample = pd.read_csv(folder / "test_2000_additional_raw.csv")

# Šaltinio duomenys su pradiniais indeksais
source = df.copy().reset_index().rename(
    columns={"index": "original_id"}
)

source["Timestamp"] = pd.to_datetime(source["Timestamp"])
sample["Timestamp"] = pd.to_datetime(sample["Timestamp"])

target_ids = set(sample["original_id"])

# Reikalingi stulpeliai
cols = [
    "original_id", "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1", "Amount Paid", "Payment Currency"
]

source = source[cols].copy()

# Skaičiuojame tik tų siuntėjų istoriją, kurie yra imtyje
sender_keys = sample[["From Bank", "Account"]].drop_duplicates()

source = source.merge(
    sender_keys,
    on=["From Bank", "Account"],
    how="inner"
)

source = source.sort_values(
    ["From Bank", "Account", "Timestamp", "original_id"],
    kind="stable"
)

results = {}

for _, group in source.groupby(
    ["From Bank", "Account"], sort=False
):
    group = group.sort_values(
        ["Timestamp", "original_id"],
        kind="stable"
    )

    history_24h = deque()
    history_7d = deque()

    amount_by_currency = Counter()
    recipient_counts = Counter()

    rows = group.to_dict("records")
    i = 0

    while i < len(rows):
        timestamp = rows[i]["Timestamp"]

        # Pašaliname pasenusią istoriją
        while history_24h and (
            timestamp - history_24h[0][0]
        ) > pd.Timedelta(hours=24):
            _, currency, amount = history_24h.popleft()
            amount_by_currency[currency] -= amount

        while history_7d and (
            timestamp - history_7d[0][0]
        ) > pd.Timedelta(days=7):
            _, recipient = history_7d.popleft()
            recipient_counts[recipient] -= 1

            if recipient_counts[recipient] == 0:
                del recipient_counts[recipient]

        # Vienodo laiko operacijos neturi matyti viena kitos
        j = i
        while j < len(rows) and rows[j]["Timestamp"] == timestamp:
            j += 1

        for k in range(i, j):
            row = rows[k]
            oid = row["original_id"]

            if oid not in target_ids:
                continue

            recipient = (row["To Bank"], row["Account.1"])
            currency = row["Payment Currency"]

            results[oid] = {
                "previous_24h_count": len(history_24h),
                "previous_7d_count": len(history_7d),
                "previous_24h_amount": float(
                    amount_by_currency[currency]
                ),
                "previous_7d_unique_recipients": len(recipient_counts),
                "same_recipient_7d_count": recipient_counts.get(
                    recipient, 0
                )
            }

        # Tik po skaičiavimo įtraukiame to paties laiko operacijas
        for k in range(i, j):
            row = rows[k]
            recipient = (row["To Bank"], row["Account.1"])
            currency = row["Payment Currency"]
            amount = float(row["Amount Paid"])

            history_24h.append((timestamp, currency, amount))
            history_7d.append((timestamp, recipient))

            amount_by_currency[currency] += amount
            recipient_counts[recipient] += 1

        i = j

context = pd.DataFrame.from_dict(
    results, orient="index"
)
context.index.name = "original_id"
context = context.reset_index()

final = sample.merge(
    context, on="original_id", how="left", validate="one_to_one"
)

features = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

assert len(final) == 2000
assert final[features].notna().all().all()

final.to_csv(
    folder / "test_2000_additional_full_context.csv",
    index=False
)

print("ISTORINIŲ POŽYMIŲ SKAIČIAVIMAS BAIGTAS")
print("=" * 65)
print("Operacijų:", len(final))
print("AML:", int(final["Is Laundering"].sum()))
print("Trūkstamų reikšmių:", int(final[features].isna().sum().sum()))

print("\nPOŽYMIŲ STATISTIKA")
print(final[features].describe().round(2).to_string())

print("\nIšsaugota: aml_models/test_2000_additional_full_context.csv")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

folder = Path("aml_models")

data = pd.read_csv(
    folder / "test_2000_additional_full_context.csv"
)

xgb_model = joblib.load(folder / "xgb_model.joblib")
xgb_preprocessor = joblib.load(folder / "preprocessor.joblib")

threshold = 0.957565

# Atkuriame XGBoost mokymo metu naudotus požymius
data["Timestamp"] = pd.to_datetime(data["Timestamp"])
data["Timestamp_sec"] = (
    data["Timestamp"].astype("int64") / 1e9
)

numeric_features = [
    "Timestamp_sec",
    "Amount Received",
    "Amount Paid"
]

categorical_features = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format"
]

features = numeric_features + categorical_features

assert len(data) == 2000
assert data["original_id"].is_unique
assert int(data["Is Laundering"].sum()) == 200
assert data[features].notna().all().all()

# Naudojame jau išmokytą duomenų transformatorių
X = xgb_preprocessor.transform(data[features])

# Prognozės
data["xgb_probability"] = xgb_model.predict_proba(X)[:, 1]

data["xgb_prediction"] = (
    data["xgb_probability"] >= threshold
).astype(int)

y_true = data["Is Laundering"].astype(int)
y_pred = data["xgb_prediction"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("XGBOOST REZULTATAI – NAUJOS 2000 OPERACIJŲ")
print("=" * 65)

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print("\nPrecision:", round(precision_score(y_true, y_pred), 4))
print("Recall:", round(recall_score(y_true, y_pred), 4))
print("F1:", round(f1_score(y_true, y_pred), 4))
print("FPR:", round(fp / (fp + tn), 4))
print("Įspėjimų skaičius:", int(y_pred.sum()))

# Išsaugome prognozes
output_path = folder / "test_2000_additional_xgb_context.csv"

data.to_csv(output_path, index=False)

print("\nIšsaugota:", output_path)

print("\nXGBoost tikimybių statistika:")
print(
    data.groupby("Is Laundering")["xgb_probability"]
        .describe()
        .round(4)
        .to_string()
)

In [ ]:
import pandas as pd
import hashlib
import json
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "test_2000_additional_xgb_context.csv"
)

prompt_path = folder / "gpt_context_prompt_v4.txt"
prompt = prompt_path.read_text(encoding="utf-8")

prompt_hash = hashlib.sha256(
    prompt.encode("utf-8")
).hexdigest()

expected_hash = (
    "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
)

raw_features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

context_features = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

gpt_features = raw_features + context_features

assert len(data) == 2000
assert data["original_id"].is_unique
assert data[gpt_features].notna().all().all()
assert prompt_hash == expected_hash, "V4 promptas nesutampa!"

# Patikriname vienos operacijos JSON struktūrą
example = data.iloc[0][gpt_features].to_dict()

# Konvertuojame numpy reikšmes į standartinius Python tipus
example = json.loads(
    json.dumps(example, default=str)
)

print("GPT-5.4-mini PARUOŠIMO PATIKRA")
print("=" * 65)
print("Operacijų:", len(data))
print("Prompt SHA256:", prompt_hash)
print("Promptas sutampa:", prompt_hash == expected_hash)
print("GPT įvesties požymių:", len(gpt_features))
print("Trūkstamų reikšmių:", int(data[gpt_features].isna().sum().sum()))

print("\nPAVYZDINĖ GPT ĮVESTIS")
print(json.dumps(example, indent=2, ensure_ascii=False))

print("\nAR GPT ĮVESTYJE NĖRA DRAUDŽIAMŲ STULPELIŲ?")
for col in [
    "Is Laundering",
    "xgb_probability",
    "xgb_prediction",
    "original_id"
]:
    print(f"{col}: {col not in example}")

print("\nPatikra baigta. API užklausų nebuvo.")

In [ ]:
import pandas as pd
import json
import time
from pathlib import Path
from openai import OpenAI

folder = Path("aml_models")

data = pd.read_csv(
    folder / "test_2000_additional_xgb_context.csv"
)

prompt = (folder / "gpt_context_prompt_v4.txt").read_text(
    encoding="utf-8"
)

gpt_features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

# Tik bandomajai imčiai parenkame žinomas klases.
# Tikroji klasė GPT modeliui NEPERDUODAMA.
pilot = pd.concat([
    data[data["Is Laundering"] == 1].head(2),
    data[data["Is Laundering"] == 0].head(3)
])

client = OpenAI(timeout=30.0, max_retries=2)

results = []

for _, row in pilot.iterrows():

    transaction = {
        col: row[col].item()
        if hasattr(row[col], "item")
        else row[col]
        for col in gpt_features
    }

    result = {
        "original_id": row["original_id"],
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(transaction, ensure_ascii=False),
            temperature=0
        )

        raw = response.output_text.strip()

        # Jei atsakymas pateiktas Markdown JSON bloke
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        parsed = json.loads(raw)
        prediction = parsed["prediction"]

        if type(prediction) is not int or prediction not in (0, 1):
            raise ValueError("Neteisinga prediction reikšmė")

        result["gpt_prediction"] = prediction
        result["gpt_reason"] = str(parsed.get("reason", ""))
        result["raw_response"] = response.output_text
        result["status"] = "OK"

        if response.usage:
            result["input_tokens"] = response.usage.input_tokens
            result["output_tokens"] = response.usage.output_tokens

    except Exception as e:
        result["raw_response"] = str(e)
        result["status"] = "ERROR"

    results.append(result)

    print(
        f"ID: {result['original_id']} | "
        f"Tikra klasė: {result['true_label']} | "
        f"GPT: {result['gpt_prediction']} | "
        f"Statusas: {result['status']}"
    )

    # Išsaugome po kiekvienos operacijos
    pd.DataFrame(results).to_csv(
        folder / "gpt_v4_additional_2000_pilot.csv",
        index=False
    )

    time.sleep(0.2)

pilot_results = pd.DataFrame(results)

print("\nBANDOMOJO VERTINIMO SUVESTINĖ")
print("=" * 60)
print("Operacijų:", len(pilot_results))
print("Sėkmingų:", int((pilot_results["status"] == "OK").sum()))
print("Klaidų:", int((pilot_results["status"] != "OK").sum()))
print("Įvesties tokenų:", int(pilot_results["input_tokens"].sum()))
print("Išvesties tokenų:", int(pilot_results["output_tokens"].sum()))

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import json
import time
import hashlib
from pathlib import Path
from openai import OpenAI

folder = Path("aml_models")

data = pd.read_csv(
    folder / "test_2000_additional_xgb_context.csv"
)

prompt = (folder / "gpt_context_prompt_v4.txt").read_text(
    encoding="utf-8"
)

expected_hash = "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
assert hashlib.sha256(prompt.encode("utf-8")).hexdigest() == expected_hash

features = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

assert len(data) == 2000
assert data["original_id"].is_unique
assert data[features].notna().all().all()

pilot_path = folder / "gpt_v4_additional_2000_pilot.csv"
output_path = folder / "gpt_v4_additional_2000_results.csv"

# Atkuriame anksčiau išsaugotus rezultatus
if output_path.exists():
    results = pd.read_csv(output_path).to_dict("records")
else:
    results = pd.read_csv(pilot_path).to_dict("records")

# Paliekame po vieną sėkmingą rezultatą kiekvienam ID
successful = {}

for result in results:
    if result["status"] == "OK":
        successful[int(result["original_id"])] = result

results = list(successful.values())

completed_ids = set(successful)

print("Jau sėkmingai įvertinta:", len(completed_ids))
print("Liko įvertinti:", len(data) - len(completed_ids))

client = OpenAI(timeout=30.0, max_retries=2)

def save_results():
    pd.DataFrame(results).to_csv(
        output_path,
        index=False
    )

save_results()

for _, row in data.iterrows():

    oid = int(row["original_id"])

    if oid in completed_ids:
        continue

    transaction = {
        col: row[col].item()
        if hasattr(row[col], "item")
        else row[col]
        for col in features
    }

    result = {
        "original_id": oid,
        "true_label": int(row["Is Laundering"]),
        "xgb_probability": float(row["xgb_probability"]),
        "gpt_prediction": None,
        "gpt_reason": "",
        "raw_response": "",
        "status": "ERROR",
        "input_tokens": 0,
        "output_tokens": 0
    }

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=json.dumps(
                transaction,
                ensure_ascii=False
            ),
            temperature=0
        )

        raw = response.output_text.strip()

        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit(
                "```", 1
            )[0].strip()

        parsed = json.loads(raw)
        prediction = parsed["prediction"]

        if type(prediction) is not int or prediction not in (0, 1):
            raise ValueError("Neteisinga prediction reikšmė")

        result["gpt_prediction"] = prediction
        result["gpt_reason"] = str(
            parsed.get("reason", "")
        )
        result["raw_response"] = response.output_text
        result["status"] = "OK"

        if response.usage:
            result["input_tokens"] = response.usage.input_tokens
            result["output_tokens"] = response.usage.output_tokens

    except Exception as e:
        result["raw_response"] = str(e)
        result["status"] = "ERROR"

    results.append(result)

    if result["status"] == "OK":
        completed_ids.add(oid)

    save_results()

    if len(completed_ids) % 50 == 0 or result["status"] != "OK":
        print(
            f"Baigta: {len(completed_ids)}/2000 | "
            f"Statusas: {result['status']}"
        )

    time.sleep(0.2)

final_results = pd.DataFrame(results)

ok = final_results[
    final_results["status"] == "OK"
].drop_duplicates("original_id")

print("\nGALUTINĖ SUVESTINĖ")
print("=" * 60)
print("Sėkmingai įvertinta:", len(ok), "/ 2000")
print("Liko:", 2000 - len(ok))
print("Įvesties tokenų:", int(ok["input_tokens"].sum()))
print("Išvesties tokenų:", int(ok["output_tokens"].sum()))
print("Rezultatai išsaugoti:", output_path)

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

# Ankstesnis 1000 operacijų eksperimentas
old = pd.read_csv(
    folder / "gpt_v4_enriched_1000_merged.csv"
)

# Naujos 2000 operacijų XGBoost prognozės
new_xgb = pd.read_csv(
    folder / "test_2000_additional_xgb_context.csv"
)

# Naujos 2000 operacijų GPT prognozės
new_gpt = pd.read_csv(
    folder / "gpt_v4_additional_2000_results.csv"
)

print("PRADINIAI DUOMENYS")
print("=" * 65)

print("Ankstesnė imtis:", len(old))
print("Nauja XGBoost imtis:", len(new_xgb))
print("Nauji GPT rezultatai:", len(new_gpt))

print("\nSTULPELIŲ PATIKRA")
print("Ankstesnės imties stulpeliai:")
print(old.columns.tolist())

# Užtikriname, kad naujos GPT prognozės yra unikalios
assert new_gpt["original_id"].is_unique
assert len(new_gpt) == 2000
assert (new_gpt["status"] == "OK").all()

# Sujungiame naujas XGBoost ir GPT prognozes
new = new_xgb.merge(
    new_gpt[
        ["original_id", "gpt_prediction", "status"]
    ],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

# Suvienodiname tikrosios klasės stulpelį
if "true_label" in old.columns and "Is Laundering" not in old.columns:
    old = old.rename(columns={"true_label": "Is Laundering"})

required = [
    "original_id",
    "Is Laundering",
    "xgb_probability",
    "xgb_prediction",
    "gpt_prediction"
]

assert all(c in old.columns for c in required)
assert all(c in new.columns for c in required)

# Sujungiame tik analizei reikalingus stulpelius
combined = pd.concat(
    [
        old[required],
        new[required]
    ],
    ignore_index=True
)

# Tikriname duomenų vientisumą
assert len(combined) == 3000
assert combined["original_id"].is_unique
assert combined[required].notna().all().all()
assert int(combined["Is Laundering"].sum()) == 300

for col in [
    "Is Laundering",
    "xgb_prediction",
    "gpt_prediction"
]:
    assert combined[col].isin([0, 1]).all()

# Hibridinių metodų prognozės
combined["or_prediction"] = (
    (combined["xgb_prediction"] == 1) |
    (combined["gpt_prediction"] == 1)
).astype(int)

combined["and_prediction"] = (
    (combined["xgb_prediction"] == 1) &
    (combined["gpt_prediction"] == 1)
).astype(int)

# Išsaugome galutinę imtį
output = folder / "aml_3000_final_comparison.csv"

combined.to_csv(output, index=False)

print("\nGALUTINĖS IMTIES PATIKRA")
print("=" * 65)

print("Operacijų:", len(combined))
print("AML:", int(combined["Is Laundering"].sum()))
print(
    "Teisėtų:",
    int((combined["Is Laundering"] == 0).sum())
)
print(
    "Unikalūs ID:",
    combined["original_id"].is_unique
)
print(
    "Trūkstamų reikšmių:",
    int(combined.isna().sum().sum())
)

print("\nMODELIŲ ĮSPĖJIMŲ SKAIČIUS")
print("=" * 65)

for name, col in [
    ("XGBoost", "xgb_prediction"),
    ("GPT-5.4-mini", "gpt_prediction"),
    ("OR hibridas", "or_prediction"),
    ("AND hibridas", "and_prediction")
]:
    print(f"{name}: {int(combined[col].sum())}")

print("\nRezultatai išsaugoti:", output)

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

folder = Path("aml_models")

datasets = {
    "1000 operacijų": pd.read_csv(
        folder / "gpt_v4_enriched_1000_merged.csv"
    ),
    "3000 operacijų": pd.read_csv(
        folder / "aml_3000_final_comparison.csv"
    )
}

models = {
    "XGBoost": "xgb_prediction",
    "GPT-5.4-mini": "gpt_prediction",
    "OR hibridas": "or_prediction",
    "AND hibridas": "and_prediction"
}

results = []

for sample_name, data in datasets.items():

    y_true = data["Is Laundering"].astype(int)

    for model_name, column in models.items():

        y_pred = data[column].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        results.append({
            "Imtis": sample_name,
            "Modelis": model_name,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "Precision": round(
                precision_score(y_true, y_pred, zero_division=0), 4
            ),
            "Recall": round(
                recall_score(y_true, y_pred), 4
            ),
            "F1": round(
                f1_score(y_true, y_pred), 4
            ),
            "FPR": round(
                fp / (fp + tn), 4
            ),
            "Balanced_accuracy": round(
                balanced_accuracy_score(y_true, y_pred), 4
            ),
            "Alerts": int(y_pred.sum())
        })

comparison = pd.DataFrame(results)

print("GALUTINIS MODELIŲ PALYGINIMAS")
print("=" * 110)
print(comparison.to_string(index=False))

comparison.to_csv(
    folder / "aml_1000_vs_3000_model_comparison.csv",
    index=False
)

print("\n3000 OPERACIJŲ REZULTATAI")
print("=" * 110)

final = comparison[
    comparison["Imtis"] == "3000 operacijų"
].copy()

print(final.drop(columns="Imtis").to_string(index=False))

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

assert len(data) == 3000
assert data["original_id"].is_unique
assert int(data["Is Laundering"].sum()) == 300

budgets = [50, 100, 200, 300, 500]

strategies = {
    "XGBoost": data.sort_values(
        ["xgb_probability", "original_id"],
        ascending=[False, True]
    ),
    "GPT + XGBoost": data.sort_values(
        ["gpt_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True]
    ),
    "OR + XGBoost": data.sort_values(
        ["or_prediction", "xgb_probability", "original_id"],
        ascending=[False, False, True]
    )
}

results = []

for budget in budgets:
    for name, ranked in strategies.items():

        selected = ranked.head(budget)

        tp = int(selected["Is Laundering"].sum())
        fp = budget - tp

        results.append({
            "Biudzetas": budget,
            "Strategija": name,
            "Aptikta_AML": tp,
            "Klaidingi_ispejimai": fp,
            "Precision": round(tp / budget, 4),
            "Recall": round(tp / 300, 4)
        })

comparison = pd.DataFrame(results)

print("VIENODO ĮSPĖJIMŲ BIUDŽETO ANALIZĖ")
print("=" * 90)
print(comparison.to_string(index=False))

pivot = comparison.pivot(
    index="Biudzetas",
    columns="Strategija",
    values="Aptikta_AML"
)

pivot["GPT_minus_XGB"] = (
    pivot["GPT + XGBoost"] - pivot["XGBoost"]
)

pivot["OR_minus_XGB"] = (
    pivot["OR + XGBoost"] - pivot["XGBoost"]
)

print("\nPAPILDOMAI APTIKTI AML, PALYGINTI SU XGBOOST")
print("=" * 75)
print(pivot.to_string())

comparison.to_csv(
    folder / "aml_3000_equal_alert_budget.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

assert len(data) == 3000
assert data["original_id"].is_unique
assert int(data["Is Laundering"].sum()) == 300

aml = data[data["Is Laundering"] == 1].copy()
legit = data[data["Is Laundering"] == 0].copy()

budgets = [50, 100, 200, 300, 500]
n_bootstrap = 2000
rng = np.random.default_rng(42)

def detected_aml(sample, budget, strategy):

    if strategy == "XGBoost":
        ranked = sample.sort_values(
            ["xgb_probability", "original_id"],
            ascending=[False, True],
            kind="stable"
        )
    else:
        ranked = sample.sort_values(
            ["or_prediction", "xgb_probability", "original_id"],
            ascending=[False, False, True],
            kind="stable"
        )

    return int(
        ranked.head(budget)["Is Laundering"].sum()
    )

results = []

for budget in budgets:

    observed = (
        detected_aml(data, budget, "OR")
        - detected_aml(data, budget, "XGBoost")
    )

    differences = []

    for _ in range(n_bootstrap):

        sampled_aml = aml.iloc[
            rng.integers(0, len(aml), len(aml))
        ]

        sampled_legit = legit.iloc[
            rng.integers(0, len(legit), len(legit))
        ]

        sample = pd.concat(
            [sampled_aml, sampled_legit],
            ignore_index=True
        )

        xgb_tp = detected_aml(sample, budget, "XGBoost")
        or_tp = detected_aml(sample, budget, "OR")

        differences.append(or_tp - xgb_tp)

    differences = np.array(differences)

    lower, upper = np.percentile(
        differences, [2.5, 97.5]
    )

    results.append({
        "Biudzetas": budget,
        "Stebetas_OR_minus_XGB": observed,
        "Bootstrap_vidurkis": round(
            differences.mean(), 3
        ),
        "CI_95_nuo": round(lower, 3),
        "CI_95_iki": round(upper, 3),
        "P_OR_geresnis": round(
            (differences > 0).mean(), 4
        )
    })

bootstrap_results = pd.DataFrame(results)

print("3000 OPERACIJŲ BOOTSTRAP ANALIZĖ")
print("=" * 95)
print(bootstrap_results.to_string(index=False))

bootstrap_results.to_csv(
    folder / "aml_3000_budget_bootstrap.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint

folder = Path("aml_models")
data = pd.read_csv(folder / "aml_3000_final_comparison.csv")

models = {
    "XGBoost": "xgb_prediction",
    "GPT-5.4-mini": "gpt_prediction",
    "OR hibridas": "or_prediction",
    "AND hibridas": "and_prediction"
}

# 1. Pasikliautinieji intervalai
ci_results = []

for name, column in models.items():
    y = data["Is Laundering"].astype(int)
    pred = data[column].astype(int)

    tp = int(((y == 1) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())

    for metric, successes, total in [
        ("Recall", tp, tp + fn),
        ("FPR", fp, fp + tn),
        ("Precision", tp, tp + fp)
    ]:
        low, high = proportion_confint(
            successes, total,
            alpha=0.05,
            method="wilson"
        )

        ci_results.append({
            "Modelis": name,
            "Rodiklis": metric,
            "Reiksme": round(successes / total, 4),
            "CI_95_nuo": round(low, 4),
            "CI_95_iki": round(high, 4)
        })

ci_table = pd.DataFrame(ci_results)

print("95 % PASIKLIAUTINIEJI INTERVALAI")
print("=" * 85)
print(ci_table.to_string(index=False))

# 2. Tikslus McNemar testas poriniams sprendimams
pairs = [
    ("XGBoost", "GPT-5.4-mini"),
    ("XGBoost", "OR hibridas"),
    ("GPT-5.4-mini", "OR hibridas"),
    ("XGBoost", "AND hibridas")
]

mcnemar_results = []

for subset_name, target in [
    ("AML operacijos", 1),
    ("Teisėtos operacijos", 0)
]:
    subset = data[data["Is Laundering"] == target]

    for model_a, model_b in pairs:
        a = subset[models[model_a]].astype(int)
        b = subset[models[model_b]].astype(int)

        a_only = int(((a == 1) & (b == 0)).sum())
        b_only = int(((a == 0) & (b == 1)).sum())

        discordant = a_only + b_only

        p_value = (
            binomtest(
                min(a_only, b_only),
                n=discordant,
                p=0.5,
                alternative="two-sided"
            ).pvalue
            if discordant > 0 else 1.0
        )

        mcnemar_results.append({
            "Grupe": subset_name,
            "Modelis_A": model_a,
            "Modelis_B": model_b,
            "Tik_A_pazymejo": a_only,
            "Tik_B_pazymejo": b_only,
            "McNemar_p": p_value
        })

mcnemar_table = pd.DataFrame(mcnemar_results)

print("\nTIKSLŪS McNEMAR TESTAI")
print("=" * 110)
print(mcnemar_table.to_string(index=False))

ci_table.to_csv(
    folder / "aml_3000_confidence_intervals.csv",
    index=False
)

mcnemar_table.to_csv(
    folder / "aml_3000_mcnemar.csv",
    index=False
)

print("\nAbi lentelės išsaugotos.")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix

folder = Path("aml_models")

combined = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

old = pd.read_csv(
    folder / "gpt_v4_enriched_1000_merged.csv"
)

# Tik naujos operacijos
new = combined[
    ~combined["original_id"].isin(old["original_id"])
].copy()

assert len(new) == 2000
assert int(new["Is Laundering"].sum()) == 200
assert new["original_id"].is_unique

models = {
    "XGBoost": "xgb_prediction",
    "GPT-5.4-mini": "gpt_prediction",
    "OR hibridas": "or_prediction",
    "AND hibridas": "and_prediction"
}

results = []

for name, column in models.items():

    tn, fp, fn, tp = confusion_matrix(
        new["Is Laundering"],
        new[column],
        labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    results.append({
        "Modelis": name,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "FPR": round(fp / (fp + tn), 4),
        "Alerts": int(tp + fp)
    })

new_results = pd.DataFrame(results)

print("NAUJŲ 2000 OPERACIJŲ ANALIZĖ")
print("=" * 105)
print(new_results.to_string(index=False))

# Palyginame naują poaibį su ankstesniu
old_results = pd.read_csv(
    folder / "aml_1000_vs_3000_model_comparison.csv"
)

old_results = old_results[
    old_results["Imtis"] == "1000 operacijų"
][["Modelis", "Recall", "FPR", "F1"]]

comparison = old_results.merge(
    new_results[["Modelis", "Recall", "FPR", "F1"]],
    on="Modelis",
    suffixes=("_1000", "_nauji_2000")
)

comparison["Recall_pokytis_pp"] = (
    (comparison["Recall_nauji_2000"] -
     comparison["Recall_1000"]) * 100
).round(2)

comparison["FPR_pokytis_pp"] = (
    (comparison["FPR_nauji_2000"] -
     comparison["FPR_1000"]) * 100
).round(2)

print("\nANKSTESNIŲ 1000 IR NAUJŲ 2000 PALYGINIMAS")
print("=" * 110)
print(comparison.to_string(index=False))

new_results.to_csv(
    folder / "aml_new_2000_model_comparison.csv",
    index=False
)

comparison.to_csv(
    folder / "aml_old_1000_vs_new_2000.csv",
    index=False
)

print("\nAbi lentelės išsaugotos.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

# Natūralus AML paplitimas pilname testavimo rinkinyje
natural_total = 863900
natural_aml = 1611
natural_legit = natural_total - natural_aml

prevalence = natural_aml / natural_total

models = {
    "XGBoost": "xgb_prediction",
    "GPT-5.4-mini": "gpt_prediction",
    "OR hibridas": "or_prediction",
    "AND hibridas": "and_prediction"
}

results = []

for name, column in models.items():

    y = data["Is Laundering"].astype(int)
    pred = data[column].astype(int)

    tp = int(((y == 1) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())

    recall = tp / (tp + fn)
    fpr = fp / (fp + tn)

    # Bayes formulė: Precision pagal tikrąjį paplitimą
    adjusted_precision = (
        recall * prevalence
        /
        (
            recall * prevalence
            + fpr * (1 - prevalence)
        )
    )

    # Projekcija visam natūraliam testavimo rinkiniui
    expected_tp = recall * natural_aml
    expected_fp = fpr * natural_legit
    expected_alerts = expected_tp + expected_fp

    expected_f1 = (
        2 * adjusted_precision * recall
        / (adjusted_precision + recall)
        if adjusted_precision + recall > 0
        else 0
    )

    results.append({
        "Modelis": name,
        "Recall": round(recall, 4),
        "FPR": round(fpr, 4),
        "Precision_10proc": round(
            tp / (tp + fp), 4
        ),
        "Precision_naturalus": round(
            adjusted_precision, 5
        ),
        "F1_naturalus": round(expected_f1, 5),
        "Prognozuojami_TP": round(expected_tp),
        "Prognozuojami_FP": round(expected_fp),
        "Prognozuojami_ispejimai": round(expected_alerts)
    })

results_df = pd.DataFrame(results)

print("NATŪRALAUS AML PAPLITIMO ANALIZĖ")
print("=" * 115)

print(f"Testavimo operacijų: {natural_total:,}")
print(f"AML operacijų: {natural_aml:,}")
print(f"AML paplitimas: {prevalence * 100:.4f}%")

print("\nPAPLITIMUI PRITAIKYTI MODELIŲ RODIKLIAI")
print("=" * 115)
print(results_df.to_string(index=False))

results_df.to_csv(
    folder / "aml_3000_natural_prevalence_adjustment.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")
data = pd.read_csv(folder / "aml_3000_final_comparison.csv")

aml = data[data["Is Laundering"] == 1]
legit = data[data["Is Laundering"] == 0]

prevalence = 1611 / 863900

models = {
    "XGBoost": "xgb_prediction",
    "GPT-5.4-mini": "gpt_prediction",
    "OR hibridas": "or_prediction",
    "AND hibridas": "and_prediction"
}

rng = np.random.default_rng(42)
n_bootstrap = 3000

results = []

for name, column in models.items():

    estimates = []

    for _ in range(n_bootstrap):

        aml_sample = aml.iloc[
            rng.integers(0, len(aml), len(aml))
        ]

        legit_sample = legit.iloc[
            rng.integers(0, len(legit), len(legit))
        ]

        recall = aml_sample[column].mean()
        fpr = legit_sample[column].mean()

        denominator = (
            recall * prevalence
            + fpr * (1 - prevalence)
        )

        ppv = (
            recall * prevalence / denominator
            if denominator > 0 else np.nan
        )

        estimates.append(ppv)

    estimates = np.array(estimates)
    estimates = estimates[np.isfinite(estimates)]

    low, high = np.percentile(estimates, [2.5, 97.5])

    results.append({
        "Modelis": name,
        "PPV_vidurkis_proc": round(
            estimates.mean() * 100, 4
        ),
        "CI_95_nuo_proc": round(low * 100, 4),
        "CI_95_iki_proc": round(high * 100, 4)
    })

bootstrap_ppv = pd.DataFrame(results)

print("NATŪRALAUS PAPLITIMO PRECISION BOOTSTRAP ANALIZĖ")
print("=" * 90)
print(bootstrap_ppv.to_string(index=False))

bootstrap_ppv.to_csv(
    folder / "aml_3000_natural_ppv_bootstrap.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

# Atskiriame AML ir teisėtas operacijas
aml = data[data["Is Laundering"] == 1].copy()
legit = data[data["Is Laundering"] == 0].copy()

def overlap_analysis(subset):
    xgb = subset["xgb_prediction"].astype(int)
    gpt = subset["gpt_prediction"].astype(int)

    return {
        "Abu_pazymejo": int(((xgb == 1) & (gpt == 1)).sum()),
        "Tik_XGBoost": int(((xgb == 1) & (gpt == 0)).sum()),
        "Tik_GPT": int(((xgb == 0) & (gpt == 1)).sum()),
        "Nei_vienas": int(((xgb == 0) & (gpt == 0)).sum())
    }

aml_overlap = overlap_analysis(aml)
legit_overlap = overlap_analysis(legit)

print("AML OPERACIJŲ SPRENDIMŲ SUTAPIMAS")
print("=" * 60)
for key, value in aml_overlap.items():
    print(f"{key}: {value}")

print("\nTEISĖTŲ OPERACIJŲ SPRENDIMŲ SUTAPIMAS")
print("=" * 60)
for key, value in legit_overlap.items():
    print(f"{key}: {value}")

# Sukuriame sprendimų kategorijas
conditions = [
    (data["xgb_prediction"] == 1) &
    (data["gpt_prediction"] == 1),

    (data["xgb_prediction"] == 1) &
    (data["gpt_prediction"] == 0),

    (data["xgb_prediction"] == 0) &
    (data["gpt_prediction"] == 1)
]

import numpy as np

data["decision_group"] = np.select(
    conditions,
    ["Abu", "Tik_XGBoost", "Tik_GPT"],
    default="Nei_vienas"
)

summary = data.groupby(
    ["Is Laundering", "decision_group"]
).agg(
    Operacijos=("original_id", "count"),
    Vidutine_XGB_tikimybe=("xgb_probability", "mean"),
    Mediana_XGB_tikimybe=("xgb_probability", "median")
).reset_index()

print("\nSPRENDIMŲ GRUPIŲ CHARAKTERISTIKOS")
print("=" * 100)
print(summary.to_string(index=False))

summary.to_csv(
    folder / "aml_3000_decision_overlap.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")
data = pd.read_csv(folder / "aml_3000_final_comparison.csv")

# XGBoost tikimybės intervalai
bins = [-0.001, 0.01, 0.1, 0.5, 0.8, 0.9, 0.95, 1.000001]

labels = [
    "0–1 %",
    "1–10 %",
    "10–50 %",
    "50–80 %",
    "80–90 %",
    "90–95 %",
    "95–100 %"
]

data["xgb_risk_group"] = pd.cut(
    data["xgb_probability"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# GPT pažymėtų ir nepažymėtų operacijų palyginimas
summary = (
    data.groupby(
        ["xgb_risk_group", "gpt_prediction"],
        observed=True
    )
    .agg(
        Operacijos=("original_id", "count"),
        AML=("Is Laundering", "sum"),
        Vidutine_XGB_tikimybe=("xgb_probability", "mean")
    )
    .reset_index()
)

summary["AML_dalis_proc"] = (
    summary["AML"] / summary["Operacijos"] * 100
).round(2)

summary["Vidutine_XGB_tikimybe"] = (
    summary["Vidutine_XGB_tikimybe"].round(4)
)

summary["GPT_sprendimas"] = summary[
    "gpt_prediction"
].map({
    0: "Nepazymejo",
    1: "Pazymejo"
})

print("GPT PAPILDOMOS INFORMACIJOS ANALIZĖ")
print("=" * 110)

print(
    summary[
        [
            "xgb_risk_group",
            "GPT_sprendimas",
            "Operacijos",
            "AML",
            "AML_dalis_proc",
            "Vidutine_XGB_tikimybe"
        ]
    ].to_string(index=False)
)

# Palyginame AML dalį tarp GPT grupių
pivot = summary.pivot(
    index="xgb_risk_group",
    columns="gpt_prediction",
    values="AML_dalis_proc"
)

pivot = pivot.rename(columns={
    0: "GPT_nepazymejo_AML_proc",
    1: "GPT_pazymejo_AML_proc"
})

pivot["Skirtumas_proc_punktais"] = (
    pivot["GPT_pazymejo_AML_proc"]
    - pivot["GPT_nepazymejo_AML_proc"]
).round(2)

print("\nAML DALIES SKIRTUMAS ESANT PANAŠIAI XGBOOST RIZIKAI")
print("=" * 100)
print(pivot.to_string())

summary.to_csv(
    folder / "aml_3000_gpt_conditional_value.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

assert len(data) == 3000
assert data["original_id"].is_unique

y = data["Is Laundering"].astype(int)

# Tikslus XGBoost rizikos balas
xgb_score = data["xgb_probability"].astype(float)

# Pirmasis modelis: tik XGBoost balas
X1 = sm.add_constant(
    pd.DataFrame({
        "xgb_score": xgb_score
    }),
    has_constant="add"
)

# Antrasis modelis: XGBoost + GPT
X2 = sm.add_constant(
    pd.DataFrame({
        "xgb_score": xgb_score,
        "gpt_prediction": data["gpt_prediction"].astype(int)
    }),
    has_constant="add"
)

model1 = sm.GLM(
    y, X1, family=sm.families.Binomial()
).fit()

model2 = sm.GLM(
    y, X2, family=sm.families.Binomial()
).fit()

# Tikėtinumo santykio testas
lr_stat = 2 * (model2.llf - model1.llf)
df_diff = int(model2.df_model - model1.df_model)
p_value = chi2.sf(lr_stat, df_diff)

# GPT koeficiento interpretacija
coef = model2.params["gpt_prediction"]
ci_low, ci_high = model2.conf_int().loc["gpt_prediction"]

odds_ratio = np.exp(coef)
or_low = np.exp(ci_low)
or_high = np.exp(ci_high)

print("GPT PAPILDOMOS INFORMACIJOS STATISTINIS TESTAS")
print("=" * 75)

print(f"1 modelio log-likelihood: {model1.llf:.4f}")
print(f"2 modelio log-likelihood: {model2.llf:.4f}")

print(f"\nLikelihood-ratio statistika: {lr_stat:.4f}")
print(f"Laisvės laipsniai: {df_diff}")
print(f"p reikšmė: {p_value:.8g}")

print("\nGPT SPRENDIMO KOEFICIENTAS")
print("=" * 75)

print(f"Koeficientas: {coef:.4f}")
print(f"Odds ratio: {odds_ratio:.4f}")
print(f"95 % OR intervalas: [{or_low:.4f}; {or_high:.4f}]")

print("\nANTROJO MODELIO SANTRAUKA")
print(model2.summary())

results = pd.DataFrame([{
    "LL_XGB": model1.llf,
    "LL_XGB_GPT": model2.llf,
    "LR_statistic": lr_stat,
    "df": df_diff,
    "p_value": p_value,
    "GPT_coefficient": coef,
    "GPT_odds_ratio": odds_ratio,
    "GPT_OR_CI_low": or_low,
    "GPT_OR_CI_high": or_high
}])

results.to_csv(
    folder / "aml_3000_gpt_incremental_value_test.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrix
from scipy.stats import chi2
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

assert len(data) == 3000
assert data["original_id"].is_unique

y = data["Is Laundering"].astype(int).reset_index(drop=True)

# Lankstus XGBoost balo modeliavimas
spline = dmatrix(
    "bs(score, df=5, degree=3, include_intercept=False)",
    {"score": data["xgb_probability"].to_numpy()},
    return_type="dataframe"
).reset_index(drop=True)

# Modelis 1: tik XGBoost balas (netiesinis ryšys)
X1 = spline.copy()

# Modelis 2: XGBoost balas + GPT sprendimas
X2 = spline.copy()
X2["gpt_prediction"] = (
    data["gpt_prediction"].astype(int).to_numpy()
)

model1 = sm.GLM(
    y, X1, family=sm.families.Binomial()
).fit()

model2 = sm.GLM(
    y, X2, family=sm.families.Binomial()
).fit()

lr_stat = 2 * (model2.llf - model1.llf)
p_value = chi2.sf(lr_stat, 1)

coef = model2.params["gpt_prediction"]
ci_low, ci_high = model2.conf_int().loc["gpt_prediction"]

print("NETIESINIO MODELIO ANALIZĖ")
print("=" * 75)

print(f"Modelis 1 (XGB splainai): {model1.llf:.4f}")
print(f"Modelis 2 (XGB splainai + GPT): {model2.llf:.4f}")

print(f"\nLR statistika: {lr_stat:.4f}")
print(f"p reikšmė: {p_value:.8g}")

print("\nGPT PAPILDOMAS POVEIKIS")
print("=" * 75)

print(f"GPT koeficientas: {coef:.4f}")
print(f"Šansų santykis: {np.exp(coef):.4f}")
print(
    f"95 % PI: "
    f"[{np.exp(ci_low):.4f}; {np.exp(ci_high):.4f}]"
)

print("\nMODELIŲ AIC")
print(f"XGB splainai: {model1.aic:.4f}")
print(f"XGB splainai + GPT: {model2.aic:.4f}")

results = pd.DataFrame([{
    "LL_XGB_spline": model1.llf,
    "LL_XGB_spline_GPT": model2.llf,
    "LR_statistic": lr_stat,
    "p_value": p_value,
    "GPT_coefficient": coef,
    "GPT_odds_ratio": np.exp(coef),
    "GPT_OR_CI_low": np.exp(ci_low),
    "GPT_OR_CI_high": np.exp(ci_high),
    "AIC_XGB_spline": model1.aic,
    "AIC_XGB_spline_GPT": model2.aic
}])

results.to_csv(
    folder / "aml_3000_gpt_nonlinear_incremental_value.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.special import logit
from scipy.stats import chi2
from pathlib import Path

folder = Path("aml_models")
data = pd.read_csv(folder / "aml_3000_final_comparison.csv")

y = data["Is Laundering"].astype(int).to_numpy()

# Apsauga nuo logit(0) ir logit(1)
p = data["xgb_probability"].astype(float).to_numpy()
p = np.clip(p, 1e-6, 1 - 1e-6)

# Transformuojame ir standartizuojame balą
z = logit(p)
z = (z - z.mean()) / z.std()

gpt = data["gpt_prediction"].astype(int).to_numpy()

# Bazinis netiesinis modelis
X1 = pd.DataFrame({
    "const": np.ones(len(data)),
    "xgb_logit": z,
    "xgb_logit_squared": z**2
})

# Papildomas GPT požymis
X2 = X1.copy()
X2["gpt_prediction"] = gpt

model1 = sm.GLM(
    y, X1,
    family=sm.families.Binomial()
).fit(maxiter=200)

model2 = sm.GLM(
    y, X2,
    family=sm.families.Binomial()
).fit(maxiter=200)

# Privaloma rezultatų patikra
assert model1.converged, "Bazinis modelis nekonvergavo"
assert model2.converged, "GPT modelis nekonvergavo"

assert np.isfinite(model1.llf)
assert np.isfinite(model2.llf)

assert np.isfinite(model1.params).all()
assert np.isfinite(model2.params).all()

# Tikėtinumo santykio testas
lr_stat = 2 * (model2.llf - model1.llf)
p_value = chi2.sf(max(lr_stat, 0), 1)

coef = model2.params["gpt_prediction"]
ci_low, ci_high = model2.conf_int().loc["gpt_prediction"]

print("PATAISYTA NETIESINĖ LOGISTINĖ REGRESIJA")
print("=" * 75)

print(f"Bazinio modelio LL: {model1.llf:.4f}")
print(f"Modelio su GPT LL: {model2.llf:.4f}")

print(f"\nLR statistika: {lr_stat:.4f}")
print(f"p reikšmė: {p_value:.8g}")

print("\nGPT PAPILDOMOS INFORMACIJOS ĮVERTINIMAS")
print("=" * 75)

print(f"GPT koeficientas: {coef:.4f}")
print(f"Šansų santykis: {np.exp(coef):.4f}")
print(
    f"95 % PI: "
    f"[{np.exp(ci_low):.4f}; {np.exp(ci_high):.4f}]"
)

print("\nMODELIŲ PALYGINIMAS")
print(f"Bazinio modelio AIC: {model1.aic:.4f}")
print(f"Modelio su GPT AIC: {model2.aic:.4f}")

print("\nKonvergencija:")
print("Bazinis modelis:", model1.converged)
print("Modelis su GPT:", model2.converged)

results = pd.DataFrame([{
    "LL_base": model1.llf,
    "LL_with_GPT": model2.llf,
    "LR_statistic": lr_stat,
    "p_value": p_value,
    "GPT_coefficient": coef,
    "GPT_odds_ratio": np.exp(coef),
    "GPT_OR_CI_low": np.exp(ci_low),
    "GPT_OR_CI_high": np.exp(ci_high),
    "AIC_base": model1.aic,
    "AIC_with_GPT": model2.aic
}])

results.to_csv(
    folder / "aml_3000_gpt_nonlinear_corrected.csv",
    index=False
)

print("\nRezultatai sėkmingai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss
)

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

old = pd.read_csv(
    folder / "gpt_v4_enriched_1000_merged.csv"
)

train = data[
    data["original_id"].isin(old["original_id"])
].copy()

test = data[
    ~data["original_id"].isin(old["original_id"])
].copy()

assert len(train) == 1000
assert len(test) == 2000
assert train["original_id"].is_unique
assert test["original_id"].is_unique
assert set(train["original_id"]).isdisjoint(test["original_id"])
assert int(train["Is Laundering"].sum()) == 100
assert int(test["Is Laundering"].sum()) == 200

def make_features(frame, with_gpt=False):

    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )

    z = logit(p)

    X = pd.DataFrame({
        "xgb_logit": z,
        "xgb_logit_squared": z**2
    })

    if with_gpt:
        X["gpt_prediction"] = (
            frame["gpt_prediction"].to_numpy(dtype=int)
        )

    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)

results = []

for name, use_gpt in [
    ("XGB kalibravimas", False),
    ("XGB + GPT", True)
]:

    X_train = make_features(train, use_gpt)
    X_test = make_features(test, use_gpt)

    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Modelis": name,
        "ROC_AUC": roc_auc_score(
            y_test, probabilities
        ),
        "Average_Precision": average_precision_score(
            y_test, probabilities
        ),
        "Log_loss": log_loss(
            y_test, probabilities
        ),
        "Brier_score": brier_score_loss(
            y_test, probabilities
        )
    })

comparison = pd.DataFrame(results)

print("GPT PAPILDOMOS VERTĖS PATIKRINIMAS")
print("=" * 85)
print("Mokymo operacijos:", len(train))
print("Tikrinimo operacijos:", len(test))
print()
print(comparison.round(6).to_string(index=False))

comparison.to_csv(
    folder / "aml_gpt_incremental_holdout_2000.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss
)

folder = Path("aml_models")

data = pd.read_csv(folder / "aml_3000_final_comparison.csv")
old = pd.read_csv(folder / "gpt_v4_enriched_1000_merged.csv")

train = data[data["original_id"].isin(old["original_id"])].copy()
test = data[~data["original_id"].isin(old["original_id"])].copy()

assert len(train) == 1000 and len(test) == 2000
assert set(train["original_id"]).isdisjoint(test["original_id"])

def features(frame, with_gpt=False):
    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )
    z = logit(p)
    X = np.column_stack([z, z**2])

    if with_gpt:
        X = np.column_stack([
            X,
            frame["gpt_prediction"].to_numpy(dtype=int)
        ])
    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)

predictions = {}

for name, with_gpt in [
    ("XGB", False),
    ("Hybrid", True)
]:
    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )
    model.fit(features(train, with_gpt), y_train)
    predictions[name] = model.predict_proba(
        features(test, with_gpt)
    )[:, 1]

def metrics(y, p):
    return np.array([
        roc_auc_score(y, p),
        average_precision_score(y, p),
        log_loss(y, p, labels=[0, 1]),
        brier_score_loss(y, p)
    ])

names = ["ROC_AUC", "Average_Precision", "Log_loss", "Brier_score"]

observed = (
    metrics(y_test, predictions["Hybrid"])
    - metrics(y_test, predictions["XGB"])
)

# Porinis stratifikuotas bootstrap
rng = np.random.default_rng(42)

positive_idx = np.where(y_test == 1)[0]
negative_idx = np.where(y_test == 0)[0]

B = 2000
differences = np.empty((B, 4))

for b in range(B):
    sample = np.concatenate([
        rng.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        ),
        rng.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )
    ])

    y_b = y_test[sample]

    differences[b] = (
        metrics(y_b, predictions["Hybrid"][sample])
        - metrics(y_b, predictions["XGB"][sample])
    )

rows = []

for i, name in enumerate(names):
    low, high = np.percentile(
        differences[:, i], [2.5, 97.5]
    )

    rows.append({
        "Rodiklis": name,
        "Stebetas_skirtumas": observed[i],
        "CI_95_low": low,
        "CI_95_high": high,
        "Intervalas_apima_0": low <= 0 <= high
    })

results = pd.DataFrame(rows)

print("HIBRIDINIO MODELIO PAGERĖJIMO BOOTSTRAP ANALIZĖ")
print("=" * 100)
print(results.round(6).to_string(index=False))

results.to_csv(
    folder / "aml_hybrid_2000_bootstrap_improvement.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression

folder = Path("aml_models")

data = pd.read_csv(folder / "aml_3000_final_comparison.csv")
old = pd.read_csv(folder / "gpt_v4_enriched_1000_merged.csv")

train = data[data["original_id"].isin(old["original_id"])].copy()
test = data[~data["original_id"].isin(old["original_id"])].copy()

assert len(train) == 1000
assert len(test) == 2000

def make_features(frame, with_gpt=False):
    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )
    z = logit(p)
    X = np.column_stack([z, z**2])

    if with_gpt:
        X = np.column_stack([
            X,
            frame["gpt_prediction"].to_numpy(dtype=int)
        ])

    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)

scores = {}

for name, with_gpt in [
    ("XGBoost", False),
    ("XGBoost_GPT", True)
]:
    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )

    model.fit(
        make_features(train, with_gpt),
        y_train
    )

    scores[name] = model.predict_proba(
        make_features(test, with_gpt)
    )[:, 1]

budgets = [25, 50, 100, 150, 200]
rows = []

for k in budgets:

    xgb_top = np.argsort(-scores["XGBoost"])[:k]
    hybrid_top = np.argsort(-scores["XGBoost_GPT"])[:k]

    xgb_tp = int(y_test[xgb_top].sum())
    hybrid_tp = int(y_test[hybrid_top].sum())

    rows.append({
        "Tikrinimu_biudzetas": k,
        "XGBoost_AML": xgb_tp,
        "Hibridinis_AML": hybrid_tp,
        "Papildomai_aptikta_AML": hybrid_tp - xgb_tp,
        "XGBoost_precision_at_K": xgb_tp / k,
        "Hibridinis_precision_at_K": hybrid_tp / k
    })

results = pd.DataFrame(rows)

print("AML APTIKIMAS ESANT VIENODAM TIKRINIMŲ BIUDŽETUI")
print("=" * 110)
print(results.round(4).to_string(index=False))

results.to_csv(
    folder / "aml_hybrid_logistic_2000_alert_budget.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression

folder = Path("aml_models")

data = pd.read_csv(folder / "aml_3000_final_comparison.csv")
old = pd.read_csv(folder / "gpt_v4_enriched_1000_merged.csv")

train = data[data["original_id"].isin(old["original_id"])].copy()
test = data[~data["original_id"].isin(old["original_id"])].copy()

assert len(train) == 1000
assert len(test) == 2000
assert train["original_id"].is_unique
assert test["original_id"].is_unique
assert set(train["original_id"]).isdisjoint(test["original_id"])

def features(frame, with_gpt=False):
    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )
    z = logit(p)
    X = np.column_stack([z, z**2])

    if with_gpt:
        X = np.column_stack([
            X,
            frame["gpt_prediction"].to_numpy(dtype=int)
        ])
    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)

scores = {}

for name, with_gpt in [
    ("XGBoost", False),
    ("Hybrid", True)
]:
    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )
    model.fit(features(train, with_gpt), y_train)
    scores[name] = model.predict_proba(
        features(test, with_gpt)
    )[:, 1]

budgets = [25, 50, 100, 150, 200]
B = 2000
rng = np.random.default_rng(42)

positive_idx = np.where(y_test == 1)[0]
negative_idx = np.where(y_test == 0)[0]

def detected_aml(y, score, k):
    top = np.argsort(-score, kind="stable")[:k]
    return int(y[top].sum())

rows = []

for k in budgets:
    observed_xgb = detected_aml(y_test, scores["XGBoost"], k)
    observed_hybrid = detected_aml(y_test, scores["Hybrid"], k)

    differences = []

    for _ in range(B):
        sample = np.concatenate([
            rng.choice(
                positive_idx,
                size=len(positive_idx),
                replace=True
            ),
            rng.choice(
                negative_idx,
                size=len(negative_idx),
                replace=True
            )
        ])

        xgb_tp = detected_aml(
            y_test[sample],
            scores["XGBoost"][sample],
            k
        )

        hybrid_tp = detected_aml(
            y_test[sample],
            scores["Hybrid"][sample],
            k
        )

        differences.append(hybrid_tp - xgb_tp)

    differences = np.array(differences)
    low, high = np.percentile(differences, [2.5, 97.5])

    rows.append({
        "Biudzetas": k,
        "XGB_AML": observed_xgb,
        "Hybrid_AML": observed_hybrid,
        "Stebetas_skirtumas": observed_hybrid - observed_xgb,
        "Bootstrap_vidurkis": differences.mean(),
        "CI_95_low": low,
        "CI_95_high": high,
        "P_Hybrid_geresnis": (differences > 0).mean()
    })

results = pd.DataFrame(rows)

print("HIBRIDINIO MODELIO TIKRINIMŲ BIUDŽETO BOOTSTRAP")
print("=" * 115)
print(results.round(4).to_string(index=False))

results.to_csv(
    folder / "aml_hybrid_logistic_2000_budget_bootstrap.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression

folder = Path("aml_models")

data = pd.read_csv(folder / "aml_3000_final_comparison.csv")
old = pd.read_csv(folder / "gpt_v4_enriched_1000_merged.csv")

train = data[data["original_id"].isin(old["original_id"])].copy()
test = data[~data["original_id"].isin(old["original_id"])].copy()

assert len(train) == 1000 and len(test) == 2000

def features(frame, with_gpt=False):
    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )
    z = logit(p)
    X = np.column_stack([z, z**2])

    if with_gpt:
        X = np.column_stack([
            X,
            frame["gpt_prediction"].to_numpy(dtype=int)
        ])
    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)
ids = test["original_id"].to_numpy()

scores = {}

for name, with_gpt in [
    ("XGBoost", False),
    ("Hybrid", True)
]:
    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )
    model.fit(features(train, with_gpt), y_train)
    scores[name] = model.predict_proba(
        features(test, with_gpt)
    )[:, 1]

print("VIENODŲ RIZIKOS BALŲ DIAGNOSTIKA")
print("=" * 95)

for name, score in scores.items():
    unique_scores, counts = np.unique(
        score, return_counts=True
    )

    print(f"\n{name}")
    print("Operacijų:", len(score))
    print("Unikalių balų:", len(unique_scores))
    print("Didžiausia vienodų balų grupė:", counts.max())

print("\nRIKIAVIMO METODŲ PALYGINIMAS")
print("=" * 115)

rows = []

for k in [25, 50, 100, 150, 200]:
    for name, score in scores.items():

        # Ankstesnis numatytasis rikiavimas
        order_default = np.argsort(-score)

        # Stabilus rikiavimas
        order_stable = np.argsort(
            -score, kind="stable"
        )

        # Deterministinis rikiavimas:
        # pirmiausia balas, tada original_id
        order_id = np.lexsort((ids, -score))

        boundary_score = score[order_id[k - 1]]

        # Kiek operacijų turi tokį pat balą
        # kaip paskutinė atrinkta operacija?
        boundary_ties = int(
            np.sum(score == boundary_score)
        )

        rows.append({
            "Biudzetas": k,
            "Modelis": name,
            "AML_default": int(
                y_test[order_default[:k]].sum()
            ),
            "AML_stable": int(
                y_test[order_stable[:k]].sum()
            ),
            "AML_original_id": int(
                y_test[order_id[:k]].sum()
            ),
            "Vienodu_balu_ties_riba": boundary_ties,
            "Riba_kerta_vienodus_balus": bool(
                np.sum(score > boundary_score) < k
                < np.sum(score >= boundary_score)
            )
        })

results = pd.DataFrame(rows)

print(results.to_string(index=False))

results.to_csv(
    folder / "aml_hybrid_ranking_tie_diagnostics.csv",
    index=False
)

print("\nDiagnostikos rezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.special import logit
from sklearn.linear_model import LogisticRegression

folder = Path("aml_models")

data = pd.read_csv(folder / "aml_3000_final_comparison.csv")
old = pd.read_csv(folder / "gpt_v4_enriched_1000_merged.csv")

train = data[data["original_id"].isin(old["original_id"])].copy()
test = data[~data["original_id"].isin(old["original_id"])].copy()

assert len(train) == 1000
assert len(test) == 2000
assert train["original_id"].is_unique
assert test["original_id"].is_unique
assert set(train["original_id"]).isdisjoint(test["original_id"])

def features(frame, with_gpt=False):
    p = np.clip(
        frame["xgb_probability"].to_numpy(dtype=float),
        1e-6, 1 - 1e-6
    )
    z = logit(p)
    X = np.column_stack([z, z**2])

    if with_gpt:
        X = np.column_stack([
            X,
            frame["gpt_prediction"].to_numpy(dtype=int)
        ])
    return X

y_train = train["Is Laundering"].to_numpy(dtype=int)
y_test = test["Is Laundering"].to_numpy(dtype=int)
ids = test["original_id"].to_numpy()

scores = {}

for name, with_gpt in [
    ("XGBoost", False),
    ("Hybrid", True)
]:
    model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )
    model.fit(features(train, with_gpt), y_train)
    scores[name] = model.predict_proba(
        features(test, with_gpt)
    )[:, 1]

def detected_aml(y, score, original_ids, k):
    # Pirma: didesnis balas
    # Antra: mažesnis original_id
    # Trečia: pozicija imtyje
    order = np.lexsort((
        np.arange(len(y)),
        original_ids,
        -score
    ))
    return int(y[order[:k]].sum())

budgets = [25, 50, 100, 150, 200]
B = 2000
rng = np.random.default_rng(42)

positive_idx = np.where(y_test == 1)[0]
negative_idx = np.where(y_test == 0)[0]

observed = {}
bootstrap_diffs = {k: np.empty(B, dtype=int) for k in budgets}

for k in budgets:
    xgb_tp = detected_aml(
        y_test, scores["XGBoost"], ids, k
    )
    hybrid_tp = detected_aml(
        y_test, scores["Hybrid"], ids, k
    )
    observed[k] = (xgb_tp, hybrid_tp)

for b in range(B):
    sample = np.concatenate([
        rng.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        ),
        rng.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )
    ])

    y_b = y_test[sample]
    ids_b = ids[sample]

    for k in budgets:
        xgb_tp = detected_aml(
            y_b,
            scores["XGBoost"][sample],
            ids_b,
            k
        )

        hybrid_tp = detected_aml(
            y_b,
            scores["Hybrid"][sample],
            ids_b,
            k
        )

        bootstrap_diffs[k][b] = hybrid_tp - xgb_tp

rows = []

for k in budgets:
    xgb_tp, hybrid_tp = observed[k]
    diffs = bootstrap_diffs[k]

    low, high = np.percentile(diffs, [2.5, 97.5])

    rows.append({
        "Biudzetas": k,
        "XGB_AML": xgb_tp,
        "Hybrid_AML": hybrid_tp,
        "Skirtumas": hybrid_tp - xgb_tp,
        "Bootstrap_vidurkis": diffs.mean(),
        "CI_95_low": low,
        "CI_95_high": high,
        "P_Hybrid_geresnis": (diffs > 0).mean()
    })

results = pd.DataFrame(rows)

print("GALUTINĖ DETERMINISTINĖ BOOTSTRAP ANALIZĖ")
print("=" * 115)
print(results.round(4).to_string(index=False))

results.to_csv(
    folder / "aml_hybrid_2000_budget_bootstrap_final.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

comparison = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

old = pd.read_csv(
    folder / "gpt_v4_enriched_1000_merged.csv"
)

new = pd.read_csv(
    folder / "test_2000_additional_xgb_context.csv"
)

context = pd.concat(
    [old, new],
    ignore_index=True
)

assert context["original_id"].is_unique
assert comparison["original_id"].is_unique

# Pridedame tik kontekstinius požymius
# ir neįtraukiame modelių sprendimų bei tikrosios klasės
history_cols = [
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count"
]

transaction_cols = [
    "Amount Received",
    "Amount Paid",
    "Payment Format",
    "Payment Currency",
    "Receiving Currency"
]

required = ["original_id"] + history_cols + transaction_cols

missing = [c for c in required if c not in context.columns]

if missing:
    raise ValueError(f"Trūksta stulpelių: {missing}")

analysis = comparison.merge(
    context[required],
    on="original_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("KLAIDŲ ANALIZĖS DUOMENŲ PATIKRA")
print("=" * 75)

print("Operacijų:", len(analysis))
print("AML operacijų:", int(analysis["Is Laundering"].sum()))
print("Unikalių ID:", analysis["original_id"].nunique())

print("\nSujungimo rezultatai:")
print(analysis["_merge"].value_counts().to_string())

print("\nTrūkstamos reikšmės:")
print(
    analysis[history_cols + transaction_cols]
    .isna()
    .sum()
    .to_string()
)

assert len(analysis) == 3000
assert (analysis["_merge"] == "both").all()
assert not analysis[history_cols + transaction_cols].isna().any().any()

analysis = analysis.drop(columns="_merge")

analysis.to_csv(
    folder / "aml_3000_error_analysis_dataset.csv",
    index=False
)

print("\nDuomenys sėkmingai parengti ir išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_error_analysis_dataset.csv"
)

# Analizuojame tik tikras AML operacijas
aml = data[data["Is Laundering"] == 1].copy()

conditions = [
    (aml["xgb_prediction"] == 1) &
    (aml["gpt_prediction"] == 1),

    (aml["xgb_prediction"] == 1) &
    (aml["gpt_prediction"] == 0),

    (aml["xgb_prediction"] == 0) &
    (aml["gpt_prediction"] == 1)
]

labels = [
    "Abu_modeliai",
    "Tik_XGBoost",
    "Tik_GPT"
]

aml["aptikimo_grupe"] = np.select(
    conditions,
    labels,
    default="Nei_vienas"
)

features = [
    "Amount Paid",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count",
    "xgb_probability"
]

# Medianų ir 90 procentilio palyginimas
summary = (
    aml.groupby("aptikimo_grupe")[features]
    .agg(["median", lambda x: x.quantile(0.90)])
)

summary.columns = [
    f"{feature}_{'mediana' if stat == 'median' else 'p90'}"
    for feature, stat in summary.columns
]

summary = summary.round(2)

counts = (
    aml["aptikimo_grupe"]
    .value_counts()
    .rename("Operaciju_skaicius")
)

summary = summary.join(counts)

print("AML OPERACIJŲ CHARAKTERISTIKOS")
print("=" * 110)

print(summary.T.to_string())

# Mokėjimo formatų pasiskirstymas
formats = pd.crosstab(
    aml["aptikimo_grupe"],
    aml["Payment Format"],
    normalize="index"
) * 100

formats = formats.round(2)

print("\nMOKĖJIMO FORMATŲ PASISKIRSTYMAS (%)")
print("=" * 100)
print(formats.to_string())

# Išsaugome
summary.to_csv(
    folder / "aml_3000_detection_group_characteristics.csv"
)

formats.to_csv(
    folder / "aml_3000_detection_group_payment_formats.csv"
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_error_analysis_dataset.csv"
)

# GPT pažymėtos operacijos
alerts = data[data["gpt_prediction"] == 1].copy()

alerts["grupe"] = np.where(
    alerts["Is Laundering"] == 1,
    "Tikras_AML",
    "Klaidingas_ispejimas"
)

features = [
    "Amount Paid",
    "previous_24h_count",
    "previous_7d_count",
    "previous_24h_amount",
    "previous_7d_unique_recipients",
    "same_recipient_7d_count",
    "xgb_probability"
]

summary = alerts.groupby("grupe")[features].agg(
    ["median", lambda x: x.quantile(0.90)]
)

summary.columns = [
    f"{feature}_{'mediana' if stat == 'median' else 'p90'}"
    for feature, stat in summary.columns
]

summary["Operaciju_skaicius"] = (
    alerts.groupby("grupe").size()
)

print("GPT TIKRŲ IR KLAIDINGŲ ĮSPĖJIMŲ PALYGINIMAS")
print("=" * 105)
print(summary.round(2).T.to_string())

# Mokėjimo formatų pasiskirstymas
formats = pd.crosstab(
    alerts["grupe"],
    alerts["Payment Format"],
    normalize="index"
) * 100

print("\nMOKĖJIMO FORMATŲ PASISKIRSTYMAS (%)")
print("=" * 105)
print(formats.round(2).to_string())

# AML dalis pagal ankstesnį aktyvumą
alerts["aktyvumo_grupe"] = pd.cut(
    alerts["previous_7d_count"],
    bins=[-1, 0, 5, 20, 100, np.inf],
    labels=["0", "1–5", "6–20", "21–100", ">100"]
)

activity = alerts.groupby(
    "aktyvumo_grupe",
    observed=True
).agg(
    Operacijos=("original_id", "count"),
    AML=("Is Laundering", "sum")
)

activity["AML_dalis_proc"] = (
    activity["AML"] / activity["Operacijos"] * 100
).round(2)

print("\nAML DALIS TARP GPT ĮSPĖJIMŲ PAGAL SIUNTĖJO AKTYVUMĄ")
print("=" * 105)
print(activity.to_string())

summary.to_csv(
    folder / "aml_gpt_true_vs_false_alerts.csv"
)

formats.to_csv(
    folder / "aml_gpt_alert_payment_formats.csv"
)

activity.to_csv(
    folder / "aml_gpt_alert_activity_analysis.csv"
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_error_analysis_dataset.csv"
)

data["aktyvumo_grupe"] = pd.cut(
    data["previous_7d_count"],
    bins=[-1, 0, 5, 20, 100, np.inf],
    labels=["0", "1–5", "6–20", "21–100", ">100"]
)

rows = []

for activity, group in data.groupby(
    "aktyvumo_grupe",
    observed=True
):
    y = group["Is Laundering"].to_numpy(dtype=int)

    for model_name, prediction_col in [
        ("XGBoost", "xgb_prediction"),
        ("GPT-5.4-mini", "gpt_prediction")
    ]:
        pred = group[prediction_col].to_numpy(dtype=int)

        tp = int(((y == 1) & (pred == 1)).sum())
        fp = int(((y == 0) & (pred == 1)).sum())
        fn = int(((y == 1) & (pred == 0)).sum())
        tn = int(((y == 0) & (pred == 0)).sum())

        rows.append({
            "Aktyvumas_7d": str(activity),
            "Modelis": model_name,
            "Operacijos": len(group),
            "AML": tp + fn,
            "Teisetos": fp + tn,
            "TP": tp,
            "FP": fp,
            "Recall_proc": (
                100 * tp / (tp + fn)
                if tp + fn else np.nan
            ),
            "FPR_proc": (
                100 * fp / (fp + tn)
                if fp + tn else np.nan
            ),
            "Precision_proc": (
                100 * tp / (tp + fp)
                if tp + fp else np.nan
            )
        })

results = pd.DataFrame(rows)

print("MODELIŲ REZULTATAI PAGAL SIUNTĖJO AKTYVUMĄ")
print("=" * 120)

print(
    results.round(2).to_string(index=False)
)

results.to_csv(
    folder / "aml_3000_model_performance_by_activity.csv",
    index=False
)

print("\nRezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_error_analysis_dataset.csv"
)

data["aktyvumo_grupe"] = pd.cut(
    data["previous_7d_count"],
    bins=[-1, 0, 5, 20, 100, float("inf")],
    labels=["0", "1–5", "6–20", "21–100", ">100"]
)

samples = []

for activity in data["aktyvumo_grupe"].cat.categories:
    subset = data[data["aktyvumo_grupe"] == activity]

    for label in [0, 1]:
        selected = subset[
            subset["Is Laundering"] == label
        ].sample(
            n=10,
            random_state=42
        )
        samples.append(selected)

pilot = pd.concat(samples, ignore_index=True)

assert len(pilot) == 100
assert pilot["original_id"].is_unique

print("ISTORINIŲ POŽYMIŲ POVEIKIO EKSPERIMENTO IMTIS")
print("=" * 75)

print(
    pd.crosstab(
        pilot["aktyvumo_grupe"],
        pilot["Is Laundering"]
    ).to_string()
)

print("\nAnkstesni GPT sprendimai:")
print(
    pd.crosstab(
        pilot["Is Laundering"],
        pilot["gpt_prediction"]
    ).to_string()
)

pilot.to_csv(
    folder / "aml_gpt_history_ablation_pilot_100.csv",
    index=False
)

print("\n100 operacijų imtis išsaugota.")

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

folder = Path("aml_models")

prompt_path = folder / "gpt_context_prompt_v4.txt"
pilot_path = folder / "aml_gpt_history_ablation_pilot_100.csv"

prompt = prompt_path.read_text(encoding="utf-8")
pilot = pd.read_csv(pilot_path)

expected_hash = (
    "7731fd680e592e0ad3187cd9d10bf5e81a6acff8c6dd00267211e05b7d79232d"
)

actual_hash = hashlib.sha256(
    prompt.encode("utf-8")
).hexdigest()

print("GPT V4 INSTRUKCIJOS PATIKRA")
print("=" * 80)

print("SHA256:", actual_hash)
print("Atitinka originalą:", actual_hash == expected_hash)

assert actual_hash == expected_hash
assert len(pilot) == 100

print("\nINSTRUKCIJOS TEKSTAS")
print("=" * 80)
print(prompt)

print("\nPILOTINĖS IMTIES STULPELIAI")
print("=" * 80)

for column in pilot.columns:
    print("-", column)

print("\nPatikra baigta. API užklausos nevykdytos.")

In [ ]:
import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path

folder = Path("aml_models")

pilot = pd.read_csv(
    folder / "aml_gpt_history_ablation_pilot_100.csv"
)

# Paliekame bendrą originalios V4 instrukcijos dalį
original_prompt = (
    folder / "gpt_context_prompt_v4.txt"
).read_text(encoding="utf-8")

prompt_no_history = original_prompt.split(
    "Additional historical features:"
)[0].strip()

# Atkuriame visas 10 originalių operacijos savybių
raw_columns = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

assert all(col in df.columns for col in raw_columns)

# original_id yra pradinio df eilutės indeksas
assert pilot["original_id"].is_unique
assert pilot["original_id"].between(0, len(df)-1).all()

raw = df.iloc[
    pilot["original_id"].astype(int).to_numpy()
][raw_columns].copy()

raw.insert(
    0,
    "original_id",
    pilot["original_id"].to_numpy()
)

assert len(raw) == 100
assert raw["original_id"].is_unique
assert not raw[raw_columns].isna().any().any()

# Paruošiame tik modeliui leidžiamus duomenis
payloads = []

for _, row in raw.iterrows():
    transaction = {}

    for col in raw_columns:
        value = row[col]

        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, np.generic):
            value = value.item()

        transaction[col] = value

    payloads.append({
        "original_id": int(row["original_id"]),
        "payload": json.dumps(
            transaction,
            ensure_ascii=False,
            default=str
        )
    })

payload_df = pd.DataFrame(payloads)

assert len(payload_df) == 100

# Patikriname, kad į GPT užklausą nepatektų
# tikros klasės, XGBoost prognozės ar ID
for payload in payload_df["payload"]:
    obj = json.loads(payload)
    assert set(obj.keys()) == set(raw_columns)

(folder / "gpt_prompt_no_history.txt").write_text(
    prompt_no_history,
    encoding="utf-8"
)

payload_df.to_csv(
    folder / "aml_gpt_no_history_payloads_100.csv",
    index=False
)

print("EKSPERIMENTO BE ISTORINIŲ POŽYMIŲ PATIKRA")
print("=" * 80)

print("Operacijų:", len(payload_df))
print("Modeliui pateikiamų požymių:", len(raw_columns))
print("Istorinių požymių:", 0)

print(
    "Naujos instrukcijos SHA256:",
    hashlib.sha256(
        prompt_no_history.encode("utf-8")
    ).hexdigest()
)

print("\nNAUJA INSTRUKCIJA")
print("=" * 80)
print(prompt_no_history)

print("\nPIRMOS OPERACIJOS JSON")
print("=" * 80)
print(payload_df["payload"].iloc[0])

print("\nDuomenys paruošti. API užklausos nevykdytos.")

In [ ]:
from openai import OpenAI
from pathlib import Path
import pandas as pd
import json
import time

folder = Path("aml_models")

payloads = pd.read_csv(
    folder / "aml_gpt_no_history_payloads_100.csv"
)

prompt = (
    folder / "gpt_prompt_no_history.txt"
).read_text(encoding="utf-8")

client = OpenAI(
    timeout=30.0,
    max_retries=2
)

results = []

print("GPT BE ISTORINIŲ POŽYMIŲ – 5 OPERACIJŲ BANDYMAS")
print("=" * 85)

for i, row in payloads.head(5).iterrows():

    original_id = int(row["original_id"])

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=row["payload"],
            temperature=0
        )

        output = json.loads(response.output_text)

        prediction = output.get("prediction")

        if prediction not in (0, 1) or type(prediction) is not int:
            raise ValueError(
                f"Neteisinga prognozė: {prediction}"
            )

        result = {
            "original_id": original_id,
            "prediction_no_history": prediction,
            "reason_no_history": output.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "status": "OK"
        }

    except Exception as e:
        result = {
            "original_id": original_id,
            "prediction_no_history": None,
            "reason_no_history": str(e),
            "input_tokens": 0,
            "output_tokens": 0,
            "status": "ERROR"
        }

    results.append(result)

    print(
        f"{i+1}/5 | ID: {original_id} | "
        f"Prognozė: {result['prediction_no_history']} | "
        f"Statusas: {result['status']}"
    )

    time.sleep(0.2)

pilot_results = pd.DataFrame(results)

pilot_results.to_csv(
    folder / "aml_gpt_no_history_pilot_5_results.csv",
    index=False
)

print("\nBANDYMO SUVESTINĖ")
print("=" * 85)

print(
    pilot_results[
        [
            "original_id",
            "prediction_no_history",
            "status",
            "input_tokens",
            "output_tokens"
        ]
    ].to_string(index=False)
)

print("\nSėkmingos užklausos:",
      (pilot_results["status"] == "OK").sum(), "/ 5")

print(
    "Iš viso input tokenų:",
    pilot_results["input_tokens"].sum()
)

print(
    "Iš viso output tokenų:",
    pilot_results["output_tokens"].sum()
)

print("\nRezultatai išsaugoti.")

In [ ]:
from openai import OpenAI
from pathlib import Path
import pandas as pd
import json
import time

folder = Path("aml_models")

payloads = pd.read_csv(
    folder / "aml_gpt_no_history_payloads_100.csv"
)

prompt = (
    folder / "gpt_prompt_no_history.txt"
).read_text(encoding="utf-8")

pilot_file = folder / "aml_gpt_no_history_pilot_5_results.csv"
results_file = folder / "aml_gpt_no_history_100_results.csv"

# Įkeliame jau turimus rezultatus
if results_file.exists():
    results = pd.read_csv(results_file)
else:
    results = pd.read_csv(pilot_file)

# Sėkmingai apdorotų operacijų ID
completed = set(
    results.loc[
        results["status"] == "OK",
        "original_id"
    ].astype(int)
)

remaining = payloads[
    ~payloads["original_id"].isin(completed)
].copy()

print("GPT BE ISTORINIŲ POŽYMIŲ – LIKUSIOS OPERACIJOS")
print("=" * 85)
print("Jau sėkmingai apdorota:", len(completed))
print("Liko apdoroti:", len(remaining))

client = OpenAI(
    timeout=30.0,
    max_retries=2
)

for i, (_, row) in enumerate(remaining.iterrows(), start=1):
    original_id = int(row["original_id"])

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=prompt,
            input=row["payload"],
            temperature=0
        )

        output = json.loads(response.output_text)
        prediction = output.get("prediction")

        if type(prediction) is not int or prediction not in (0, 1):
            raise ValueError(f"Neteisinga prognozė: {prediction}")

        result = {
            "original_id": original_id,
            "prediction_no_history": prediction,
            "reason_no_history": output.get("reason", ""),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "status": "OK"
        }

    except Exception as e:
        result = {
            "original_id": original_id,
            "prediction_no_history": None,
            "reason_no_history": str(e),
            "input_tokens": 0,
            "output_tokens": 0,
            "status": "ERROR"
        }

    # Pakeičiame ankstesnį įrašą, jei tai pakartotinis bandymas
    results = results[
        results["original_id"] != original_id
    ]

    results = pd.concat(
        [results, pd.DataFrame([result])],
        ignore_index=True
    )

    # Išsaugome po kiekvienos užklausos
    results.to_csv(results_file, index=False)

    if i % 10 == 0 or i == len(remaining):
        print(
            f"Apdorota {i}/{len(remaining)} | "
            f"Paskutinės užklausos statusas: {result['status']}"
        )

    time.sleep(0.2)

# Galutinė patikra
results = pd.read_csv(results_file)

success = results[results["status"] == "OK"]

print("\nGALUTINĖ SUVESTINĖ")
print("=" * 85)

print("Iš viso įrašų:", len(results))
print("Sėkmingų užklausų:", len(success))
print("Klaidų:", (results["status"] == "ERROR").sum())
print("Unikalių ID:", results["original_id"].nunique())

print("Input tokenų:", int(results["input_tokens"].sum()))
print("Output tokenų:", int(results["output_tokens"].sum()))

assert len(success) == 100, (
    "Ne visos užklausos sėkmingos. "
    "Paleisk šį langelį dar kartą – "
    "bus kartojamos tik nesėkmingos užklausos."
)

print("\nVisos 100 operacijų sėkmingai išanalizuotos!")
print("Rezultatai išsaugoti:", results_file)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

folder = Path("aml_models")

pilot = pd.read_csv(
    folder / "aml_gpt_history_ablation_pilot_100.csv"
)

new = pd.read_csv(
    folder / "aml_gpt_no_history_100_results.csv"
)

assert len(pilot) == 100
assert len(new) == 100
assert pilot["original_id"].is_unique
assert new["original_id"].is_unique
assert (new["status"] == "OK").all()

data = pilot.merge(
    new[["original_id", "prediction_no_history"]],
    on="original_id",
    how="inner",
    validate="one_to_one"
)

assert len(data) == 100
assert not data["prediction_no_history"].isna().any()

y = data["Is Laundering"].astype(int).to_numpy()

predictions = {
    "Su istorija (V4)": data["gpt_prediction"].astype(int).to_numpy(),
    "Be istorijos": data["prediction_no_history"].astype(int).to_numpy()
}

rows = []

for name, pred in predictions.items():
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "Modelis": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision_proc": 100 * tp / (tp + fp) if tp + fp else np.nan,
        "Recall_proc": 100 * tp / (tp + fn) if tp + fn else np.nan,
        "FPR_proc": 100 * fp / (fp + tn) if fp + tn else np.nan,
        "F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    })

metrics = pd.DataFrame(rows)

print("GPT SU ISTORIJA IR BE ISTORIJOS")
print("=" * 110)
print(metrics.round(3).to_string(index=False))

# Sprendimų pokyčiai
data["pasikeite"] = (
    data["gpt_prediction"].astype(int)
    != data["prediction_no_history"].astype(int)
)

print("\nSPRENDIMŲ POKYČIAI")
print("=" * 85)

print("Pasikeitusių sprendimų:", int(data["pasikeite"].sum()))

print("\nPerėjimai tarp prognozių:")
print(
    pd.crosstab(
        data["gpt_prediction"],
        data["prediction_no_history"],
        rownames=["Su_istorija"],
        colnames=["Be_istorijos"]
    ).to_string()
)

# Tikriname, kaip pasikeitė sprendimų teisingumas
correct_old = (
    data["gpt_prediction"].astype(int).to_numpy() == y
)
correct_new = (
    data["prediction_no_history"].astype(int).to_numpy() == y
)

old_only_correct = int((correct_old & ~correct_new).sum())
new_only_correct = int((~correct_old & correct_new).sum())

table = [
    [int((correct_old & correct_new).sum()), old_only_correct],
    [new_only_correct, int((~correct_old & ~correct_new).sum())]
]

test = mcnemar(table, exact=True)

print("\nMCNEMAR TESTAS – SPRENDIMŲ TEISINGUMAS")
print("=" * 85)
print("Teisingas tik su istorija:", old_only_correct)
print("Teisingas tik be istorijos:", new_only_correct)
print("p reikšmė:", round(test.pvalue, 8))

# Pokyčiai pagal aktyvumo grupę
activity_rows = []

for group, subset in data.groupby(
    "aktyvumo_grupe", observed=True
):
    for label in [0, 1]:
        s = subset[subset["Is Laundering"] == label]

        activity_rows.append({
            "Aktyvumas": str(group),
            "Tikra_klase": label,
            "Operacijos": len(s),
            "Su_istorija_pazymeta_1": int(s["gpt_prediction"].sum()),
            "Be_istorijos_pazymeta_1": int(
                s["prediction_no_history"].sum()
            ),
            "Pasikeite": int(s["pasikeite"].sum())
        })

activity = pd.DataFrame(activity_rows)

print("\nPOKYČIAI PAGAL SIUNTĖJO AKTYVUMĄ")
print("=" * 110)
print(activity.to_string(index=False))

metrics.to_csv(folder / "aml_gpt_history_ablation_metrics.csv", index=False)
activity.to_csv(folder / "aml_gpt_history_ablation_by_activity.csv", index=False)
data.to_csv(folder / "aml_gpt_history_ablation_comparison_100.csv", index=False)

print("\nVisi palyginimo rezultatai išsaugoti.")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

folder = Path("aml_models")

data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

model = joblib.load(folder / "xgb_model.joblib")
preprocessor = joblib.load(folder / "preprocessor.joblib")

feature_cols = [
    "Timestamp_sec",
    "Amount Received",
    "Amount Paid",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format"
]

ids = data["original_id"].astype(int).to_numpy()

raw = df.iloc[ids].copy()

raw["Timestamp_sec"] = (
    pd.to_datetime(raw["Timestamp"])
    .astype("int64") / 1e9
)

X = preprocessor.transform(raw[feature_cols])

# Naujos tikrosios modelio tikimybės
fresh_prob = model.predict_proba(X)[:, 1]

saved_prob = data["xgb_probability"].to_numpy(dtype=float)

difference = np.abs(fresh_prob - saved_prob)

threshold = 0.957565

print("XGBOOST RIZIKOS BALŲ PATIKRA")
print("=" * 85)

print("Operacijų:", len(data))
print("Unikalių išsaugotų balų:", len(np.unique(saved_prob)))
print("Unikalių naujai apskaičiuotų balų:", len(np.unique(fresh_prob)))

print("\nBALŲ PALYGINIMAS")
print("=" * 85)

print("Didžiausias absoliutus skirtumas:", difference.max())
print("Vidutinis absoliutus skirtumas:", difference.mean())

print(
    "Sutampa (paklaida iki 1e-6):",
    int((difference <= 1e-6).sum()),
    "/",
    len(data)
)

print("\nDAŽNIAUSI IŠSAUGOTI BALAI")
print("=" * 85)

print(
    pd.Series(saved_prob)
    .value_counts()
    .head(10)
    .to_string()
)

print("\nSLENKSČIO PATIKRA")
print("=" * 85)

print(
    "Išsaugoti balai arti slenksčio:",
    int(np.isclose(saved_prob, threshold, atol=1e-6).sum())
)

print(
    "Naujai apskaičiuoti balai arti slenksčio:",
    int(np.isclose(fresh_prob, threshold, atol=1e-6).sum())
)

print("\nPROGNOZIŲ ATITIKIMAS")
print("=" * 85)

saved_pred = data["xgb_prediction"].astype(int).to_numpy()
fresh_pred = (fresh_prob >= threshold).astype(int)

print(
    "Sutampančios klasifikacijos:",
    int((saved_pred == fresh_pred).sum()),
    "/",
    len(data)
)

diagnostics = pd.DataFrame({
    "original_id": ids,
    "saved_probability": saved_prob,
    "fresh_probability": fresh_prob,
    "absolute_difference": difference,
    "saved_prediction": saved_pred,
    "fresh_prediction": fresh_pred
})

diagnostics.to_csv(
    folder / "aml_xgb_probability_integrity_check.csv",
    index=False
)

print("\nPatikros rezultatai išsaugoti.")

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("aml_models")

files = {
    "Pagrindinis 3000 operacijų palyginimas":
        "aml_3000_final_comparison.csv",

    "Modelių pasikliautinieji intervalai":
        "aml_3000_confidence_intervals.csv",

    "McNemar testai":
        "aml_3000_mcnemar.csv",

    "Hibridinio modelio vertinimas":
        "aml_gpt_incremental_holdout_2000.csv",

    "Hibridinio modelio bootstrap":
        "aml_hybrid_2000_bootstrap_improvement.csv",

    "Tikrinimų biudžetų bootstrap":
        "aml_hybrid_2000_budget_bootstrap_final.csv",

    "GPT istorijos eksperimento rezultatai":
        "aml_gpt_history_ablation_metrics.csv",

    "GPT istorijos eksperimento McNemar duomenys":
        "aml_gpt_history_ablation_comparison_100.csv",

    "XGBoost balų patikra":
        "aml_xgb_probability_integrity_check.csv"
}

print("GALUTINĖ MAGISTRINIO TYRIMO REZULTATŲ PATIKRA")
print("=" * 90)

all_exist = True

for name, filename in files.items():
    path = folder / filename
    exists = path.exists()

    print(
        f"{'OK' if exists else 'TRŪKSTA':8} | "
        f"{name}"
    )

    if not exists:
        all_exist = False

print("\n" + "=" * 90)

if all_exist:
    print("Visi pagrindiniai rezultatų failai rasti.")
else:
    print("Trūksta dalies rezultatų. Patikrinsime prieš rašydami.")

# Pagrindinė klasifikavimo suvestinė
data = pd.read_csv(
    folder / "aml_3000_final_comparison.csv"
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

rows = []

for model, column in [
    ("XGBoost", "xgb_prediction"),
    ("GPT-5.4-mini", "gpt_prediction"),
    ("OR", "or_prediction"),
    ("AND", "and_prediction")
]:
    y = data["Is Laundering"]
    pred = data[column]

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "Modelis": model,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0)
    })

summary = pd.DataFrame(rows)

print("\nPAGRINDINIAI REZULTATAI")
print("=" * 90)
print(summary.round(4).to_string(index=False))

summary.to_csv(
    folder / "aml_final_thesis_main_results.csv",
    index=False
)

print("\nPagrindinė magistrinio darbo rezultatų lentelė išsaugota.")

In [ ]:

import pandas as pd

df = pd.read_csv("aml_models/gpt_v4_natural_3000_results.csv")

tp = df[
    (df["true_label"] == 1) &
    (df["gpt_prediction"] == 1)
]

fp = df[
    (df["true_label"] == 0) &
    (df["gpt_prediction"] == 1)
]

columns = [
    "original_id",
    "true_label",
    "gpt_prediction",
    "gpt_reason",
    "raw_response"
]

print("TEISINGAI APTIKTAS PINIGŲ PLOVIMO ATVEJIS:")
print(tp[columns].head(1).to_string(index=False))

print("\nKLAIDINGAS ĮSPĖJIMAS:")
print(fp[columns].head(1).to_string(index=False))


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# y_test – tikrosios testavimo operacijų klasės
# y_pred – XGBoost prognozės, naudojant 0.957565 ribą

tn, fp, fn, tp = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).ravel()

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
print(f"Recall: {recall_score(y_test, y_pred)*100:.2f}%")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"FPR: {fp/(fp+tn)*100:.3f}%")
print("Įspėjimų skaičius:", tp + fp)

In [ ]:

import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Galutinio XGBoost eksperimento klasifikavimo rezultatai
tn = 859194
fp = 3095
fn = 1213
tp = 398

cm = np.array([
    [tn, fp],
    [fn, tp]
])

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
fpr = fp / (fp + tn)
alerts = tp + fp

print("Klasifikavimo matrica:")
print(cm)

print(f"\nPrecision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1: {f1:.4f}")
print(f"FPR: {fpr * 100:.3f}%")
print(f"Įspėjimų skaičius: {alerts}")


In [ ]:
with open("aml_models/gpt_context_prompt_v4.txt", "r", encoding="utf-8") as f:
    print(f.read())